# Arabic Handwritten Character Recognition — Experiment Notebook

**Dataset:** Arabic Handwritten Characters Dataset (AHCD) · **28 classes** · **TensorFlow/Keras**

This notebook documents an experimental research workflow: dataset inspection, quality checks, preprocessing, convolutional neural networks, augmentation ablations, three-seed replications, Residual CNN experiments, Grad-CAM, final test evaluation, and preparation of a Gradio demonstration.

> **GitHub edition.** The original notebook is preserved separately. Cell outputs and execution-specific metadata have been cleared for readability, privacy, and repository size; **the original source code cells are preserved**. The numerical summaries below are transcribed from the original saved experiment outputs, not newly computed by this edition.

## Main findings

| Metric | Baseline CNN | Residual CNN |
|:--|--:|--:|
| Parameters | 103,420 | 230,780 |
| Validation accuracy (3 seeds, mean ± sample SD) | 94.6594 ± 0.1878 % | **97.8175 ± 0.1488 %** |
| Validation macro-F1 (3 seeds, mean ± sample SD) | 94.6546 ± 0.1894 % | **97.8160 ± 0.1509 %** |

The paired mean validation-accuracy difference across matched seeds **123, 456, 789** was **+3.1581 percentage points** (sample SD **0.2545** points). Three random seeds provide descriptive evidence of consistency, not a claim of statistical significance. Since the Residual CNN is larger, its improvement cannot be attributed uniquely to skip connections.

**Final held-out test (selected Residual CNN, seed 789, AHCD test set of 3,360 images):**

| Metric | Result |
|:--|--:|
| Accuracy | **98.3929 %** |
| Macro-F1 | **98.3896 %** |
| Top-3 accuracy | **99.8810 %** |
| Errors | 54 / 3,360 |

The best model was selected using validation results before the final test evaluation. Test data should **not** be used for additional model selection or tuning. Performance for independently uploaded or drawn characters may differ from AHCD test performance.

## How to read and use this notebook

1. **Read it as a record of completed experiments, not a one-click rerunnable script.** Some cells are historical recovery/troubleshooting steps, repeated training runs, and model-saving commands. **Do not select _Run all_**: doing so would retrain models and could overwrite Drive artifacts.
2. The notebook was developed in **Google Colab** and uses Google Drive paths such as `/content/drive/MyDrive/Arabic_Handwritten_CNN/`. To reproduce selected experiments, first download AHCD and create the expected Drive directories or adapt paths. Saved checkpoints, preprocessing data, experiment CSVs, and generated figures are **not embedded in this notebook**.
3. Some later cells load previously saved checkpoints and will not work without those external artifacts. The standalone interactive application lives in the repository's **`app/`** directory (including `app.py`, model weights, dependencies, and image preprocessing).
4. Outputs, plots, and training logs were removed for GitHub readability; reproduce the key figures from saved result tables or add approved PNGs separately under `results/figures/`.
5. The original AHCD CSV split does not provide verified writer identifiers in this notebook. We therefore do **not** claim that its internal train/validation partition was demonstrably writer-independent. No accuracy claim is made for arbitrary handwriting outside the AHCD distribution.

## Dataset attribution

AHCD dataset repository: https://github.com/mloey/Arabic-Handwritten-Characters-Dataset

---

## Experiment source cells

The following cells retain the original code and experiment ordering. Markdown headings have been normalized for GitHub rendering.


# Project: Arabic Handwritten Character Recognition Using Deep Learning

# Environment Setup and Verification

In [ ]:
import sys
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import cv2

# Display Python and library versions
print("Python version:", sys.version.split()[0])
print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("OpenCV version:", cv2.__version__)

# Check GPU availability
print("\n--- GPU AVAILABILITY CHECK ---")

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU detected:", gpus)
    print("Environment is ready for deep learning!")
else:
    print("No GPU detected by TensorFlow.")
    print("The project can still run on CPU.")


# Dataset Acquisition and Loading

In [ ]:
from pathlib import Path
import requests
import zipfile

# Define the dataset directory
DATA_DIR = Path("/content/arabic_cnn/data/raw")

# Create the directory if it does not exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Official AHCD dataset repository URL
url = (
    "https://raw.githubusercontent.com/mloey/"
    "Arabic-Handwritten-Characters-Dataset/master/"
    "Arabic%20Handwritten%20Characters%20Dataset%20CSV.zip"
)

# Define the ZIP file path
zip_path = DATA_DIR / "ahcd_csv.zip"

# Download the dataset
response = requests.get(url, timeout=120)
response.raise_for_status()

# Save the downloaded ZIP file
zip_path.write_bytes(response.content)

# Extract the dataset
with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(DATA_DIR)

# Locate all extracted CSV files
csv_files = sorted(DATA_DIR.rglob("*.csv"))

print("Dataset downloaded and extracted successfully!")
print("\nExtracted CSV files:")

for file in csv_files:
    print("-", file.name)

print("\nTotal CSV files:", len(csv_files))


# Load and Verify the Dataset

In [ ]:
import pandas as pd
import numpy as np

# Define a function to locate a CSV file by its prefix
def get_csv(prefix):
    matches = [
        file for file in csv_files
        if file.name.startswith(prefix)
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            f"File not found or ambiguous: {prefix}"
        )

    return matches[0]


# Load training images and labels
X_train = pd.read_csv(
    get_csv("csvTrainImages"), header=None
)

y_train = pd.read_csv(
    get_csv("csvTrainLabel"), header=None
)

# Load testing images and labels
X_test = pd.read_csv(
    get_csv("csvTestImages"), header=None
)

y_test = pd.read_csv(
    get_csv("csvTestLabel"), header=None
)

# Display dataset dimensions
print("--- DATASET SHAPES ---")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

# Verify the expected dataset dimensions
assert X_train.shape == (13440, 1024)
assert y_train.shape == (13440, 1)

assert X_test.shape == (3360, 1024)
assert y_test.shape == (3360, 1)

# Display the available character classes
print("\nAvailable classes:")
print(np.sort(y_train[0].unique()))

print("\nDataset verification completed successfully!")


# Exploratory Data Analysis (EDA)

### Image Visualization and Orientation Verification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Convert Pandas DataFrames into NumPy arrays
images = X_train.to_numpy()
labels = y_train.iloc[:, 0].to_numpy()

# Select the first eight character classes
classes = np.unique(labels)[:8]

# Select one image from each class
indices = [
    np.where(labels == class_id)[0][0]
    for class_id in classes
]

# Create a figure to compare image orientations
fig, axes = plt.subplots(
    2, 8, figsize=(16, 5)
)

for j, idx in enumerate(indices):

    # Reshape the flattened image into a 32x32 matrix
    image = images[idx].reshape(32, 32)

    # Display the original image
    axes[0, j].imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=255
    )

    axes[0, j].set_title(
        f"Class {labels[idx]}"
    )
    axes[0, j].axis("off")

    # Display the transposed image
    axes[1, j].imshow(
        image.T,
        cmap="gray",
        vmin=0,
        vmax=255
    )

    axes[1, j].set_title(
        f"Class {labels[idx]}"
    )
    axes[1, j].axis("off")

# Add a title to the figure
fig.suptitle(
    "AHCD: Original Images (Top) vs Transposed Images (Bottom)",
    fontsize=14
)

plt.tight_layout()
plt.show()

# Display pixel intensity range
print("--- PIXEL INTENSITY ANALYSIS ---")

print("Minimum pixel value:", images.min())
print("Maximum pixel value:", images.max())


### Class Distribution Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Extract labels from the training and testing datasets
train_labels = y_train.iloc[:, 0].to_numpy()
test_labels = y_test.iloc[:, 0].to_numpy()

# Count the number of samples in each class
train_counts = pd.Series(train_labels).value_counts().sort_index()
test_counts = pd.Series(test_labels).value_counts().sort_index()

# Create a class distribution summary
class_distribution = pd.DataFrame({
    "Class": train_counts.index,
    "Training Samples": train_counts.values,
    "Testing Samples": test_counts.reindex(
        train_counts.index, fill_value=0
    ).values
})

# Display the class distribution table
print("=== CLASS DISTRIBUTION ===")
print(class_distribution.to_string(index=False))

# Visualize class distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Training class distribution
axes[0].bar(
    train_counts.index,
    train_counts.values,
    color="steelblue"
)
axes[0].set_title("Training Set - Class Distribution")
axes[0].set_xlabel("Class Label")
axes[0].set_ylabel("Number of Samples")
axes[0].set_xticks(range(1, 29))
axes[0].grid(axis="y", alpha=0.3)

# Testing class distribution
axes[1].bar(
    test_counts.index,
    test_counts.values,
    color="mediumseagreen"
)
axes[1].set_title("Testing Set - Class Distribution")
axes[1].set_xlabel("Class Label")
axes[1].set_ylabel("Number of Samples")
axes[1].set_xticks(range(1, 29))
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Check class balance
print("\n=== CLASS BALANCE VERIFICATION ===")

print("Number of training classes:", len(train_counts))
print("Number of testing classes:", len(test_counts))

print("Minimum training samples per class:", train_counts.min())
print("Maximum training samples per class:", train_counts.max())

print("Minimum testing samples per class:", test_counts.min())
print("Maximum testing samples per class:", test_counts.max())

print("\nTraining set balanced:", train_counts.nunique() == 1)
print("Testing set balanced:", test_counts.nunique() == 1)


### Pixel Intensity and Image Quality Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use only training data for exploratory analysis
train_images = X_train.to_numpy()

# Check for missing values
missing_values = np.isnan(train_images).sum()

# Check pixel intensity range
min_pixel = train_images.min()
max_pixel = train_images.max()

print("=== PIXEL INTENSITY STATISTICS ===")
print("Minimum pixel intensity:", min_pixel)
print("Maximum pixel intensity:", max_pixel)
print("Mean pixel intensity:", train_images.mean())
print("Standard deviation:", train_images.std())
print("Missing pixel values:", missing_values)

# Calculate pixel intensity distribution
pixel_counts = np.bincount(
    train_images.astype(np.uint8).ravel(),
    minlength=256
)

# Calculate foreground pixel ratio per image
# Pixels above 20 are considered potential foreground
foreground_ratio = (train_images > 20).mean(axis=1)

# Detect completely black or white images
black_images = np.sum(train_images.max(axis=1) == 0)
white_images = np.sum(train_images.min(axis=1) == 255)

# Display image quality statistics
print("\n=== IMAGE QUALITY STATISTICS ===")
print("Completely black images:", black_images)
print("Completely white images:", white_images)

print(
    "Average foreground ratio:",
    round(foreground_ratio.mean() * 100, 2), "%"
)

print(
    "Minimum foreground ratio:",
    round(foreground_ratio.min() * 100, 2), "%"
)

print(
    "Maximum foreground ratio:",
    round(foreground_ratio.max() * 100, 2), "%"
)

# Visualize pixel statistics
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pixel intensity histogram
axes[0].bar(
    np.arange(256),
    pixel_counts,
    color="steelblue",
    width=1
)

axes[0].set_title("Pixel Intensity Distribution")
axes[0].set_xlabel("Pixel Intensity (0-255)")
axes[0].set_ylabel("Pixel Count")
axes[0].grid(alpha=0.3)

# Foreground ratio distribution
axes[1].hist(
    foreground_ratio * 100,
    bins=40,
    color="mediumseagreen",
    edgecolor="black"
)

axes[1].set_title("Foreground Pixel Ratio Distribution")
axes[1].set_xlabel("Foreground Pixels (%)")
axes[1].set_ylabel("Number of Images")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


#### Anomalous Image Investigation

In [ ]:
#ANOMALOUS IMAGE INVESTIGATION

import numpy as np
import matplotlib.pyplot as plt

# Convert training images and labels to NumPy arrays
train_images = X_train.to_numpy()
train_labels = y_train.iloc[:, 0].to_numpy()

# Identify completely black images
black_indices = np.where(
    np.all(train_images == 0, axis=1)
)[0]

print("=== BLACK IMAGE DETECTION ===")
print("Number of black images:", len(black_indices))
print("Black image indices:", black_indices)

# Display each black image and its associated label
for idx in black_indices:

    image = train_images[idx].reshape(32, 32)

    print("\nImage index:", idx)
    print("Class label:", train_labels[idx])
    print("Minimum pixel value:", image.min())
    print("Maximum pixel value:", image.max())

    fig, axes = plt.subplots(1, 2, figsize=(7, 3))

    axes[0].imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=255
    )
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(
        image.T,
        cmap="gray",
        vmin=0,
        vmax=255
    )
    axes[1].set_title("Transposed Image")
    axes[1].axis("off")

    plt.suptitle(
        f"Anomalous Sample - Class {train_labels[idx]}"
    )
    plt.tight_layout()
    plt.show()

# Detect images with very little visible foreground
foreground_ratio = (train_images > 20).mean(axis=1)

low_foreground_indices = np.where(
    foreground_ratio < 0.01
)[0]

print("\n=== LOW FOREGROUND ANALYSIS ===")
print(
    "Images with less than 1% foreground:",
    len(low_foreground_indices)
)

print("Indices:", low_foreground_indices[:20])


#### Training Data Cleaning

In [ ]:
import numpy as np

# Detect completely black training images
black_mask = np.all(
    X_train.to_numpy() == 0,
    axis=1
)

# Record the original indices of removed samples
removed_indices = np.flatnonzero(black_mask)

# Create cleaned training datasets
# Preserve the original X_train and y_train
X_train_clean = X_train.loc[
    ~black_mask
].reset_index(drop=True).copy()

y_train_clean = y_train.loc[
    ~black_mask
].reset_index(drop=True).copy()

# Verify the cleaning operation
assert len(X_train_clean) == len(y_train_clean)

assert not np.any(
    np.all(X_train_clean.to_numpy() == 0, axis=1)
)

# Display cleaning results
print("=== DATA CLEANING SUMMARY ===")

print("Original training samples:", len(X_train))
print("Removed samples:", len(removed_indices))
print("Removed original indices:", removed_indices.tolist())
print("Cleaned training samples:", len(X_train_clean))

print("\n=== CLASS DISTRIBUTION AFTER CLEANING ===")

class_counts_clean = (
    y_train_clean.iloc[:, 0]
    .value_counts()
    .sort_index()
)

print("Class 1 samples:", class_counts_clean.loc[1])
print("Minimum samples per class:", class_counts_clean.min())
print("Maximum samples per class:", class_counts_clean.max())

print("\n=== TEST SET INTEGRITY ===")

print("Test samples:", len(X_test))
print("Test set remains unchanged.")

print("\nTraining data cleaning completed successfully!")


## Intra-Class Variability Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Use the cleaned training dataset
train_images = X_train_clean.to_numpy(dtype=np.uint8)
train_labels = y_train_clean.iloc[:, 0].to_numpy()

# Set a random seed for reproducibility
rng = np.random.default_rng(42)

# Create a figure with 28 character classes
fig, axes = plt.subplots(7, 4, figsize=(16, 17))

for ax, class_id in zip(axes.flat, range(1, 29)):

    # Find all image indices belonging to this class
    class_indices = np.where(
        train_labels == class_id
    )[0]

    # Randomly select three different samples
    selected_indices = rng.choice(
        class_indices,
        size=3,
        replace=False
    )

    # Reconstruct and correct image orientation
    sample_images = [
        train_images[idx].reshape(32, 32).T
        for idx in selected_indices
    ]

    # Combine the three images into one horizontal strip
    image_strip = np.hstack(sample_images)

    # Display the selected samples
    ax.imshow(
        image_strip,
        cmap="gray",
        vmin=0,
        vmax=255
    )

    ax.set_title(
        f"Class {class_id} - {len(class_indices)} samples",
        fontsize=10
    )
    ax.axis("off")

# Add the main figure title
fig.suptitle(
    "AHCD - Handwriting Variability Across 28 Classes",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

# Save the figure for project documentation
FIGURES_DIR = Path(
    "/content/arabic_cnn/results/figures"
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

figure_path = (
    FIGURES_DIR / "intra_class_variability.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Visualization completed successfully!")
print("Figure saved to:", figure_path)


## Inter-Class Similarity Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define groups of visually similar Arabic characters
similar_groups = {
    "Baa - Taa - Thaa": [2, 3, 4],
    "Jeem - Haa - Khaa": [5, 6, 7],
    "Seen - Sheen": [12, 13],
    "Ayn - Ghayn": [18, 19]
}

# Use cleaned training data
images = X_train_clean.to_numpy(dtype=np.uint8)
labels = y_train_clean.iloc[:, 0].to_numpy()

# Ensure reproducible sample selection
rng = np.random.default_rng(42)

# Count the total number of classes to display
total_rows = sum(len(group) for group in similar_groups.values())

fig, axes = plt.subplots(
    total_rows, 3,
    figsize=(9, total_rows * 1.7)
)

row_idx = 0

for group_name, class_ids in similar_groups.items():

    for class_id in class_ids:

        # Find all samples belonging to the current class
        class_indices = np.where(
            labels == class_id
        )[0]

        # Randomly select three different samples
        selected_indices = rng.choice(
            class_indices,
            size=3,
            replace=False
        )

        for col_idx, idx in enumerate(selected_indices):

            # Restore the original image dimensions
            image = images[idx].reshape(32, 32).T

            # Display the correctly oriented image
            axes[row_idx, col_idx].imshow(
                image,
                cmap="gray",
                vmin=0,
                vmax=255
            )

            axes[row_idx, col_idx].axis("off")

            # Display the class and similarity group
            if col_idx == 0:
                axes[row_idx, col_idx].set_title(
                    f"{group_name}\nClass {class_id}",
                    fontsize=10
                )

        row_idx += 1

fig.suptitle(
    "AHCD - Visual Similarity Between Arabic Character Classes",
    fontsize=14
)

plt.tight_layout(rect=[0, 0, 1, 0.97])

# Save the analysis figure
figure_path = (
    FIGURES_DIR / "inter_class_similarity.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Inter-class similarity visualization completed!")
print("Figure saved to:", figure_path)


## Duplicate Images and Label Consistency Analysis

In [ ]:
import numpy as np

# Load cleaned training images and labels
images = X_train_clean.to_numpy(dtype=np.uint8)
labels = y_train_clean.iloc[:, 0].to_numpy()

# Identify unique images and duplicate occurrences
unique_images, inverse_indices, counts = np.unique(
    images,
    axis=0,
    return_inverse=True,
    return_counts=True
)

# Find groups containing identical images
duplicate_group_ids = np.where(counts > 1)[0]

# Calculate duplicate statistics
duplicate_groups = len(duplicate_group_ids)
repeated_samples = int(counts[counts > 1].sum())
extra_copies = repeated_samples - duplicate_groups

print("=== DUPLICATE IMAGE ANALYSIS ===")
print("Total training images:", len(images))
print("Unique images:", len(unique_images))
print("Duplicate groups:", duplicate_groups)
print("Samples in duplicate groups:", repeated_samples)
print("Extra duplicate copies:", extra_copies)

# Detect conflicting labels among identical images
conflicting_groups = []

for group_id in duplicate_group_ids:

    # Find all occurrences of the same image
    indices = np.where(
        inverse_indices == group_id
    )[0]

    # Find their associated class labels
    group_labels = np.unique(labels[indices])

    if len(group_labels) > 1:
        conflicting_groups.append({
            "indices": indices.tolist(),
            "labels": group_labels.tolist()
        })

print("\n=== LABEL CONSISTENCY ANALYSIS ===")
print("Conflicting label groups:", len(conflicting_groups))

# Display a few examples of conflicting labels
for conflict in conflicting_groups[:5]:
    print(conflict)

print("\nData quality audit completed successfully!")
print("No samples have been removed during this analysis.")


# Data Preprocessing

## Stratified Train/Validation Split

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Set the random seed for reproducibility
RANDOM_STATE = 42

# Extract labels from the cleaned training dataset
clean_labels = y_train_clean.iloc[:, 0].to_numpy()

# Create sample indices
sample_indices = np.arange(len(X_train_clean))

# Split indices into training and validation subsets
train_indices, val_indices = train_test_split(
    sample_indices,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=clean_labels
)

# Create the training subset
X_train_split = X_train_clean.iloc[
    train_indices
].to_numpy(dtype=np.uint8)

y_train_split = clean_labels[train_indices]

# Create the validation subset
X_val = X_train_clean.iloc[
    val_indices
].to_numpy(dtype=np.uint8)

y_val = clean_labels[val_indices]

# Keep the original test set unchanged
print("=== DATASET SPLIT SUMMARY ===")
print("Training samples:", len(X_train_split))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

# Verify that training and validation do not overlap
overlap = np.intersect1d(
    train_indices,
    val_indices
)

assert len(overlap) == 0
assert len(X_train_split) + len(X_val) == len(X_train_clean)

print("\nTraining/Validation index overlap:", len(overlap))

# Analyze class distributions after splitting
train_counts = pd.Series(
    y_train_split
).value_counts().sort_index()

val_counts = pd.Series(
    y_val
).value_counts().sort_index()

distribution = pd.DataFrame({
    "Training": train_counts,
    "Validation": val_counts
})

print("\n=== CLASS DISTRIBUTION AFTER SPLIT ===")
print(distribution.to_string())

# Verify that all 28 classes are represented
assert len(train_counts) == 28
assert len(val_counts) == 28

print("\nStratified split completed successfully!")


## Image Reshaping, Orientation Correction and Normalization

In [ ]:
#IMAGE RESHAPING, ORIENTATION CORRECTION AND NORMALIZATION

import numpy as np
import matplotlib.pyplot as plt

# Define image dimensions
IMG_HEIGHT = 32
IMG_WIDTH = 32
CHANNELS = 1

# Define a reusable preprocessing function
def preprocess_images(images):

    # Convert input data to a NumPy array
    images = np.asarray(images, dtype=np.float32)

    # Restore the original spatial dimensions
    images = images.reshape(-1, IMG_HEIGHT, IMG_WIDTH)

    # Correct AHCD image orientation
    images = np.transpose(images, (0, 2, 1))

    # Add the grayscale channel dimension
    images = images[..., np.newaxis]

    # Normalize pixel intensities to [0, 1]
    images = images / 255.0

    # Return a contiguous float32 array
    return np.ascontiguousarray(images, dtype=np.float32)


# Preprocess training images
X_train_cnn = preprocess_images(X_train_split)

# Preprocess validation images
X_val_cnn = preprocess_images(X_val)

# Preprocess test images without evaluating them
X_test_cnn = preprocess_images(X_test)


# ============================================================
# VERIFY PREPROCESSED DATA
# ============================================================

print("=== PREPROCESSED DATASET SHAPES ===")
print("Training:", X_train_cnn.shape)
print("Validation:", X_val_cnn.shape)
print("Testing:", X_test_cnn.shape)

print("\n=== PIXEL INTENSITY RANGE ===")
print("Minimum:", X_train_cnn.min())
print("Maximum:", X_train_cnn.max())

print("\n=== DATA TYPE ===")
print("Training dtype:", X_train_cnn.dtype)

# Verify expected dimensions
assert X_train_cnn.shape == (11423, 32, 32, 1)
assert X_val_cnn.shape == (2016, 32, 32, 1)
assert X_test_cnn.shape == (3360, 32, 32, 1)

# Verify normalized pixel values
for dataset in [X_train_cnn, X_val_cnn, X_test_cnn]:
    assert np.all(np.isfinite(dataset))
    assert dataset.min() >= 0.0
    assert dataset.max() <= 1.0

print("\nImage preprocessing completed successfully!")


# ============================================================
# VISUALIZE PREPROCESSED TRAINING IMAGES
# ============================================================

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for i, ax in enumerate(axes):

    ax.imshow(
        X_train_cnn[i, :, :, 0],
        cmap="gray",
        vmin=0,
        vmax=1
    )

    ax.set_title(f"Class {y_train_split[i]}")
    ax.axis("off")

plt.suptitle(
    "Preprocessed Arabic Handwritten Characters",
    fontsize=13
)

plt.tight_layout()
plt.show()


## Label Encoding and Target Preparation

In [ ]:
import numpy as np

# Define the number of character classes
NUM_CLASSES = 28

# Define the Arabic alphabet in AHCD class order
arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

assert len(arabic_characters) == NUM_CLASSES

# Create mappings between encoded labels and characters
idx_to_char = {
    idx: char
    for idx, char in enumerate(arabic_characters)
}

char_to_idx = {
    char: idx
    for idx, char in idx_to_char.items()
}

# Convert original labels (1-28) to encoded labels (0-27)
y_train_cnn = np.asarray(
    y_train_split, dtype=np.int64
) - 1

y_val_cnn = np.asarray(
    y_val, dtype=np.int64
) - 1

y_test_cnn = (
    y_test.iloc[:, 0].to_numpy(dtype=np.int64) - 1
)

# Verify label ranges and data integrity
for encoded_labels in [
    y_train_cnn, y_val_cnn, y_test_cnn
]:
    assert encoded_labels.min() >= 0
    assert encoded_labels.max() < NUM_CLASSES

# Verify that training and validation contain all classes
assert len(np.unique(y_train_cnn)) == NUM_CLASSES
assert len(np.unique(y_val_cnn)) == NUM_CLASSES

# Verify image-label alignment
assert len(X_train_cnn) == len(y_train_cnn)
assert len(X_val_cnn) == len(y_val_cnn)
assert len(X_test_cnn) == len(y_test_cnn)

print("=== ENCODED LABEL INFORMATION ===")
print("Number of classes:", NUM_CLASSES)
print("Training labels shape:", y_train_cnn.shape)
print("Validation labels shape:", y_val_cnn.shape)
print("Testing labels shape:", y_test_cnn.shape)

print("\nMinimum encoded label:", y_train_cnn.min())
print("Maximum encoded label:", y_train_cnn.max())

print("\n=== LABEL MAPPING EXAMPLES ===")

for original_label in [1, 2, 5, 20, 28]:
    encoded_label = original_label - 1
    character = idx_to_char[encoded_label]

    print(
        f"Original: {original_label} | "
        f"Encoded: {encoded_label} | "
        f"Character: {character}"
    )

print("\n=== FIRST FIVE TRAINING LABELS ===")
print("Original labels:", y_train_split[:5])
print("Encoded labels:", y_train_cnn[:5])

print("\nLabel encoding completed successfully!")


## Data Augmentation

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Define reproducibility seed
RANDOM_STATE = 42

# Build an augmentation pipeline for training images only
data_augmentation = tf.keras.Sequential(
    [
        # Apply small random rotations (+/- 5.4 degrees)
        tf.keras.layers.RandomRotation(
            factor=0.015,
            fill_mode="constant",
            fill_value=0.0,
            seed=42
        ),

        # Apply small horizontal and vertical translations
        tf.keras.layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=43
        ),

        # Apply small random zoom transformations
        tf.keras.layers.RandomZoom(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=44
        )
    ],
    name="training_data_augmentation"
)

print("=== DATA AUGMENTATION PIPELINE ===")
data_augmentation.summary()

# Select representative characters with different dot patterns
selected_classes = [1, 2, 3]

selected_indices = [
    np.where(y_train_cnn == class_id)[0][0]
    for class_id in selected_classes
]

# Extract original training samples
original_images = X_train_cnn[selected_indices]

# Generate four augmented versions of each image
augmented_versions = [
    data_augmentation(
        original_images,
        training=True
    ).numpy()
    for _ in range(4)
]

# Visualize original and augmented images
fig, axes = plt.subplots(
    3, 5,
    figsize=(12, 8)
)

for row, class_id in enumerate(selected_classes):

    # Display the original image
    axes[row, 0].imshow(
        original_images[row, :, :, 0],
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[row, 0].set_title(
        f"Original - Class {class_id + 1}"
    )
    axes[row, 0].axis("off")

    # Display four augmented versions
    for col in range(1, 5):

        augmented_image = augmented_versions[col - 1][
            row, :, :, 0
        ]

        axes[row, col].imshow(
            augmented_image,
            cmap="gray",
            vmin=0,
            vmax=1
        )

        axes[row, col].set_title(
            f"Augmented {col}"
        )
        axes[row, col].axis("off")

fig.suptitle(
    "AHCD - Original vs Augmented Handwritten Characters",
    fontsize=15
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# Save the visualization
FIGURES_DIR = Path(
    "/content/arabic_cnn/results/figures"
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

figure_path = (
    FIGURES_DIR / "data_augmentation_samples.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Verify augmented image properties
sample_augmented = augmented_versions[0]

assert sample_augmented.shape == original_images.shape
assert np.all(np.isfinite(sample_augmented))
assert sample_augmented.min() >= 0.0
assert sample_augmented.max() <= 1.0

print("\n=== AUGMENTATION VERIFICATION ===")
print("Original shape:", original_images.shape)
print("Augmented shape:", sample_augmented.shape)
print("Minimum pixel value:", sample_augmented.min())
print("Maximum pixel value:", sample_augmented.max())

print("\nData augmentation pipeline created successfully!")
print("Figure saved to:", figure_path)


### Augmentation Boundary Safety Check

In [ ]:
import numpy as np

# Select a reproducible sample of training images
rng = np.random.default_rng(42)

sample_indices = rng.choice(
    len(X_train_cnn),
    size=1000,
    replace=False
)

original_samples = X_train_cnn[sample_indices]

# Generate augmented samples
augmented_samples = data_augmentation(
    original_samples,
    training=True
).numpy()

# Check whether visible strokes touch image boundaries
def touches_boundary(images, threshold=0.2):

    mask = images[..., 0] > threshold

    return (
        mask[:, 0, :].any(axis=1) |
        mask[:, -1, :].any(axis=1) |
        mask[:, :, 0].any(axis=1) |
        mask[:, :, -1].any(axis=1)
    )

original_boundary = touches_boundary(original_samples)
augmented_boundary = touches_boundary(augmented_samples)

# Identify newly detected boundary contacts
new_boundary_contacts = (
    ~original_boundary & augmented_boundary
)

print("=== AUGMENTATION BOUNDARY ANALYSIS ===")

print("Analyzed images:", len(original_samples))

print(
    "Original images touching boundaries:",
    int(original_boundary.sum())
)

print(
    "Augmented images touching boundaries:",
    int(augmented_boundary.sum())
)

print(
    "New boundary contacts after augmentation:",
    int(new_boundary_contacts.sum())
)

print(
    "New boundary contact rate:",
    round(new_boundary_contacts.mean() * 100, 2), "%"
)

print("\nBoundary safety analysis completed!")


## Data Pipeline Verification

In [ ]:
#TENSORFLOW DATA PIPELINE AND FINAL VERIFICATION


import tensorflow as tf
import numpy as np

# Define training configuration
BATCH_SIZE = 64
RANDOM_STATE = 42
AUTOTUNE = tf.data.AUTOTUNE

# Set random seeds for reproducibility
tf.keras.utils.set_random_seed(RANDOM_STATE)

# Create the training dataset
train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train_cnn, y_train_cnn)
)

# Shuffle training samples
train_ds = train_ds.shuffle(
    buffer_size=len(X_train_cnn),
    seed=RANDOM_STATE,
    reshuffle_each_iteration=True
)

# Create mini-batches and optimize data loading
train_ds = (
    train_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Create validation dataset without shuffling
val_ds = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Prepare the untouched test dataset for final evaluation
test_ds = (
    tf.data.Dataset.from_tensor_slices(
        (X_test_cnn, y_test_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ============================================================
# DATA PIPELINE VERIFICATION
# ============================================================

print("=== DATASET INFORMATION ===")

print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Testing samples:", len(X_test_cnn))

print("\n=== BATCH CONFIGURATION ===")
print("Batch size:", BATCH_SIZE)

print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))
print("Testing batches:", len(test_ds))

# Inspect one training batch
batch_images, batch_labels = next(iter(train_ds))

print("\n=== SAMPLE BATCH VERIFICATION ===")

print("Image batch shape:", batch_images.shape)
print("Label batch shape:", batch_labels.shape)

print("Image data type:", batch_images.dtype)
print("Label data type:", batch_labels.dtype)

print("Minimum pixel value:", tf.reduce_min(batch_images).numpy())
print("Maximum pixel value:", tf.reduce_max(batch_images).numpy())

# Validate batch dimensions and pixel values
assert batch_images.shape == (BATCH_SIZE, 32, 32, 1)
assert batch_labels.shape == (BATCH_SIZE,)

assert bool(tf.reduce_all(tf.math.is_finite(batch_images)))
assert float(tf.reduce_min(batch_images)) >= 0.0
assert float(tf.reduce_max(batch_images)) <= 1.0

# Verify encoded labels
assert int(tf.reduce_min(batch_labels)) >= 0
assert int(tf.reduce_max(batch_labels)) < NUM_CLASSES

print("\n=== FINAL VERIFICATION ===")
print("Training pipeline: Ready")
print("Validation pipeline: Ready")
print("Test pipeline: Reserved for final evaluation")

print("\nData pipeline verification completed successfully!")


## Save the Preprocessed Dataset to Google Drive

In [ ]:
#SAVE PREPROCESSED DATA TO GOOGLE DRIVE


from google.colab import drive
from pathlib import Path
import numpy as np
import json

# Mount Google Drive
drive.mount("/content/drive")

# Define project directories
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Define checkpoint file path
checkpoint_path = (
    CHECKPOINT_DIR / "preprocessed_ahcd_v1.npz"
)

# Save image tensors, labels, and split indices
np.savez_compressed(
    checkpoint_path,
    X_train=X_train_cnn,
    y_train=y_train_cnn,
    X_val=X_val_cnn,
    y_val=y_val_cnn,
    X_test=X_test_cnn,
    y_test=y_test_cnn,
    train_indices=train_indices,
    val_indices=val_indices
)

# Save preprocessing configuration
config = {
    "dataset": "AHCD",
    "num_classes": 28,
    "image_shape": [32, 32, 1],
    "normalization": "divide_by_255",
    "orientation_correction": "transpose",
    "label_encoding": "original_label_minus_1",
    "validation_fraction": 0.15,
    "random_state": 42,
    "batch_size": 64,
    "removed_original_training_indices": [6050],
    "augmentation": {
        "rotation_factor": 0.015,
        "translation_factor": 0.04,
        "zoom_factor": 0.04
    },
    "arabic_characters": arabic_characters
}

config_path = CHECKPOINT_DIR / "preprocessing_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(config, file, ensure_ascii=False, indent=4)

# Verify that the saved files exist
assert checkpoint_path.exists()
assert config_path.exists()

print("=== CHECKPOINT SAVE SUMMARY ===")
print("Dataset checkpoint:", checkpoint_path)
print("Configuration file:", config_path)

print(
    "Checkpoint size:",
    round(checkpoint_path.stat().st_size / (1024 ** 2), 2),
    "MB"
)

# Verify stored array shapes
with np.load(checkpoint_path, allow_pickle=False) as data:
    print("\n=== SAVED DATASET VERIFICATION ===")
    for name in [
        "X_train", "y_train",
        "X_val", "y_val",
        "X_test", "y_test"
    ]:
        print(name, data[name].shape)

print("\nProject checkpoint saved successfully!")


# Baseline CNN Development

## Build the Baseline CNN

In [ ]:
import tensorflow as tf
import numpy as np

# Set the random seed for reproducibility
RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)

# Define the input shape and number of classes
INPUT_SHAPE = (32, 32, 1)
NUM_CLASSES = 28


def build_baseline_cnn(input_shape, num_classes):

    # Input layer
    inputs = tf.keras.Input(
        shape=input_shape,
        name="input_image"
    )

    # --------------------------------------------------------
    # CONVOLUTIONAL BLOCK 1
    # --------------------------------------------------------
    x = tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False,
        name="conv_block1"
    )(inputs)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2)
    )(x)

    # --------------------------------------------------------
    # CONVOLUTIONAL BLOCK 2
    # --------------------------------------------------------
    x = tf.keras.layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False,
        name="conv_block2"
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2)
    )(x)

    # --------------------------------------------------------
    # CONVOLUTIONAL BLOCK 3
    # --------------------------------------------------------
    x = tf.keras.layers.Conv2D(
        filters=128,
        kernel_size=(3, 3),
        padding="same",
        use_bias=False,
        name="conv_block3"
    )(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    # --------------------------------------------------------
    # CLASSIFICATION HEAD
    # --------------------------------------------------------

    # Convert feature maps into a feature vector
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # Fully connected layer
    x = tf.keras.layers.Dense(
        units=64,
        activation="relu",
        name="dense_features"
    )(x)

    # Regularization
    x = tf.keras.layers.Dropout(
        rate=0.30,
        name="dropout"
    )(x)

    # Output layer for 28 Arabic character classes
    outputs = tf.keras.layers.Dense(
        units=num_classes,
        activation="softmax",
        name="class_probabilities"
    )(x)

    # Create the CNN model
    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="AHCD_Baseline_CNN"
    )

    return model


# Build the baseline model
baseline_cnn = build_baseline_cnn(
    INPUT_SHAPE,
    NUM_CLASSES
)

# Display the model architecture
print("=== BASELINE CNN ARCHITECTURE ===")
baseline_cnn.summary()

# Display model parameter information
print("\n=== MODEL INFORMATION ===")
print("Model name:", baseline_cnn.name)
print("Input shape:", baseline_cnn.input_shape)
print("Output shape:", baseline_cnn.output_shape)
print("Total parameters:", baseline_cnn.count_params())

# Verify the model output using dummy inputs
dummy_images = tf.zeros(
    shape=(2, 32, 32, 1),
    dtype=tf.float32
)

dummy_predictions = baseline_cnn(
    dummy_images,
    training=False
)

assert dummy_predictions.shape == (2, NUM_CLASSES)

assert np.allclose(
    tf.reduce_sum(dummy_predictions, axis=1).numpy(),
    1.0,
    atol=1e-6
)

print("\n=== MODEL VERIFICATION ===")
print("Prediction shape:", dummy_predictions.shape)
print("Softmax probability sums:",
      tf.reduce_sum(dummy_predictions, axis=1).numpy())

print("\nBaseline CNN architecture created successfully!")


## Model Compilation and Training Configuration

### Compile the Baseline CNN

In [ ]:
import tensorflow as tf

# Set the initial training configuration
LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS = 30
RANDOM_STATE = 42

# Set the random seed for reproducibility
tf.keras.utils.set_random_seed(RANDOM_STATE)

# Define the Adam optimizer
optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

# Define the loss function
loss_function = (
    tf.keras.losses.SparseCategoricalCrossentropy()
)

# Define evaluation metrics
metrics = [
    tf.keras.metrics.SparseCategoricalAccuracy(
        name="accuracy"
    ),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(
        k=3,
        name="top3_accuracy"
    )
]

# Compile the baseline CNN
baseline_cnn.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=metrics
)

# Display the training configuration
print("=== TRAINING CONFIGURATION ===")

print("Model:", baseline_cnn.name)
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)
print("Loss function: SparseCategoricalCrossentropy")
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)
print("Number of classes:", NUM_CLASSES)

print("\n=== EVALUATION METRICS ===")
print("Top-1 Accuracy")
print("Top-3 Accuracy")

# Verify model compilation
assert baseline_cnn.optimizer is not None
assert baseline_cnn.loss is not None

print("\n=== COMPILATION VERIFICATION ===")
print("Model optimizer:", type(baseline_cnn.optimizer).__name__)
print("Model compiled successfully!")


## Baseline CNN Training

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

# ============================================================
# 1. DEFINE EXPERIMENT DIRECTORIES
# ============================================================

# Use the Google Drive project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "logs"

# Create directories if they do not exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Define paths for model checkpoints and training logs
baseline_model_path = (
    MODEL_DIR / "baseline_cnn_best.keras"
)

baseline_log_path = (
    LOG_DIR / "baseline_training_history.csv"
)

# ============================================================
# 2. DEFINE TRAINING CALLBACKS
# ============================================================

# Save the model with the lowest validation loss
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(baseline_model_path),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

# Stop training after 6 epochs without improvement
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

# Reduce the learning rate when validation loss stagnates
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    mode="min",
    min_lr=1e-5,
    verbose=1
)

# Save training metrics after each epoch
csv_logger = tf.keras.callbacks.CSVLogger(
    filename=str(baseline_log_path),
    append=False
)

# Stop training if the loss becomes NaN or infinite
terminate_on_nan = tf.keras.callbacks.TerminateOnNaN()

callbacks = [
    model_checkpoint,
    early_stopping,
    reduce_lr,
    csv_logger,
    terminate_on_nan
]

# ============================================================
# 3. TRAIN THE BASELINE CNN
# ============================================================

print("=== BASELINE CNN TRAINING ===")

print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)

print("\nStarting model training...\n")

# Train without data augmentation for the baseline experiment
history_baseline = baseline_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# 4. SUMMARIZE TRAINING RESULTS
# ============================================================

# Identify the epoch with the lowest validation loss
best_epoch_index = int(
    np.argmin(history_baseline.history["val_loss"])
)

best_epoch = best_epoch_index + 1

# Extract the metrics from the best epoch
best_val_loss = history_baseline.history[
    "val_loss"
][best_epoch_index]

best_val_accuracy = history_baseline.history[
    "val_accuracy"
][best_epoch_index]

best_val_top3 = history_baseline.history[
    "val_top3_accuracy"
][best_epoch_index]

print("\n=== TRAINING SUMMARY ===")

print(
    "Completed epochs:",
    len(history_baseline.history["loss"])
)

print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    round(best_val_loss, 4)
)

print(
    "Validation accuracy at best epoch:",
    round(best_val_accuracy * 100, 2), "%"
)

print(
    "Validation Top-3 accuracy at best epoch:",
    round(best_val_top3 * 100, 2), "%"
)

print("\n=== SAVED FILES ===")

print("Model checkpoint:", baseline_model_path)
print("Training history:", baseline_log_path)

assert baseline_model_path.exists()
assert baseline_log_path.exists()

print("\nBaseline training completed successfully!")


## Training History Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. LOAD TRAINING HISTORY
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

history_path = (
    PROJECT_DIR / "logs/baseline_training_history.csv"
)

history_df = pd.read_csv(history_path)

# Identify training epochs
epochs = np.arange(1, len(history_df) + 1)

# Identify the epoch with the lowest validation loss
best_idx = history_df["val_loss"].idxmin()
best_epoch = best_idx + 1

print("=== TRAINING HISTORY INFORMATION ===")
print("Total epochs:", len(history_df))
print("Best epoch:", best_epoch)
print("Available metrics:", history_df.columns.tolist())

# ============================================================
# 2. CREATE TRAINING VISUALIZATIONS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ------------------------------------------------------------
# PLOT 1: TRAINING AND VALIDATION ACCURACY
# ------------------------------------------------------------

axes[0, 0].plot(
    epochs,
    history_df["accuracy"],
    label="Training Accuracy",
    linewidth=2
)

axes[0, 0].plot(
    epochs,
    history_df["val_accuracy"],
    label="Validation Accuracy",
    linewidth=2
)

axes[0, 0].axvline(
    best_epoch,
    linestyle="--",
    color="gray",
    label=f"Best Epoch ({best_epoch})"
)

axes[0, 0].set_title("Training vs Validation Accuracy")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# ------------------------------------------------------------
# PLOT 2: TRAINING AND VALIDATION LOSS
# ------------------------------------------------------------

axes[0, 1].plot(
    epochs,
    history_df["loss"],
    label="Training Loss",
    linewidth=2
)

axes[0, 1].plot(
    epochs,
    history_df["val_loss"],
    label="Validation Loss",
    linewidth=2
)

axes[0, 1].axvline(
    best_epoch,
    linestyle="--",
    color="gray",
    label=f"Best Epoch ({best_epoch})"
)

axes[0, 1].set_title("Training vs Validation Loss")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# ------------------------------------------------------------
# PLOT 3: TOP-3 ACCURACY
# ------------------------------------------------------------

axes[1, 0].plot(
    epochs,
    history_df["top3_accuracy"],
    label="Training Top-3 Accuracy",
    linewidth=2
)

axes[1, 0].plot(
    epochs,
    history_df["val_top3_accuracy"],
    label="Validation Top-3 Accuracy",
    linewidth=2
)

axes[1, 0].set_title("Training vs Validation Top-3 Accuracy")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Top-3 Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# ------------------------------------------------------------
# PLOT 4: LEARNING RATE EVOLUTION
# ------------------------------------------------------------

if "learning_rate" in history_df.columns:

    axes[1, 1].plot(
        epochs,
        history_df["learning_rate"],
        marker="o",
        markersize=3,
        linewidth=2
    )

    axes[1, 1].set_yscale("log")
    axes[1, 1].set_title("Learning Rate Evolution")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Learning Rate")

else:
    axes[1, 1].text(
        0.5, 0.5,
        "Learning rate history unavailable",
        ha="center",
        va="center"
    )

    axes[1, 1].set_title("Learning Rate Evolution")

axes[1, 1].grid(alpha=0.3)

# ============================================================
# 3. SAVE VISUALIZATION
# ============================================================

fig.suptitle(
    "AHCD - Baseline CNN Training Analysis",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

FIGURES_DIR = PROJECT_DIR / "results/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

figure_path = (
    FIGURES_DIR / "baseline_training_curves.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 4. FINAL VERIFICATION
# ============================================================

print("\n=== TRAINING HISTORY SUMMARY ===")
print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    round(history_df.loc[best_idx, "val_loss"], 4)
)

print(
    "Validation accuracy at best epoch:",
    round(
        history_df.loc[best_idx, "val_accuracy"] * 100,
        2
    ),
    "%"
)

print("\nFigure saved to:", figure_path)
print("Training history visualization completed!")


## Validation Error Analysis and Confusion Matrix

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score
)

# ============================================================
# 1. LOAD THE BEST BASELINE MODEL
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

model_path = PROJECT_DIR / "models/baseline_cnn_best.keras"

best_baseline = tf.keras.models.load_model(model_path)

print("=== MODEL LOADING ===")
print("Model loaded successfully:", best_baseline.name)

# ============================================================
# 2. GENERATE VALIDATION PREDICTIONS
# ============================================================

# Predict class probabilities on validation images only
y_prob = best_baseline.predict(
    X_val_cnn,
    batch_size=64,
    verbose=1
)

# Extract the most probable class
y_pred = np.argmax(y_prob, axis=1)

# Extract true validation labels
y_true = np.asarray(y_val_cnn)

assert len(y_pred) == len(y_true)
assert y_prob.shape == (len(y_true), NUM_CLASSES)

# ============================================================
# 3. COMPUTE VALIDATION METRICS
# ============================================================

accuracy = np.mean(y_pred == y_true)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

incorrect_predictions = np.sum(y_pred != y_true)

print("\n=== VALIDATION METRICS ===")

print("Validation accuracy:", round(accuracy * 100, 2), "%")
print("Macro F1-score:", round(macro_f1, 4))
print("Weighted F1-score:", round(weighted_f1, 4))
print("Incorrect predictions:", incorrect_predictions)
print("Total validation samples:", len(y_true))

# ============================================================
# 4. COMPUTE CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES)
)

fig, ax = plt.subplots(figsize=(13, 11))

im = ax.imshow(cm, interpolation="nearest", cmap="Blues")

plt.colorbar(im, ax=ax)

ax.set_title(
    "AHCD - Baseline CNN Validation Confusion Matrix",
    fontsize=15
)

ax.set_xlabel("Predicted Class")
ax.set_ylabel("True Class")

ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_yticks(np.arange(NUM_CLASSES))

ax.set_xticklabels(np.arange(1, NUM_CLASSES + 1))
ax.set_yticklabels(np.arange(1, NUM_CLASSES + 1))

plt.setp(
    ax.get_xticklabels(),
    rotation=90,
    fontsize=8
)

plt.setp(
    ax.get_yticklabels(),
    fontsize=8
)

plt.tight_layout()

# Save the confusion matrix
FIGURES_DIR = PROJECT_DIR / "results/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

figure_path = (
    FIGURES_DIR / "baseline_validation_confusion_matrix.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 5. IDENTIFY MOST CONFUSED CLASS PAIRS
# ============================================================

confusion_pairs = []

for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):

        if true_class != predicted_class:

            count = cm[true_class, predicted_class]

            if count > 0:
                confusion_pairs.append({
                    "True Class": true_class + 1,
                    "Predicted Class": predicted_class + 1,
                    "True Character": arabic_characters[true_class],
                    "Predicted Character": arabic_characters[predicted_class],
                    "Errors": int(count)
                })

confusion_df = pd.DataFrame(confusion_pairs)

confusion_df = confusion_df.sort_values(
    by="Errors",
    ascending=False
)

print("\n=== TOP 10 MOST CONFUSED CLASS PAIRS ===")
print(confusion_df.head(10).to_string(index=False))

print("\nValidation error analysis completed successfully!")
print("Figure saved to:", figure_path)


## Visual Inspection of Misclassified Images

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. IDENTIFY MISCLASSIFIED VALIDATION IMAGES
# ============================================================

# Find indices where predictions differ from true labels
wrong_indices = np.where(y_pred != y_true)[0]

# Extract the predicted class confidence
predicted_confidence = y_prob[
    np.arange(len(y_true)), y_pred
]

# Extract the probability assigned to the true class
true_class_probability = y_prob[
    np.arange(len(y_true)), y_true
]

print("=== MISCLASSIFICATION SUMMARY ===")
print("Total validation images:", len(y_true))
print("Incorrect predictions:", len(wrong_indices))

# ============================================================
# 2. SELECT HIGH-CONFIDENCE ERRORS
# ============================================================

# Sort incorrect predictions by model confidence
sorted_wrong_indices = wrong_indices[
    np.argsort(
        predicted_confidence[wrong_indices]
    )[::-1]
]

# Select the 12 most confident incorrect predictions
selected_indices = sorted_wrong_indices[:12]

# ============================================================
# 3. VISUALIZE MISCLASSIFIED IMAGES
# ============================================================

fig, axes = plt.subplots(
    3, 4,
    figsize=(14, 11)
)

for ax, idx in zip(axes.flat, selected_indices):

    # Extract the incorrectly classified image
    image = X_val_cnn[idx, :, :, 0]

    # Display the image
    ax.imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    # Retrieve true and predicted class numbers
    true_class = y_true[idx] + 1
    predicted_class = y_pred[idx] + 1

    # Retrieve prediction confidence
    confidence = predicted_confidence[idx] * 100

    ax.set_title(
        f"True: {true_class} | Predicted: {predicted_class}",
        fontsize=11
    )

    ax.set_xlabel(
        f"Wrong prediction confidence: {confidence:.1f}%",
        fontsize=9
    )

    ax.set_xticks([])
    ax.set_yticks([])

for ax in axes.flat[len(selected_indices):]:
    ax.axis("off")

fig.suptitle(
    "AHCD - High-Confidence Misclassified Validation Images",
    fontsize=15
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# ============================================================
# 4. SAVE THE VISUALIZATION
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

FIGURES_DIR = PROJECT_DIR / "results/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

figure_path = (
    FIGURES_DIR / "baseline_misclassified_images.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 5. EXPORT ERROR DETAILS
# ============================================================

error_data = []

for idx in wrong_indices:

    error_data.append({
        "Validation Index": int(idx),
        "True Class": int(y_true[idx] + 1),
        "Predicted Class": int(y_pred[idx] + 1),
        "True Character": arabic_characters[y_true[idx]],
        "Predicted Character": arabic_characters[y_pred[idx]],
        "Prediction Confidence": float(
            predicted_confidence[idx]
        ),
        "True Class Probability": float(
            true_class_probability[idx]
        )
    })

error_df = pd.DataFrame(error_data)

error_df = error_df.sort_values(
    "Prediction Confidence",
    ascending=False
)

RESULTS_DIR = PROJECT_DIR / "results/tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

csv_path = RESULTS_DIR / "baseline_validation_errors.csv"

error_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n=== TOP 12 HIGH-CONFIDENCE ERRORS ===")
print(error_df.head(12).to_string(index=False))

print("\n=== SAVED FILES ===")
print("Visualization:", figure_path)
print("Error analysis table:", csv_path)

print("\nMisclassification analysis completed successfully!")


## Experimental Comparison

### Build the CNN with Data Augmentation

In [ ]:
import tensorflow as tf
import numpy as np

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

RANDOM_STATE = 42
INPUT_SHAPE = (32, 32, 1)
NUM_CLASSES = 28
LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS = 30

# Set the random seed for reproducibility
tf.keras.utils.set_random_seed(RANDOM_STATE)

# ============================================================
# 2. CREATE A FRESH BASELINE CNN
# ============================================================

# Initialize a new CNN with untrained weights
augmented_backbone = build_baseline_cnn(
    INPUT_SHAPE,
    NUM_CLASSES
)

# ============================================================
# 3. DEFINE TRAINING DATA AUGMENTATION
# ============================================================

augmentation_pipeline = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=0.015,
            fill_mode="constant",
            fill_value=0.0,
            seed=42
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=43
        ),

        tf.keras.layers.RandomZoom(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=44
        )
    ],
    name="data_augmentation"
)

# ============================================================
# 4. BUILD THE AUGMENTED MODEL
# ============================================================

inputs = tf.keras.Input(
    shape=INPUT_SHAPE,
    name="input_image"
)

# Apply augmentation only during training
x = augmentation_pipeline(inputs)

# Pass augmented images through the CNN
outputs = augmented_backbone(x)

# Build the complete model
augmented_cnn = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="AHCD_Augmented_CNN"
)

# ============================================================
# 5. COMPILE THE MODEL
# ============================================================

augmented_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

# ============================================================
# 6. VERIFY THE MODEL
# ============================================================

print("=== AUGMENTED CNN ARCHITECTURE ===")
augmented_cnn.summary()

# Verify that augmentation is disabled during inference
sample_images = tf.convert_to_tensor(
    X_train_cnn[:2]
)

inference_images = augmentation_pipeline(
    sample_images,
    training=False
)

assert np.allclose(
    sample_images.numpy(),
    inference_images.numpy()
)

# Verify prediction shape and probability sums
dummy_predictions = augmented_cnn(
    sample_images,
    training=False
)

assert dummy_predictions.shape == (2, NUM_CLASSES)

assert np.allclose(
    tf.reduce_sum(
        dummy_predictions, axis=1
    ).numpy(),
    1.0,
    atol=1e-6
)

# Verify that both experiments use the same CNN size
assert (
    augmented_cnn.count_params()
    == baseline_cnn.count_params()
)

print("\n=== EXPERIMENT CONFIGURATION ===")
print("Model:", augmented_cnn.name)
print("Input shape:", augmented_cnn.input_shape)
print("Output shape:", augmented_cnn.output_shape)
print("Total parameters:", augmented_cnn.count_params())
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)

print("\n=== AUGMENTATION VERIFICATION ===")
print("Training augmentation: Enabled")
print("Inference augmentation: Disabled")
print("CNN architecture: Same as baseline")
print("Model weights: Freshly initialized")

print("\nAugmented CNN created and compiled successfully!")


## Train the Augmented CNN

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

# ============================================================
# 1. DEFINE EXPERIMENT DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "logs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Use separate files to preserve baseline results
augmented_model_path = (
    MODEL_DIR / "augmented_cnn_best.keras"
)

augmented_log_path = (
    LOG_DIR / "augmented_training_history.csv"
)

# ============================================================
# 2. DEFINE TRAINING CALLBACKS
# ============================================================

# Save the model with the lowest validation loss
augmented_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(augmented_model_path),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

# Stop training when validation loss stops improving
augmented_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

# Reduce the learning rate when validation loss stagnates
augmented_reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    mode="min",
    min_lr=1e-5,
    verbose=1
)

# Save training metrics for each epoch
augmented_csv_logger = tf.keras.callbacks.CSVLogger(
    filename=str(augmented_log_path),
    append=False
)

# Stop training if the loss becomes NaN or infinite
augmented_terminate_nan = (
    tf.keras.callbacks.TerminateOnNaN()
)

augmented_callbacks = [
    augmented_checkpoint,
    augmented_early_stopping,
    augmented_reduce_lr,
    augmented_csv_logger,
    augmented_terminate_nan
]

# ============================================================
# 3. TRAIN THE AUGMENTED CNN
# ============================================================

print("=== AUGMENTED CNN TRAINING ===")

print("Experiment: CNN with Data Augmentation")
print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)
print("Data augmentation: Enabled during training")

print("\nStarting augmented CNN training...\n")

# Train the new model using the same training dataset
history_augmented = augmented_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=augmented_callbacks,
    verbose=1
)

# ============================================================
# 4. SUMMARIZE THE TRAINING RESULTS
# ============================================================

# Identify the epoch with the lowest validation loss
best_epoch_index = int(
    np.argmin(history_augmented.history["val_loss"])
)

best_epoch = best_epoch_index + 1

# Extract validation metrics at the best epoch
best_val_loss = history_augmented.history[
    "val_loss"
][best_epoch_index]

best_val_accuracy = history_augmented.history[
    "val_accuracy"
][best_epoch_index]

best_val_top3 = history_augmented.history[
    "val_top3_accuracy"
][best_epoch_index]

print("\n=== AUGMENTED CNN TRAINING SUMMARY ===")

print(
    "Completed epochs:",
    len(history_augmented.history["loss"])
)

print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    round(best_val_loss, 4)
)

print(
    "Validation accuracy at best epoch:",
    round(best_val_accuracy * 100, 2), "%"
)

print(
    "Validation Top-3 accuracy at best epoch:",
    round(best_val_top3 * 100, 2), "%"
)

# ============================================================
# 5. VERIFY SAVED FILES
# ============================================================

print("\n=== SAVED FILES ===")

print("Best model:", augmented_model_path)
print("Training history:", augmented_log_path)

assert augmented_model_path.exists()
assert augmented_log_path.exists()

print("\nAugmented CNN training completed successfully!")


## Baseline CNN vs Augmented CNN Comparison

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, log_loss

# ============================================================
# 1. LOAD THE BEST SAVED MODELS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

baseline_model = tf.keras.models.load_model(
    PROJECT_DIR / "models/baseline_cnn_best.keras"
)

augmented_model = tf.keras.models.load_model(
    PROJECT_DIR / "models/augmented_cnn_best.keras"
)

print("Both models loaded successfully!")

# ============================================================
# 2. GENERATE VALIDATION PREDICTIONS
# ============================================================

y_true = np.asarray(y_val_cnn).ravel()

baseline_probs = baseline_model.predict(
    X_val_cnn, batch_size=64, verbose=0
)

augmented_probs = augmented_model.predict(
    X_val_cnn, batch_size=64, verbose=0
)

baseline_preds = np.argmax(baseline_probs, axis=1)
augmented_preds = np.argmax(augmented_probs, axis=1)

# ============================================================
# 3. COMPUTE COMPARISON METRICS
# ============================================================

def calculate_metrics(y_true, probabilities):

    predictions = np.argmax(probabilities, axis=1)

    top3_indices = np.argsort(
        probabilities, axis=1
    )[:, -3:]

    top3_accuracy = np.mean(
        np.any(top3_indices == y_true[:, None], axis=1)
    )

    return {
        "Accuracy": accuracy_score(
            y_true, predictions
        ) * 100,

        "Macro F1": f1_score(
            y_true, predictions, average="macro"
        ) * 100,

        "Top-3 Accuracy": top3_accuracy * 100,

        "Loss": log_loss(
            y_true,
            probabilities,
            labels=np.arange(28)
        ),

        "Incorrect Predictions": int(
            np.sum(y_true != predictions)
        )
    }


baseline_metrics = calculate_metrics(
    y_true, baseline_probs
)

augmented_metrics = calculate_metrics(
    y_true, augmented_probs
)

# Build the comparison table
comparison_df = pd.DataFrame({
    "Metric": list(baseline_metrics.keys()),
    "Baseline CNN": list(baseline_metrics.values()),
    "Augmented CNN": list(augmented_metrics.values())
})

print("\n=== MODEL COMPARISON ===")
print(comparison_df.to_string(index=False))

# ============================================================
# 4. ANALYZE PAIRED PREDICTIONS
# ============================================================

baseline_correct = baseline_preds == y_true
augmented_correct = augmented_preds == y_true

both_correct = np.sum(
    baseline_correct & augmented_correct
)

baseline_only = np.sum(
    baseline_correct & ~augmented_correct
)

augmented_only = np.sum(
    ~baseline_correct & augmented_correct
)

both_wrong = np.sum(
    ~baseline_correct & ~augmented_correct
)

print("\n=== PAIRED PREDICTION ANALYSIS ===")

print("Correct by both models:", both_correct)
print("Correct only by baseline:", baseline_only)
print("Correct only by augmented CNN:", augmented_only)
print("Incorrect by both models:", both_wrong)

assert (
    both_correct + baseline_only
    + augmented_only + both_wrong
) == len(y_true)

# ============================================================
# 5. VISUALIZE MODEL PERFORMANCE
# ============================================================

metric_names = [
    "Accuracy",
    "Macro F1",
    "Top-3 Accuracy"
]

baseline_values = [
    baseline_metrics[name] for name in metric_names
]

augmented_values = [
    augmented_metrics[name] for name in metric_names
]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    x - width / 2,
    baseline_values,
    width,
    label="Baseline CNN",
    color="steelblue"
)

ax.bar(
    x + width / 2,
    augmented_values,
    width,
    label="Augmented CNN",
    color="mediumseagreen"
)

ax.set_title(
    "AHCD - Baseline vs Augmented CNN"
)

ax.set_ylabel("Score (%)")
ax.set_ylim(0, 105)

ax.set_xticks(x)
ax.set_xticklabels(metric_names)

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

# ============================================================
# 6. SAVE COMPARISON RESULTS
# ============================================================

RESULTS_DIR = PROJECT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

comparison_path = (
    TABLES_DIR / "cnn_augmentation_comparison.csv"
)

figure_path = (
    FIGURES_DIR / "cnn_augmentation_comparison.png"
)

comparison_df.to_csv(
    comparison_path, index=False
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 7. DISPLAY FINAL COMPARISON
# ============================================================

accuracy_difference = (
    augmented_metrics["Accuracy"]
    - baseline_metrics["Accuracy"]
)

print("\n=== FINAL COMPARISON SUMMARY ===")

print(
    "Augmented CNN accuracy difference:",
    round(accuracy_difference, 2),
    "percentage points"
)

print("Comparison table:", comparison_path)
print("Comparison figure:", figure_path)

print("\nExperimental comparison completed successfully!")


## Per-Class Performance Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score

# ============================================================
# 1. COMPUTE PER-CLASS METRICS
# ============================================================

# Use predictions from Step 6.3
class_ids = np.arange(NUM_CLASSES)

baseline_f1 = f1_score(
    y_true,
    baseline_preds,
    labels=class_ids,
    average=None,
    zero_division=0
)

augmented_f1 = f1_score(
    y_true,
    augmented_preds,
    labels=class_ids,
    average=None,
    zero_division=0
)

results = []

for class_id in class_ids:

    # Identify validation samples of the current class
    mask = y_true == class_id
    sample_count = int(mask.sum())

    # Calculate class-specific accuracy
    baseline_acc = np.mean(
        baseline_preds[mask] == y_true[mask]
    ) * 100

    augmented_acc = np.mean(
        augmented_preds[mask] == y_true[mask]
    ) * 100

    results.append({
        "Class": class_id + 1,
        "Character": arabic_characters[class_id],
        "Samples": sample_count,
        "Baseline Accuracy (%)": baseline_acc,
        "Augmented Accuracy (%)": augmented_acc,
        "Accuracy Difference (pp)": (
            augmented_acc - baseline_acc
        ),
        "Baseline F1": baseline_f1[class_id],
        "Augmented F1": augmented_f1[class_id],
        "Baseline Errors": int(
            np.sum(baseline_preds[mask] != y_true[mask])
        ),
        "Augmented Errors": int(
            np.sum(augmented_preds[mask] != y_true[mask])
        )
    })

per_class_df = pd.DataFrame(results)

# Verify all classes are included
assert len(per_class_df) == 28
assert per_class_df["Samples"].sum() == len(y_true)

print("=== PER-CLASS PERFORMANCE ===")
print(per_class_df.round(3).to_string(index=False))

# ============================================================
# 2. IDENTIFY IMPROVED AND DEGRADED CLASSES
# ============================================================

print("\n=== TOP 5 IMPROVED CLASSES ===")

print(
    per_class_df.nlargest(
        5, "Accuracy Difference (pp)"
    )[
        [
            "Class",
            "Character",
            "Baseline Accuracy (%)",
            "Augmented Accuracy (%)",
            "Accuracy Difference (pp)"
        ]
    ].round(2).to_string(index=False)
)

print("\n=== TOP 5 DEGRADED CLASSES ===")

print(
    per_class_df.nsmallest(
        5, "Accuracy Difference (pp)"
    )[
        [
            "Class",
            "Character",
            "Baseline Accuracy (%)",
            "Augmented Accuracy (%)",
            "Accuracy Difference (pp)"
        ]
    ].round(2).to_string(index=False)
)

# ============================================================
# 3. VISUALIZE PER-CLASS ACCURACY DIFFERENCES
# ============================================================

sorted_df = per_class_df.sort_values(
    "Accuracy Difference (pp)"
)

fig, ax = plt.subplots(figsize=(11, 10))

ax.barh(
    sorted_df["Class"].astype(str),
    sorted_df["Accuracy Difference (pp)"]
)

ax.axvline(
    0,
    color="black",
    linewidth=1
)

ax.set_title(
    "AHCD - Impact of Data Augmentation by Class"
)

ax.set_xlabel(
    "Accuracy Difference: Augmented - Baseline (pp)"
)

ax.set_ylabel("Arabic Character Class")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()

# ============================================================
# 4. SAVE RESULTS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

TABLES_DIR = PROJECT_DIR / "results/tables"
FIGURES_DIR = PROJECT_DIR / "results/figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

table_path = (
    TABLES_DIR / "augmentation_per_class_analysis.csv"
)

figure_path = (
    FIGURES_DIR / "augmentation_per_class_comparison.png"
)

per_class_df.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 5. FINAL SUMMARY
# ============================================================

differences = per_class_df["Accuracy Difference (pp)"]

print("\n=== PER-CLASS COMPARISON SUMMARY ===")

print("Classes improved:", int((differences > 0).sum()))
print("Classes degraded:", int((differences < 0).sum()))
print("Classes unchanged:", int((differences == 0).sum()))

print("\nAnalysis completed successfully!")
print("Table saved to:", table_path)
print("Figure saved to:", figure_path)


## Analyze Augmentation-Induced Errors

In [ ]:
#CLASS-LEVEL ERROR TRANSITION ANALYSIS


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. IDENTIFY PAIRED PREDICTION OUTCOMES
# ============================================================

baseline_correct = baseline_preds == y_true
augmented_correct = augmented_preds == y_true

# Define classes with the largest performance degradation
target_classes = [22, 25, 3]

comparison_results = []

for original_class in range(1, 29):

    # Convert original class ID to zero-based index
    class_id = original_class - 1

    class_mask = y_true == class_id

    # Count correct and incorrect prediction combinations
    baseline_only = np.sum(
        class_mask & baseline_correct & ~augmented_correct
    )

    augmented_only = np.sum(
        class_mask & ~baseline_correct & augmented_correct
    )

    both_wrong = np.sum(
        class_mask & ~baseline_correct & ~augmented_correct
    )

    comparison_results.append({
        "Class": original_class,
        "Character": arabic_characters[class_id],
        "Baseline Only Correct": int(baseline_only),
        "Augmented Only Correct": int(augmented_only),
        "Both Incorrect": int(both_wrong),
        "Net Accuracy Change (pp)": (
            (augmented_only - baseline_only)
            / np.sum(class_mask) * 100
        )
    })

transition_df = pd.DataFrame(comparison_results)

print("=== ERROR TRANSITION ANALYSIS ===")

print(
    transition_df[
        transition_df["Class"].isin(target_classes)
    ].round(2).to_string(index=False)
)

# ============================================================
# 2. SELECT AUGMENTATION-RELATED ERRORS
# ============================================================

fig, axes = plt.subplots(
    3, 4,
    figsize=(13, 10)
)

for row, original_class in enumerate(target_classes):

    class_id = original_class - 1

    # Find images correctly classified by the baseline
    # but incorrectly classified by the augmented model
    selected_indices = np.where(
        (y_true == class_id)
        & baseline_correct
        & ~augmented_correct
    )[0]

    # Sort by augmented model confidence
    selected_indices = selected_indices[
        np.argsort(
            augmented_probs[
                selected_indices,
                augmented_preds[selected_indices]
            ]
        )[::-1]
    ]

    # Display up to four examples per class
    selected_indices = selected_indices[:4]

    for col, idx in enumerate(selected_indices):

        ax = axes[row, col]

        ax.imshow(
            X_val_cnn[idx, :, :, 0],
            cmap="gray",
            vmin=0,
            vmax=1
        )

        confidence = augmented_probs[
            idx, augmented_preds[idx]
        ] * 100

        ax.set_title(
            f"True: {original_class} | "
            f"Pred: {augmented_preds[idx] + 1}\n"
            f"Confidence: {confidence:.1f}%",
            fontsize=10
        )

        ax.axis("off")

    # Hide unused axes
    for col in range(len(selected_indices), 4):
        axes[row, col].axis("off")

fig.suptitle(
    "Baseline-Correct vs Augmented-Incorrect Predictions",
    fontsize=15
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# ============================================================
# 3. SAVE ANALYSIS RESULTS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

FIGURES_DIR = PROJECT_DIR / "results/figures"
TABLES_DIR = PROJECT_DIR / "results/tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

figure_path = (
    FIGURES_DIR / "augmentation_error_transitions.png"
)

table_path = (
    TABLES_DIR / "augmentation_error_transitions.csv"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

transition_df.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig"
)

plt.show()

print("\n=== ANALYSIS SUMMARY ===")

print(
    "Total baseline-only correct:",
    transition_df["Baseline Only Correct"].sum()
)

print(
    "Total augmented-only correct:",
    transition_df["Augmented Only Correct"].sum()
)

print("\nFigure saved to:", figure_path)
print("Table saved to:", table_path)

print("\nError transition analysis completed successfully!")


## Build a CNN with Lighter Data Augmentation

In [ ]:
# BUILD AND VERIFY THE LIGHTLY AUGMENTED CNN
# REDUCED AUGMENTATION INTENSITY


import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

RANDOM_STATE = 42
INPUT_SHAPE = (32, 32, 1)
NUM_CLASSES = 28
LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS = 30

# Reset the random seed before model initialization
tf.keras.utils.set_random_seed(RANDOM_STATE)

# ============================================================
# 2. BUILD A FRESH CNN BACKBONE
# ============================================================

light_backbone = build_baseline_cnn(
    INPUT_SHAPE,
    NUM_CLASSES
)

# ============================================================
# 3. DEFINE LIGHT DATA AUGMENTATION
# ============================================================

light_data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=(-0.008, 0.008),
            fill_mode="constant",
            fill_value=0.0,
            seed=42
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=(-0.02, 0.02),
            width_factor=(-0.02, 0.02),
            fill_mode="constant",
            fill_value=0.0,
            seed=43
        ),

        tf.keras.layers.RandomZoom(
            height_factor=(-0.02, 0.02),
            width_factor=(-0.02, 0.02),
            fill_mode="constant",
            fill_value=0.0,
            seed=44
        )
    ],
    name="light_data_augmentation"
)

# ============================================================
# 4. BUILD THE COMPLETE MODEL
# ============================================================

inputs = tf.keras.Input(
    shape=INPUT_SHAPE,
    name="input_image"
)

x = light_data_augmentation(inputs)

outputs = light_backbone(x)

light_augmented_cnn = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="AHCD_Light_Augmented_CNN"
)

# ============================================================
# 5. COMPILE THE MODEL
# ============================================================

light_augmented_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

# ============================================================
# 6. VERIFY THE ARCHITECTURE
# ============================================================

print("=== LIGHT AUGMENTED CNN ARCHITECTURE ===")
light_augmented_cnn.summary()

assert light_augmented_cnn.count_params() == 103420
assert light_augmented_cnn.output_shape == (None, 28)

# Verify that augmentation is disabled during inference
sample_images = tf.convert_to_tensor(
    X_train_cnn[:3]
)

inference_images = light_data_augmentation(
    sample_images,
    training=False
)

assert np.allclose(
    sample_images.numpy(),
    inference_images.numpy()
)

# Verify the model output
predictions = light_augmented_cnn(
    sample_images,
    training=False
)

assert predictions.shape == (3, NUM_CLASSES)

assert np.allclose(
    tf.reduce_sum(predictions, axis=1).numpy(),
    1.0,
    atol=1e-6
)

# ============================================================
# 7. VISUALIZE LIGHT AUGMENTATION
# ============================================================

selected_classes = [2, 21, 24]

selected_indices = [
    np.where(y_train_cnn == class_id)[0][0]
    for class_id in selected_classes
]

original_images = X_train_cnn[selected_indices]

augmented_versions = [
    light_data_augmentation(
        original_images,
        training=True
    ).numpy()
    for _ in range(3)
]

fig, axes = plt.subplots(
    3, 4,
    figsize=(12, 9)
)

for row, class_id in enumerate(selected_classes):

    axes[row, 0].imshow(
        original_images[row, :, :, 0],
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[row, 0].set_title(
        f"Original - Class {class_id + 1}"
    )
    axes[row, 0].axis("off")

    for col in range(1, 4):

        axes[row, col].imshow(
            augmented_versions[col - 1][row, :, :, 0],
            cmap="gray",
            vmin=0,
            vmax=1
        )

        axes[row, col].set_title(
            f"Light Augmented {col}"
        )
        axes[row, col].axis("off")

fig.suptitle(
    "AHCD - Light Data Augmentation Visualization",
    fontsize=15
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

# ============================================================
# 8. SAVE THE VISUALIZATION
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

FIGURES_DIR = PROJECT_DIR / "results/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

figure_path = (
    FIGURES_DIR / "light_augmentation_samples.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 9. FINAL VERIFICATION
# ============================================================

print("\n=== EXPERIMENT 3 CONFIGURATION ===")

print("Model:", light_augmented_cnn.name)
print("Total parameters:", light_augmented_cnn.count_params())
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)

print("\n=== AUGMENTATION SETTINGS ===")
print("Rotation: +/- 2.88 degrees")
print("Translation: +/- 2%")
print("Zoom: +/- 2%")

print("\n=== MODEL VERIFICATION ===")
print("Inference augmentation: Disabled")
print("Architecture verification: Passed")
print("Model weights: Freshly initialized")
print("Training status: Not started")

print("\nLightly augmented CNN created successfully!")
print("Figure saved to:", figure_path)


## Train the Lightly Augmented CNN

In [ ]:
# LIGHTLY AUGMENTED CNN TRAINING
# LIGHT DATA AUGMENTATION


import tensorflow as tf
import numpy as np
from pathlib import Path

# ============================================================
# 1. DEFINE EXPERIMENT DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "logs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Define unique file paths for Experiment 3
light_model_path = (
    MODEL_DIR / "light_augmented_cnn_best.keras"
)

light_log_path = (
    LOG_DIR / "light_augmented_training_history.csv"
)

# ============================================================
# 2. DEFINE TRAINING CALLBACKS
# ============================================================

# Save the model with the lowest validation loss
light_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(light_model_path),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

# Stop training after six epochs without improvement
light_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate when validation loss stagnates
light_reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    mode="min",
    min_lr=1e-5,
    verbose=1
)

# Save the training history
light_csv_logger = tf.keras.callbacks.CSVLogger(
    filename=str(light_log_path),
    append=False
)

# Stop training if the loss becomes invalid
light_terminate_nan = (
    tf.keras.callbacks.TerminateOnNaN()
)

light_callbacks = [
    light_checkpoint,
    light_early_stopping,
    light_reduce_lr,
    light_csv_logger,
    light_terminate_nan
]

# ============================================================
# 3. TRAIN THE LIGHTLY AUGMENTED CNN
# ============================================================

print("=== LIGHTLY AUGMENTED CNN TRAINING ===")

print("Experiment: Light Data Augmentation")
print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)

print("\nStarting model training...\n")

history_light = light_augmented_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=light_callbacks,
    verbose=1
)

# ============================================================
# 4. EXTRACT BEST EPOCH RESULTS
# ============================================================

history = history_light.history

# Identify the epoch with the lowest validation loss
best_epoch_index = int(
    np.argmin(history["val_loss"])
)

best_epoch = best_epoch_index + 1

# Extract metrics at the best epoch
best_val_loss = history[
    "val_loss"
][best_epoch_index]

best_val_accuracy = history[
    "val_accuracy"
][best_epoch_index]

best_val_top3 = history[
    "val_top3_accuracy"
][best_epoch_index]

# ============================================================
# 5. DISPLAY TRAINING RESULTS
# ============================================================

print("\n=== LIGHT AUGMENTED CNN TRAINING SUMMARY ===")

print(
    "Completed epochs:",
    len(history["loss"])
)

print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    round(best_val_loss, 4)
)

print(
    "Validation accuracy at best epoch:",
    round(best_val_accuracy * 100, 2), "%"
)

print(
    "Validation Top-3 accuracy at best epoch:",
    round(best_val_top3 * 100, 2), "%"
)

# ============================================================
# 6. VERIFY SAVED FILES
# ============================================================

print("\n=== SAVED FILES ===")

print("Best model:", light_model_path)
print("Training history:", light_log_path)

assert light_model_path.exists()
assert light_log_path.exists()

print("\nLightly augmented CNN training completed successfully!")


## Compare All Three CNN Models

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss
)

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

MODEL_DIR = PROJECT_DIR / "models"

# Define the three saved model paths
model_paths = {
    "Baseline CNN": MODEL_DIR / "baseline_cnn_best.keras",
    "Standard Augmentation": MODEL_DIR / "augmented_cnn_best.keras",
    "Light Augmentation": MODEL_DIR / "light_augmented_cnn_best.keras"
}

# ============================================================
# 2. LOAD MODELS AND GENERATE VALIDATION PREDICTIONS
# ============================================================

y_true = np.asarray(y_val_cnn).ravel()

model_probabilities = {}
model_predictions = {}

for model_name, model_path in model_paths.items():

    # Load the best saved model
    model = tf.keras.models.load_model(model_path)

    # Generate predictions on the validation set
    probabilities = model.predict(
        X_val_cnn,
        batch_size=64,
        verbose=0
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    assert probabilities.shape == (
        len(y_true), NUM_CLASSES
    )

    model_probabilities[model_name] = probabilities
    model_predictions[model_name] = predictions

    print(f"{model_name}: predictions completed")

print("\nAll models evaluated on the same validation set!")

# ============================================================
# 3. CALCULATE EVALUATION METRICS
# ============================================================

comparison_results = []

for model_name, probabilities in model_probabilities.items():

    predictions = model_predictions[model_name]

    # Calculate classification accuracy
    accuracy = accuracy_score(
        y_true, predictions
    ) * 100

    # Calculate macro F1-score
    macro_f1 = f1_score(
        y_true,
        predictions,
        average="macro"
    ) * 100

    # Calculate Top-3 accuracy
    top3_indices = np.argsort(
        probabilities,
        axis=1
    )[:, -3:]

    top3_accuracy = np.mean(
        np.any(
            top3_indices == y_true[:, None],
            axis=1
        )
    ) * 100

    # Calculate categorical cross-entropy
    validation_loss = log_loss(
        y_true,
        probabilities,
        labels=np.arange(NUM_CLASSES)
    )

    # Count incorrect predictions
    errors = int(
        np.sum(predictions != y_true)
    )

    comparison_results.append({
        "Model": model_name,
        "Accuracy (%)": accuracy,
        "Macro F1 (%)": macro_f1,
        "Top-3 Accuracy (%)": top3_accuracy,
        "Validation Loss": validation_loss,
        "Errors": errors
    })

comparison_df = pd.DataFrame(comparison_results)

print("\n=== THREE-MODEL COMPARISON ===")
print(comparison_df.round(4).to_string(index=False))

# ============================================================
# 4. ANALYZE PERFORMANCE DIFFERENCES
# ============================================================

baseline_accuracy = comparison_df.loc[
    comparison_df["Model"] == "Baseline CNN",
    "Accuracy (%)"
].iloc[0]

print("\n=== ACCURACY DIFFERENCES VS BASELINE ===")

for _, row in comparison_df.iterrows():

    difference = row["Accuracy (%)"] - baseline_accuracy

    print(
        f"{row['Model']}: {difference:+.2f} "
        "percentage points"
    )

# ============================================================
# 5. VISUALIZE MODEL COMPARISON
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5)
)

model_names = comparison_df["Model"]

# Compare classification accuracy
axes[0].bar(
    model_names,
    comparison_df["Accuracy (%)"],
    color=["steelblue", "mediumseagreen", "orange"]
)

axes[0].set_title("Validation Accuracy Comparison")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_ylim(0, 100)
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.3)

# Compare validation loss
axes[1].bar(
    model_names,
    comparison_df["Validation Loss"],
    color=["steelblue", "mediumseagreen", "orange"]
)

axes[1].set_title("Validation Loss Comparison")
axes[1].set_ylabel("Cross-Entropy Loss")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle(
    "AHCD - Three CNN Experimental Comparison",
    fontsize=15
)

plt.tight_layout()

# ============================================================
# 6. SAVE EXPERIMENTAL RESULTS
# ============================================================

TABLES_DIR = PROJECT_DIR / "results/tables"
FIGURES_DIR = PROJECT_DIR / "results/figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

table_path = (
    TABLES_DIR / "three_model_comparison.csv"
)

figure_path = (
    FIGURES_DIR / "three_model_comparison.png"
)

comparison_df.to_csv(
    table_path,
    index=False
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 7. IDENTIFY THE BEST VALIDATION MODEL
# ============================================================

best_model_row = comparison_df.loc[
    comparison_df["Validation Loss"].idxmin()
]

print("\n=== EXPERIMENTAL COMPARISON SUMMARY ===")

print("Best model:", best_model_row["Model"])

print(
    "Best validation accuracy:",
    round(best_model_row["Accuracy (%)"], 2), "%"
)

print(
    "Best validation loss:",
    round(best_model_row["Validation Loss"], 4)
)

print("\nComparison table:", table_path)
print("Comparison figure:", figure_path)

print("\nThree-model comparison completed successfully!")


## Multi-Seed Reproducibility Study

### Controlled Training Replication

In [ ]:
# CONTROLLED MULTI-SEED REPLICATION
# EXPERIMENTAL SEED: 123


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

EXPERIMENT_SEED = 123

INPUT_SHAPE = (32, 32, 1)
NUM_CLASSES = 28
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = PROJECT_DIR / "replications" / f"seed_{EXPERIMENT_SEED}"

RUN_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. DEFINE AUGMENTATION STRATEGIES
# ============================================================

def create_augmentation(strategy, seed):

    if strategy == "baseline":
        return None

    if strategy == "standard":
        rotation = 0.015
        translation = 0.04
        zoom = 0.04

    elif strategy == "light":
        rotation = 0.008
        translation = 0.02
        zoom = 0.02

    else:
        raise ValueError("Unknown augmentation strategy")

    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomRotation(
                factor=rotation,
                fill_mode="constant",
                fill_value=0.0,
                seed=seed + 1
            ),

            tf.keras.layers.RandomTranslation(
                height_factor=translation,
                width_factor=translation,
                fill_mode="constant",
                fill_value=0.0,
                seed=seed + 2
            ),

            tf.keras.layers.RandomZoom(
                height_factor=zoom,
                width_factor=zoom,
                fill_mode="constant",
                fill_value=0.0,
                seed=seed + 3
            )
        ],
        name=f"{strategy}_augmentation"
    )


# ============================================================
# 3. BUILD MODELS WITH CONTROLLED INITIALIZATION
# ============================================================

def build_experiment_model(strategy, seed):

    # Reset random generators before initializing each CNN
    tf.keras.utils.set_random_seed(seed)

    # Build the same untrained CNN architecture
    backbone = build_baseline_cnn(
        INPUT_SHAPE,
        NUM_CLASSES
    )

    # Store the initial weights for comparison
    initial_weights = [
        weight.copy()
        for weight in backbone.get_weights()
    ]

    if strategy == "baseline":
        model = backbone

    else:
        augmentation = create_augmentation(
            strategy, seed
        )

        inputs = tf.keras.Input(shape=INPUT_SHAPE)

        x = augmentation(inputs)

        outputs = backbone(x)

        model = tf.keras.Model(
            inputs=inputs,
            outputs=outputs,
            name=f"AHCD_{strategy}_seed_{seed}"
        )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=3,
                name="top3_accuracy"
            )
        ]
    )

    return model, initial_weights


# ============================================================
# 4. CREATE IDENTICAL DATA SHUFFLING CONFIGURATIONS
# ============================================================

def create_training_dataset(seed):

    dataset = tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )

    dataset = dataset.shuffle(
        buffer_size=len(X_train_cnn),
        seed=seed,
        reshuffle_each_iteration=True
    )

    return dataset.batch(BATCH_SIZE).prefetch(
        tf.data.AUTOTUNE
    )


# ============================================================
# 5. TRAIN THE THREE EXPERIMENTAL CONFIGURATIONS
# ============================================================

strategies = ["baseline", "standard", "light"]

experiment_results = []
reference_initial_weights = None

for strategy in strategies:

    print("\n" + "=" * 55)
    print(f"TRAINING: {strategy.upper()}")
    print(f"RANDOM SEED: {EXPERIMENT_SEED}")
    print("=" * 55)

    # Release previous Keras graph resources
    tf.keras.backend.clear_session()

    # Create a freshly initialized model
    model, initial_weights = build_experiment_model(
        strategy,
        EXPERIMENT_SEED
    )

    # Verify identical initial CNN weights
    if reference_initial_weights is None:
        reference_initial_weights = initial_weights

    else:
        assert len(initial_weights) == len(
            reference_initial_weights
        )

        for reference, current in zip(
            reference_initial_weights,
            initial_weights
        ):
            np.testing.assert_allclose(
                reference,
                current,
                rtol=0,
                atol=0
            )

    print("Identical CNN initialization: VERIFIED")

    # Create a fresh dataset with the same shuffle seed
    current_train_ds = create_training_dataset(
        EXPERIMENT_SEED
    )

    # Define independent experiment files
    model_path = RUN_DIR / f"{strategy}_best.keras"
    log_path = RUN_DIR / f"{strategy}_history.csv"

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            str(model_path),
            monitor="val_loss",
            save_best_only=True,
            mode="min",
            verbose=0
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True,
            mode="min",
            verbose=0
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            mode="min",
            verbose=0
        ),

        tf.keras.callbacks.CSVLogger(
            str(log_path)
        ),

        tf.keras.callbacks.TerminateOnNaN()
    ]

    # Train the model
    history = model.fit(
        current_train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=0
    )

    # Identify the best epoch using validation loss
    best_idx = int(
        np.argmin(history.history["val_loss"])
    )

    # Evaluate the saved best checkpoint on validation
    best_model = tf.keras.models.load_model(
        model_path
    )

    metrics = best_model.evaluate(
        val_ds,
        verbose=0,
        return_dict=True
    )

    result = {
        "Seed": EXPERIMENT_SEED,
        "Strategy": strategy,
        "Completed Epochs": len(history.history["loss"]),
        "Best Epoch": best_idx + 1,
        "Validation Accuracy (%)": metrics["accuracy"] * 100,
        "Validation Top-3 (%)": metrics["top3_accuracy"] * 100,
        "Validation Loss": metrics["loss"]
    }

    experiment_results.append(result)

    print("Best epoch:", result["Best Epoch"])
    print(
        "Validation accuracy:",
        round(result["Validation Accuracy (%)"], 2), "%"
    )
    print(
        "Validation loss:",
        round(result["Validation Loss"], 4)
    )
    print("Model checkpoint saved:", model_path.name)


# ============================================================
# 6. SAVE THE REPLICATION RESULTS
# ============================================================

replication_df = pd.DataFrame(experiment_results)

results_path = RUN_DIR / "replication_results.csv"

replication_df.to_csv(
    results_path,
    index=False
)

print("\n=== SEED 123 REPLICATION SUMMARY ===")
print(replication_df.round(4).to_string(index=False))

print("\n=== EXPERIMENT VERIFICATION ===")
print("Same CNN architecture: YES")
print("Same initial CNN weights: VERIFIED")
print("Same data shuffle seed: YES")
print("Same validation set: YES")
print("Test set used: NO")

print("\nResults saved to:", results_path)
print("Controlled replication completed successfully!")


In [ ]:

# ============================================================
# VERIFY RECOVERED BASELINE CHECKPOINT
# ============================================================

from google.colab import files
import tensorflow as tf
from pathlib import Path

# 1. Upload the recovered model from your computer
print("=== UPLOAD RECOVERED MODEL ===")

uploaded = files.upload()

assert len(uploaded) == 1, "Please upload exactly one model file."

filename = next(iter(uploaded))
model_path = Path("/content") / filename

assert model_path.suffix == ".keras", "Expected a .keras file."

print("Uploaded file:", filename)

# 2. Load the recovered CNN
recovered_model = tf.keras.models.load_model(
    model_path
)

# 3. Evaluate on the validation set
results = recovered_model.evaluate(
    X_val_cnn,
    y_val_cnn,
    batch_size=64,
    verbose=0,
    return_dict=True
)

# 4. Display results
print("\n=== RECOVERED BASELINE RESULTS ===")

print(
    "Validation Accuracy:",
    round(results["accuracy"] * 100, 4), "%"
)

print(
    "Validation Loss:",
    round(results["loss"], 6)
)

# 5. Compare with the original results
print("\n=== ORIGINAL BASELINE ===")
print("Expected Accuracy: 95.5853 %")
print("Expected Loss: 0.168644")

print("\n=== VERIFICATION ===")

if (
    abs(results["accuracy"] * 100 - 95.5853) < 0.01
    and abs(results["loss"] - 0.168644) < 0.001
):
    print("SUCCESS: Original baseline performance recovered!")
else:
    print("Results differ from the original baseline.")


In [ ]:

# ============================================================
# SECURE THE RECOVERED BASELINE CHECKPOINT
# ============================================================

from pathlib import Path
import shutil
import hashlib

# 1. Define source and destination paths
source_path = Path(
    "/content/baseline_cnn_best.keras"
)

destination_path = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN/"
    "models/baseline_cnn_original_recovered.keras"
)

# 2. Verify the recovered model is available
assert source_path.exists(), "Recovered model not found!"

# 3. Prevent accidental overwriting
assert not destination_path.exists(), (
    "Backup already exists. No file was overwritten."
)

# 4. Save a separate backup to Google Drive
destination_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    source_path,
    destination_path
)

# 5. Verify the copied file is identical
def file_hash(path):
    sha256 = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()

assert file_hash(source_path) == file_hash(
    destination_path
), "Backup verification failed!"

print("=== BASELINE RECOVERY BACKUP ===")
print("Original validation accuracy: 95.5853 %")
print("Original validation loss: 0.168644")
print("Backup path:", destination_path)
print("File integrity: VERIFIED")
print("Existing checkpoints: NOT MODIFIED")

print("\nRecovered baseline safely backed up!")


## Train Baseline with Seed 456

In [ ]:
# CONTROLLED REPLICATION
# SEED 456 - BASELINE CNN


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. VERIFY SESSION REQUIREMENTS
# ============================================================

required_objects = [
    "X_train_cnn",
    "y_train_cnn",
    "X_val_cnn",
    "y_val_cnn",
    "build_baseline_cnn"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing session objects: "
        + ", ".join(missing)
        + ". Stop here and restore these objects first."
    )

# ============================================================
# 2. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 456
BATCH_SIZE = 64
EPOCHS = 30

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = PROJECT_DIR / "replications" / "seed_456"
RUN_DIR.mkdir(parents=True, exist_ok=True)

model_path = RUN_DIR / "baseline_best.keras"
history_path = RUN_DIR / "baseline_history.csv"
weights_path = RUN_DIR / "baseline_initial_weights.npz"
results_path = RUN_DIR / "baseline_results.csv"

# Protect previous results against accidental overwriting
for path in [
    model_path,
    history_path,
    weights_path,
    results_path
]:
    if path.exists():
        raise RuntimeError(
            f"Existing experiment file: {path.name}. "
            "Training stopped to prevent overwriting."
        )

# Verify the recovered baseline backup is still available
recovered_path = (
    PROJECT_DIR
    / "models"
    / "baseline_cnn_original_recovered.keras"
)

assert recovered_path.exists(), (
    "Recovered original baseline backup is missing!"
)

# ============================================================
# 3. PREPARE CONTROLLED INITIALIZATION
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = build_baseline_cnn(
    (32, 32, 1),
    28
)

assert model.count_params() == 103420

# Save initial weights for the next two configurations
initial_weights = model.get_weights()

np.savez_compressed(
    weights_path,
    *initial_weights
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

# ============================================================
# 4. REBUILD THE DATA PIPELINE
# ============================================================

train_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )
    .shuffle(
        buffer_size=len(X_train_cnn),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 5. DEFINE TRAINING CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 6. TRAIN BASELINE CNN
# ============================================================

print("=== SEED 456 - BASELINE TRAINING ===")
print("Device:", "GPU" if tf.config.list_physical_devices("GPU") else "CPU")
print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Maximum epochs:", EPOCHS)
print("Initial weights saved:", weights_path.name)

history = model.fit(
    train_ds_456,
    validation_data=val_ds_456,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 7. EVALUATE THE BEST SAVED CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    model_path
)

metrics = best_model.evaluate(
    val_ds_456,
    verbose=0,
    return_dict=True
)

best_epoch = int(
    np.argmin(history.history["val_loss"])
) + 1

result = pd.DataFrame([{
    "Seed": SEED,
    "Strategy": "baseline",
    "Completed Epochs": len(history.history["loss"]),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": metrics["accuracy"] * 100,
    "Validation Top-3 (%)": metrics["top3_accuracy"] * 100,
    "Validation Loss": metrics["loss"]
}])

result.to_csv(
    results_path,
    index=False
)

# ============================================================
# 8. DISPLAY SUMMARY
# ============================================================

print("\n=== SEED 456 BASELINE SUMMARY ===")
print(result.round(4).to_string(index=False))

print("\n=== FILE VERIFICATION ===")
print("Model saved:", model_path.exists())
print("History saved:", history_path.exists())
print("Initial weights saved:", weights_path.exists())
print("Results saved:", results_path.exists())
print("Original baseline backup preserved:", recovered_path.exists())
print("Test set used: NO")

print("\nSeed 456 baseline training completed successfully!")


## standard Augmentation, Seed 456

In [ ]:
# STANDARD AUGMENTATION
# CONTROLLED REPLICATION - SEED 456


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 456
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = PROJECT_DIR / "replications" / "seed_456"

weights_path = RUN_DIR / "baseline_initial_weights.npz"
baseline_path = RUN_DIR / "baseline_best.keras"

model_path = RUN_DIR / "standard_best.keras"
history_path = RUN_DIR / "standard_history.csv"
results_path = RUN_DIR / "standard_results.csv"

# Verify that baseline initialization is available
assert weights_path.exists(), "Initial weights not found!"
assert baseline_path.exists(), "Seed 456 baseline not found!"

# Prevent overwriting an existing experiment
for path in [model_path, history_path, results_path]:
    if path.exists():
        raise RuntimeError(
            f"Existing file: {path.name}. "
            "Experiment stopped to prevent overwriting."
        )

assert "build_baseline_cnn" in globals(), (
    "CNN architecture function is missing!"
)

# ============================================================
# 2. BUILD CNN WITH IDENTICAL INITIAL WEIGHTS
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

backbone = build_baseline_cnn(
    (32, 32, 1),
    NUM_CLASSES
)

# Restore the exact initial weights from Seed 456 baseline
with np.load(weights_path, allow_pickle=False) as saved:
    initial_weights = [
        saved[f"arr_{i}"]
        for i in range(len(saved.files))
    ]

assert len(initial_weights) == len(backbone.get_weights())

backbone.set_weights(initial_weights)

# Verify exact weight equality
for expected, actual in zip(
    initial_weights,
    backbone.get_weights()
):
    np.testing.assert_array_equal(expected, actual)

print("=== INITIALIZATION VERIFICATION ===")
print("Seed:", SEED)
print("Identical initial CNN weights: VERIFIED")
print("Backbone parameters:", backbone.count_params())

# ============================================================
# 3. DEFINE STANDARD DATA AUGMENTATION
# ============================================================

standard_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=0.015,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 1
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 2
        ),

        tf.keras.layers.RandomZoom(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 3
        )
    ],
    name="standard_augmentation"
)

# ============================================================
# 4. BUILD THE COMPLETE MODEL
# ============================================================

inputs = tf.keras.Input(
    shape=(32, 32, 1)
)

x = standard_augmentation(inputs)

outputs = backbone(x)

standard_model_456 = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="AHCD_Standard_Seed456"
)

standard_model_456.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

assert standard_model_456.count_params() == 103420

# ============================================================
# 5. CREATE CONTROLLED DATASETS
# ============================================================

train_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )
    .shuffle(
        buffer_size=len(X_train_cnn),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 6. DEFINE CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        mode="min",
        restore_best_weights=True,
        verbose=0
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        mode="min",
        min_lr=1e-5,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 7. TRAIN STANDARD AUGMENTED CNN
# ============================================================

print("\n=== SEED 456 - STANDARD TRAINING ===")

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Maximum epochs:", EPOCHS)
print("Test set used: NO")

history_standard_456 = standard_model_456.fit(
    train_ds_456,
    validation_data=val_ds_456,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 8. EVALUATE THE BEST CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    model_path
)

metrics = best_model.evaluate(
    val_ds_456,
    verbose=0,
    return_dict=True
)

best_epoch = int(
    np.argmin(
        history_standard_456.history["val_loss"]
    )
) + 1

result = pd.DataFrame([{
    "Seed": SEED,
    "Strategy": "standard",
    "Completed Epochs": len(
        history_standard_456.history["loss"]
    ),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": metrics["accuracy"] * 100,
    "Validation Top-3 (%)": metrics["top3_accuracy"] * 100,
    "Validation Loss": metrics["loss"]
}])

result.to_csv(
    results_path,
    index=False
)

# ============================================================
# 9. DISPLAY EXPERIMENT SUMMARY
# ============================================================

print("\n=== SEED 456 STANDARD SUMMARY ===")

print(
    result.round(4).to_string(index=False)
)

print("\n=== FILE VERIFICATION ===")

print("Model saved:", model_path.exists())
print("History saved:", history_path.exists())
print("Results saved:", results_path.exists())
print("Initial weights unchanged:", weights_path.exists())
print("Baseline checkpoint preserved:", baseline_path.exists())
print("Test set used: NO")

print(
    "\nSeed 456 standard training completed successfully!"
)


## Light Augmentation, Seed 456

In [ ]:
# LIGHT AUGMENTATION
# CONTROLLED REPLICATION - SEED 456

import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 456
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = PROJECT_DIR / "replications" / "seed_456"

weights_path = RUN_DIR / "baseline_initial_weights.npz"

model_path = RUN_DIR / "light_best.keras"
history_path = RUN_DIR / "light_history.csv"
results_path = RUN_DIR / "light_results.csv"

# ============================================================
# 2. VERIFY REQUIRED FILES AND DATA
# ============================================================

assert weights_path.exists(), "Initial weights not found!"

assert (RUN_DIR / "baseline_best.keras").exists()
assert (RUN_DIR / "standard_best.keras").exists()

assert "build_baseline_cnn" in globals()

for name in [
    "X_train_cnn",
    "y_train_cnn",
    "X_val_cnn",
    "y_val_cnn"
]:
    assert name in globals(), f"Missing variable: {name}"

# Prevent accidental overwriting
for path in [model_path, history_path, results_path]:
    if path.exists():
        raise RuntimeError(
            f"Existing experiment file: {path.name}. "
            "Training stopped to prevent overwriting."
        )

# ============================================================
# 3. BUILD CNN WITH IDENTICAL INITIAL WEIGHTS
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

backbone = build_baseline_cnn(
    (32, 32, 1),
    NUM_CLASSES
)

# Restore the original Seed 456 initialization
with np.load(weights_path, allow_pickle=False) as saved:
    initial_weights = [
        saved[f"arr_{i}"]
        for i in range(len(saved.files))
    ]

backbone.set_weights(initial_weights)

# Verify exact initial weight equality
for expected, actual in zip(
    initial_weights,
    backbone.get_weights()
):
    np.testing.assert_array_equal(expected, actual)

print("=== INITIALIZATION VERIFICATION ===")
print("Seed:", SEED)
print("Identical initial CNN weights: VERIFIED")
print("Backbone parameters:", backbone.count_params())

# ============================================================
# 4. DEFINE LIGHT DATA AUGMENTATION
# ============================================================

light_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=0.008,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 1
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.02,
            width_factor=0.02,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 2
        ),

        tf.keras.layers.RandomZoom(
            height_factor=0.02,
            width_factor=0.02,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 3
        )
    ],
    name="light_augmentation"
)

# ============================================================
# 5. BUILD AND COMPILE MODEL
# ============================================================

inputs = tf.keras.Input(
    shape=(32, 32, 1)
)

x = light_augmentation(inputs)
outputs = backbone(x)

light_model_456 = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="AHCD_Light_Seed456"
)

light_model_456.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

assert light_model_456.count_params() == 103420

# ============================================================
# 6. CREATE TRAINING AND VALIDATION DATASETS
# ============================================================

train_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )
    .shuffle(
        buffer_size=len(X_train_cnn),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 7. DEFINE TRAINING CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=0
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 8. TRAIN LIGHT AUGMENTED CNN
# ============================================================

print("\n=== SEED 456 - LIGHT TRAINING ===")

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Maximum epochs:", EPOCHS)
print("Test set used: NO")

history_light_456 = light_model_456.fit(
    train_ds_456,
    validation_data=val_ds_456,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 9. EVALUATE THE BEST SAVED CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    model_path
)

metrics = best_model.evaluate(
    val_ds_456,
    verbose=0,
    return_dict=True
)

best_epoch = int(
    np.argmin(history_light_456.history["val_loss"])
) + 1

result = pd.DataFrame([{
    "Seed": SEED,
    "Strategy": "light",
    "Completed Epochs": len(
        history_light_456.history["loss"]
    ),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": metrics["accuracy"] * 100,
    "Validation Top-3 (%)": metrics["top3_accuracy"] * 100,
    "Validation Loss": metrics["loss"]
}])

result.to_csv(
    results_path,
    index=False
)

# ============================================================
# 10. DISPLAY EXPERIMENT SUMMARY
# ============================================================

print("\n=== SEED 456 LIGHT SUMMARY ===")

print(result.round(4).to_string(index=False))

print("\n=== FILE VERIFICATION ===")

print("Model saved:", model_path.exists())
print("History saved:", history_path.exists())
print("Results saved:", results_path.exists())
print("Initial weights preserved:", weights_path.exists())
print(
    "Baseline checkpoint preserved:",
    (RUN_DIR / "baseline_best.keras").exists()
)
print(
    "Standard checkpoint preserved:",
    (RUN_DIR / "standard_best.keras").exists()
)
print("Test set used: NO")

print("\nSeed 456 light training completed successfully!")


## Seed 789: Baseline CNN

In [ ]:
# BASELINE CNN - SEED 789
# CONTROLLED REPLICATION


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 789
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = PROJECT_DIR / "replications" / "seed_789"
RUN_DIR.mkdir(parents=True, exist_ok=True)

model_path = RUN_DIR / "baseline_best.keras"
history_path = RUN_DIR / "baseline_history.csv"
weights_path = RUN_DIR / "baseline_initial_weights.npz"
results_path = RUN_DIR / "baseline_results.csv"

# ============================================================
# 2. VERIFY REQUIRED OBJECTS AND FILES
# ============================================================

required_objects = [
    "build_baseline_cnn",
    "X_train_cnn",
    "y_train_cnn",
    "X_val_cnn",
    "y_val_cnn"
]

for name in required_objects:
    assert name in globals(), f"Missing object: {name}"

# Prevent accidental overwriting
for path in [
    model_path,
    history_path,
    weights_path,
    results_path
]:
    if path.exists():
        raise RuntimeError(
            f"Existing file: {path.name}. "
            "Training stopped to prevent overwriting."
        )

# Verify the recovered original baseline
original_backup = (
    PROJECT_DIR
    / "models"
    / "baseline_cnn_original_recovered.keras"
)

assert original_backup.exists(), (
    "Original baseline backup is missing!"
)

# ============================================================
# 3. INITIALIZE THE BASELINE CNN
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

baseline_789 = build_baseline_cnn(
    (32, 32, 1),
    NUM_CLASSES
)

assert baseline_789.count_params() == 103420

# Save initial CNN weights for paired experiments
initial_weights = baseline_789.get_weights()

np.savez_compressed(
    weights_path,
    *initial_weights
)

# ============================================================
# 4. COMPILE THE MODEL
# ============================================================

baseline_789.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

# ============================================================
# 5. BUILD CONTROLLED DATA PIPELINES
# ============================================================

train_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )
    .shuffle(
        buffer_size=len(X_train_cnn),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 6. TRAINING CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=0
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 7. TRAIN THE MODEL
# ============================================================

print("=== SEED 789 - BASELINE TRAINING ===")

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Maximum epochs:", EPOCHS)
print("Initial weights:", weights_path.name)
print("Test set used: NO")

history_baseline_789 = baseline_789.fit(
    train_ds_789,
    validation_data=val_ds_789,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 8. EVALUATE THE BEST SAVED CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    model_path
)

metrics = best_model.evaluate(
    val_ds_789,
    verbose=0,
    return_dict=True
)

best_epoch = int(
    np.argmin(
        history_baseline_789.history["val_loss"]
    )
) + 1

result = pd.DataFrame([{
    "Seed": SEED,
    "Strategy": "baseline",
    "Completed Epochs": len(
        history_baseline_789.history["loss"]
    ),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": metrics["accuracy"] * 100,
    "Validation Top-3 (%)": metrics["top3_accuracy"] * 100,
    "Validation Loss": metrics["loss"]
}])

result.to_csv(
    results_path,
    index=False
)

# ============================================================
# 9. DISPLAY RESULTS
# ============================================================

print("\n=== SEED 789 BASELINE SUMMARY ===")
print(result.round(4).to_string(index=False))

print("\n=== FILE VERIFICATION ===")

print("Model saved:", model_path.exists())
print("Training history saved:", history_path.exists())
print("Initial weights saved:", weights_path.exists())
print("Results saved:", results_path.exists())
print("Original baseline backup preserved:", original_backup.exists())
print("Test set used: NO")

print("\nSeed 789 baseline training completed successfully!")


## Standard Augmentation with Seed 789

In [ ]:
# STANDARD AUGMENTATION - SEED 789
# CONTROLLED REPLICATION

import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 789
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = PROJECT_DIR / "replications" / "seed_789"

weights_path = RUN_DIR / "baseline_initial_weights.npz"
baseline_path = RUN_DIR / "baseline_best.keras"

model_path = RUN_DIR / "standard_best.keras"
history_path = RUN_DIR / "standard_history.csv"
results_path = RUN_DIR / "standard_results.csv"

# ============================================================
# 2. VERIFY EXPERIMENT REQUIREMENTS
# ============================================================

assert weights_path.exists(), "Initial weights not found!"
assert baseline_path.exists(), "Seed 789 baseline not found!"

required_objects = [
    "build_baseline_cnn",
    "X_train_cnn",
    "y_train_cnn",
    "X_val_cnn",
    "y_val_cnn"
]

for name in required_objects:
    assert name in globals(), f"Missing object: {name}"

# Prevent accidental overwriting
for path in [model_path, history_path, results_path]:
    if path.exists():
        raise RuntimeError(
            f"Existing experiment file: {path.name}. "
            "Training stopped to prevent overwriting."
        )

# Protect the recovered original baseline checkpoint
original_backup = (
    PROJECT_DIR / "models"
    / "baseline_cnn_original_recovered.keras"
)

assert original_backup.exists(), (
    "Original baseline backup is missing!"
)

# ============================================================
# 3. BUILD BACKBONE WITH IDENTICAL INITIAL WEIGHTS
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

backbone = build_baseline_cnn(
    (32, 32, 1),
    NUM_CLASSES
)

# Load the original Seed 789 initialization
with np.load(weights_path, allow_pickle=False) as saved:
    initial_weights = [
        saved[f"arr_{i}"]
        for i in range(len(saved.files))
    ]

assert len(initial_weights) == len(backbone.get_weights())

backbone.set_weights(initial_weights)

# Verify exact weight equality
for expected, actual in zip(
    initial_weights,
    backbone.get_weights()
):
    np.testing.assert_array_equal(expected, actual)

print("=== INITIALIZATION VERIFICATION ===")
print("Seed:", SEED)
print("Identical initial CNN weights: VERIFIED")
print("Backbone parameters:", backbone.count_params())

# ============================================================
# 4. DEFINE STANDARD AUGMENTATION
# ============================================================

standard_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=0.015,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 1
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 2
        ),

        tf.keras.layers.RandomZoom(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 3
        )
    ],
    name="standard_augmentation"
)

# ============================================================
# 5. BUILD AND COMPILE THE MODEL
# ============================================================

inputs = tf.keras.Input(shape=(32, 32, 1))

x = standard_augmentation(inputs)
outputs = backbone(x)

standard_model_789 = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="AHCD_Standard_Seed789"
)

standard_model_789.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

assert standard_model_789.count_params() == 103420

# ============================================================
# 6. CREATE CONTROLLED DATA PIPELINES
# ============================================================

train_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )
    .shuffle(
        buffer_size=len(X_train_cnn),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 7. DEFINE TRAINING CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=0
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 8. TRAIN THE MODEL
# ============================================================

print("\n=== SEED 789 - STANDARD TRAINING ===")

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Maximum epochs:", EPOCHS)
print("Test set used: NO")

history_standard_789 = standard_model_789.fit(
    train_ds_789,
    validation_data=val_ds_789,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 9. EVALUATE THE BEST SAVED CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    model_path
)

metrics = best_model.evaluate(
    val_ds_789,
    verbose=0,
    return_dict=True
)

best_epoch = int(
    np.argmin(
        history_standard_789.history["val_loss"]
    )
) + 1

result = pd.DataFrame([{
    "Seed": SEED,
    "Strategy": "standard",
    "Completed Epochs": len(
        history_standard_789.history["loss"]
    ),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": metrics["accuracy"] * 100,
    "Validation Top-3 (%)": metrics["top3_accuracy"] * 100,
    "Validation Loss": metrics["loss"]
}])

result.to_csv(
    results_path,
    index=False
)

# ============================================================
# 10. DISPLAY EXPERIMENT SUMMARY
# ============================================================

print("\n=== SEED 789 STANDARD SUMMARY ===")
print(result.round(4).to_string(index=False))

print("\n=== FILE VERIFICATION ===")

print("Model saved:", model_path.exists())
print("History saved:", history_path.exists())
print("Results saved:", results_path.exists())
print("Initial weights preserved:", weights_path.exists())
print("Baseline checkpoint preserved:", baseline_path.exists())
print("Original baseline backup preserved:", original_backup.exists())
print("Test set used: NO")

print("\nSeed 789 standard training completed successfully!")


## Light Augmentation with Seed 789

In [ ]:
# LIGHT AUGMENTATION - SEED 789
# FINAL CONTROLLED REPLICATION


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 789
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = PROJECT_DIR / "replications" / "seed_789"

weights_path = RUN_DIR / "baseline_initial_weights.npz"

baseline_path = RUN_DIR / "baseline_best.keras"
standard_path = RUN_DIR / "standard_best.keras"

model_path = RUN_DIR / "light_best.keras"
history_path = RUN_DIR / "light_history.csv"
results_path = RUN_DIR / "light_results.csv"

# ============================================================
# 2. VERIFY FILES AND SESSION OBJECTS
# ============================================================

assert weights_path.exists(), "Initial weights not found!"
assert baseline_path.exists(), "Baseline checkpoint not found!"
assert standard_path.exists(), "Standard checkpoint not found!"

required_objects = [
    "build_baseline_cnn",
    "X_train_cnn",
    "y_train_cnn",
    "X_val_cnn",
    "y_val_cnn"
]

for name in required_objects:
    assert name in globals(), f"Missing object: {name}"

# Prevent accidental overwriting of previous results
for path in [model_path, history_path, results_path]:
    if path.exists():
        raise RuntimeError(
            f"Existing experiment file: {path.name}. "
            "Training stopped to prevent overwriting."
        )

# Protect the original recovered baseline
original_backup = (
    PROJECT_DIR
    / "models"
    / "baseline_cnn_original_recovered.keras"
)

assert original_backup.exists(), (
    "Original baseline backup is missing!"
)

# ============================================================
# 3. RESTORE IDENTICAL INITIAL WEIGHTS
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

backbone = build_baseline_cnn(
    (32, 32, 1),
    NUM_CLASSES
)

with np.load(weights_path, allow_pickle=False) as saved:
    initial_weights = [
        saved[f"arr_{i}"]
        for i in range(len(saved.files))
    ]

assert len(initial_weights) == len(backbone.get_weights())

backbone.set_weights(initial_weights)

# Verify exact equality with the saved initialization
for expected, actual in zip(
    initial_weights,
    backbone.get_weights()
):
    np.testing.assert_array_equal(expected, actual)

print("=== INITIALIZATION VERIFICATION ===")
print("Seed:", SEED)
print("Identical initial CNN weights: VERIFIED")
print("Backbone parameters:", backbone.count_params())

# ============================================================
# 4. DEFINE LIGHT DATA AUGMENTATION
# ============================================================

light_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomRotation(
            factor=0.008,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 1
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.02,
            width_factor=0.02,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 2
        ),

        tf.keras.layers.RandomZoom(
            height_factor=0.02,
            width_factor=0.02,
            fill_mode="constant",
            fill_value=0.0,
            seed=SEED + 3
        )
    ],
    name="light_augmentation"
)

# ============================================================
# 5. BUILD AND COMPILE THE MODEL
# ============================================================

inputs = tf.keras.Input(
    shape=(32, 32, 1)
)

x = light_augmentation(inputs)
outputs = backbone(x)

light_model_789 = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="AHCD_Light_Seed789"
)

light_model_789.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

assert light_model_789.count_params() == 103420

# ============================================================
# 6. CREATE CONTROLLED DATA PIPELINES
# ============================================================

train_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )
    .shuffle(
        buffer_size=len(X_train_cnn),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 7. DEFINE CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=0
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 8. TRAIN LIGHT AUGMENTED CNN
# ============================================================

print("\n=== SEED 789 - LIGHT TRAINING ===")

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Maximum epochs:", EPOCHS)
print("Test set used: NO")

history_light_789 = light_model_789.fit(
    train_ds_789,
    validation_data=val_ds_789,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 9. EVALUATE THE BEST SAVED CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    model_path
)

metrics = best_model.evaluate(
    val_ds_789,
    verbose=0,
    return_dict=True
)

best_epoch = int(
    np.argmin(
        history_light_789.history["val_loss"]
    )
) + 1

result = pd.DataFrame([{
    "Seed": SEED,
    "Strategy": "light",
    "Completed Epochs": len(
        history_light_789.history["loss"]
    ),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": metrics["accuracy"] * 100,
    "Validation Top-3 (%)": metrics["top3_accuracy"] * 100,
    "Validation Loss": metrics["loss"]
}])

result.to_csv(
    results_path,
    index=False
)

# ============================================================
# 10. DISPLAY EXPERIMENT SUMMARY
# ============================================================

print("\n=== SEED 789 LIGHT SUMMARY ===")
print(result.round(4).to_string(index=False))

print("\n=== FILE VERIFICATION ===")
print("Model saved:", model_path.exists())
print("History saved:", history_path.exists())
print("Results saved:", results_path.exists())
print("Initial weights preserved:", weights_path.exists())
print("Baseline checkpoint preserved:", baseline_path.exists())
print("Standard checkpoint preserved:", standard_path.exists())
print("Original baseline backup preserved:", original_backup.exists())
print("Test set used: NO")

print("\nSeed 789 light training completed successfully!")


## Multi-Seed Statistical Analysis

In [ ]:
# MULTI-SEED STATISTICAL ANALYSIS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. DEFINE PROJECT DIRECTORIES
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

REPLICATIONS_DIR = PROJECT_DIR / "replications"
TABLES_DIR = PROJECT_DIR / "results/tables"
FIGURES_DIR = PROJECT_DIR / "results/figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [123, 456, 789]
STRATEGIES = ["baseline", "standard", "light"]

# ============================================================
# 2. LOAD ALL SAVED EXPERIMENT RESULTS
# ============================================================

dataframes = []

# Seed 123: all strategies in a single CSV file
seed_123_path = (
    REPLICATIONS_DIR
    / "seed_123"
    / "replication_results.csv"
)

assert seed_123_path.exists(), (
    "Seed 123 results file not found!"
)

dataframes.append(
    pd.read_csv(seed_123_path)
)

# Seeds 456 and 789: separate CSV files per strategy
for seed in [456, 789]:

    for strategy in STRATEGIES:

        result_path = (
            REPLICATIONS_DIR
            / f"seed_{seed}"
            / f"{strategy}_results.csv"
        )

        assert result_path.exists(), (
            f"Missing experiment results: {result_path}"
        )

        dataframes.append(
            pd.read_csv(result_path)
        )

all_results = pd.concat(
    dataframes,
    ignore_index=True
)

all_results["Strategy"] = (
    all_results["Strategy"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ============================================================
# 3. VERIFY EXPERIMENT COMPLETENESS
# ============================================================

expected_pairs = {
    (seed, strategy)
    for seed in SEEDS
    for strategy in STRATEGIES
}

actual_pairs = list(
    zip(
        all_results["Seed"],
        all_results["Strategy"]
    )
)

assert len(all_results) == 9, (
    "Expected exactly nine experimental results!"
)

assert set(actual_pairs) == expected_pairs, (
    "Missing or unexpected seed-strategy combination!"
)

assert len(set(actual_pairs)) == 9, (
    "Duplicate seed-strategy combination detected!"
)

metric_columns = [
    "Validation Accuracy (%)",
    "Validation Top-3 (%)",
    "Validation Loss"
]

assert np.isfinite(
    all_results[metric_columns].to_numpy(dtype=float)
).all(), "Invalid metric values detected!"

assert all_results[
    "Validation Accuracy (%)"
].between(0, 100).all()

assert all_results[
    "Validation Top-3 (%)"
].between(0, 100).all()

print("=== EXPERIMENT COMPLETENESS ===")
print("Seeds:", SEEDS)
print("Strategies:", STRATEGIES)
print("Expected experiments: 9")
print("Loaded experiments:", len(all_results))
print("Verification: PASSED")
print("Test set used: NO")

# ============================================================
# 4. DISPLAY ALL NINE EXPERIMENTS
# ============================================================

all_results["Strategy"] = pd.Categorical(
    all_results["Strategy"],
    categories=STRATEGIES,
    ordered=True
)

all_results = (
    all_results
    .sort_values(["Seed", "Strategy"])
    .reset_index(drop=True)
)

print("\n=== ALL EXPERIMENTAL RESULTS ===")

print(
    all_results[
        [
            "Seed",
            "Strategy",
            "Validation Accuracy (%)",
            "Validation Top-3 (%)",
            "Validation Loss",
            "Best Epoch"
        ]
    ].round(4).to_string(index=False)
)

# ============================================================
# 5. CALCULATE MEAN AND STANDARD DEVIATION
# ============================================================

summary_df = (
    all_results
    .groupby("Strategy", observed=True)
    .agg(
        Runs=("Seed", "count"),
        Mean_Accuracy=(
            "Validation Accuracy (%)", "mean"
        ),
        Std_Accuracy=(
            "Validation Accuracy (%)", "std"
        ),
        Min_Accuracy=(
            "Validation Accuracy (%)", "min"
        ),
        Max_Accuracy=(
            "Validation Accuracy (%)", "max"
        ),
        Mean_Top3=(
            "Validation Top-3 (%)", "mean"
        ),
        Mean_Loss=(
            "Validation Loss", "mean"
        ),
        Std_Loss=(
            "Validation Loss", "std"
        )
    )
    .reset_index()
)

summary_df.columns = [
    "Strategy",
    "Number of Runs",
    "Mean Accuracy (%)",
    "Std Accuracy (pp)",
    "Min Accuracy (%)",
    "Max Accuracy (%)",
    "Mean Top-3 (%)",
    "Mean Validation Loss",
    "Std Validation Loss"
]

print("\n=== MULTI-SEED STATISTICAL SUMMARY ===")

print(
    summary_df.round(4).to_string(index=False)
)

# ============================================================
# 6. CALCULATE PAIRED ACCURACY DIFFERENCES
# ============================================================

accuracy_pivot = all_results.pivot(
    index="Seed",
    columns="Strategy",
    values="Validation Accuracy (%)"
)

accuracy_pivot = accuracy_pivot[
    STRATEGIES
].sort_index()

paired_df = pd.DataFrame({
    "Seed": accuracy_pivot.index,
    "Baseline Accuracy (%)": accuracy_pivot[
        "baseline"
    ].to_numpy(),
    "Standard Accuracy (%)": accuracy_pivot[
        "standard"
    ].to_numpy(),
    "Light Accuracy (%)": accuracy_pivot[
        "light"
    ].to_numpy()
})

paired_df["Standard vs Baseline (pp)"] = (
    paired_df["Standard Accuracy (%)"]
    - paired_df["Baseline Accuracy (%)"]
)

paired_df["Light vs Baseline (pp)"] = (
    paired_df["Light Accuracy (%)"]
    - paired_df["Baseline Accuracy (%)"]
)

paired_df["Light vs Standard (pp)"] = (
    paired_df["Light Accuracy (%)"]
    - paired_df["Standard Accuracy (%)"]
)

print("\n=== PAIRED ACCURACY DIFFERENCES ===")

print(
    paired_df.round(4).to_string(index=False)
)

# ============================================================
# 7. SUMMARIZE PAIRED DIFFERENCES
# ============================================================

difference_columns = [
    "Standard vs Baseline (pp)",
    "Light vs Baseline (pp)",
    "Light vs Standard (pp)"
]

print("\n=== MEAN PAIRED DIFFERENCES ===")

for column in difference_columns:

    mean_difference = paired_df[column].mean()
    std_difference = paired_df[column].std(ddof=1)

    print(
        f"{column}: "
        f"{mean_difference:+.4f} pp "
        f"(SD: {std_difference:.4f} pp)"
    )

# ============================================================
# 8. VISUALIZE MULTI-SEED PERFORMANCE
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5)
)

# Plot validation accuracy for each strategy and seed
for strategy in STRATEGIES:

    subset = all_results[
        all_results["Strategy"] == strategy
    ]

    axes[0].plot(
        subset["Seed"],
        subset["Validation Accuracy (%)"],
        marker="o",
        linewidth=2,
        label=strategy.capitalize()
    )

axes[0].set_title(
    "Validation Accuracy Across Random Seeds"
)

axes[0].set_xlabel("Random Seed")
axes[0].set_ylabel("Validation Accuracy (%)")
axes[0].set_xticks(SEEDS)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot paired accuracy differences relative to baseline
axes[1].axhline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)

axes[1].plot(
    paired_df["Seed"],
    paired_df["Standard vs Baseline (pp)"],
    marker="o",
    linewidth=2,
    label="Standard vs Baseline"
)

axes[1].plot(
    paired_df["Seed"],
    paired_df["Light vs Baseline (pp)"],
    marker="s",
    linewidth=2,
    label="Light vs Baseline"
)

axes[1].set_title(
    "Paired Accuracy Differences"
)

axes[1].set_xlabel("Random Seed")
axes[1].set_ylabel("Accuracy Difference (pp)")
axes[1].set_xticks(SEEDS)
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle(
    "AHCD - Multi-Seed Experimental Analysis",
    fontsize=15
)

plt.tight_layout()

# ============================================================
# 9. SAVE STATISTICAL RESULTS
# ============================================================

raw_results_path = (
    TABLES_DIR / "multi_seed_raw_results.csv"
)

summary_path = (
    TABLES_DIR / "multi_seed_statistical_summary.csv"
)

paired_path = (
    TABLES_DIR / "multi_seed_paired_differences.csv"
)

figure_path = (
    FIGURES_DIR / "multi_seed_statistical_analysis.png"
)

all_results.to_csv(
    raw_results_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)

paired_df.to_csv(
    paired_path,
    index=False
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 10. FINAL VERIFICATION
# ============================================================

print("\n=== STATISTICAL ANALYSIS SUMMARY ===")

best_strategy = summary_df.loc[
    summary_df["Mean Accuracy (%)"].idxmax(),
    "Strategy"
]

print("Best strategy by mean accuracy:", best_strategy)
print("Number of controlled seeds:", len(SEEDS))
print("Number of controlled experiments:", len(all_results))
print("Sample SD used: YES")
print("Statistical significance claimed: NO")
print("Test set used: NO")

print("\n=== SAVED FILES ===")
print("Raw results:", raw_results_path)
print("Statistical summary:", summary_path)
print("Paired differences:", paired_path)
print("Figure:", figure_path)

print("\nMulti-seed statistical analysis completed successfully!")


## Multi-Seed Macro F1 Analysis

In [ ]:
# MULTI-SEED MACRO F1 ANALYSIS


import tensorflow as tf
import numpy as np
import pandas as pd
import gc

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

REPLICATIONS_DIR = PROJECT_DIR / "replications"
TABLES_DIR = PROJECT_DIR / "results/tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [123, 456, 789]
STRATEGIES = ["baseline", "standard", "light"]
NUM_CLASSES = 28

# ============================================================
# 2. VERIFY VALIDATION DATA
# ============================================================

assert "X_val_cnn" in globals()
assert "y_val_cnn" in globals()

y_true = np.asarray(y_val_cnn).ravel()

assert X_val_cnn.shape == (2016, 32, 32, 1)
assert len(y_true) == 2016
assert np.all((y_true >= 0) & (y_true < NUM_CLASSES))

print("=== VALIDATION DATA VERIFICATION ===")
print("Validation samples:", len(y_true))
print("Classes:", NUM_CLASSES)
print("Test set used: NO")

# ============================================================
# 3. LOAD CHECKPOINTS AND CALCULATE METRICS
# ============================================================

results = []

for seed in SEEDS:

    seed_dir = REPLICATIONS_DIR / f"seed_{seed}"

    for strategy in STRATEGIES:

        print(
            f"\nEvaluating seed {seed} - {strategy}..."
        )

        model_path = (
            seed_dir / f"{strategy}_best.keras"
        )

        assert model_path.exists(), (
            f"Missing checkpoint: {model_path}"
        )

        # Release previous model resources
        tf.keras.backend.clear_session()

        # Load the saved model without recompiling
        model = tf.keras.models.load_model(
            model_path,
            compile=False
        )

        # Generate validation probabilities
        probabilities = model.predict(
            X_val_cnn,
            batch_size=64,
            verbose=0
        )

        assert probabilities.shape == (
            len(y_true), NUM_CLASSES
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        # Compute accuracy and macro F1
        accuracy = accuracy_score(
            y_true,
            predictions
        ) * 100

        macro_f1 = f1_score(
            y_true,
            predictions,
            labels=np.arange(NUM_CLASSES),
            average="macro",
            zero_division=0
        ) * 100

        errors = int(
            np.sum(predictions != y_true)
        )

        # Load the previously saved accuracy
        if seed == 123:
            previous_results = pd.read_csv(
                seed_dir / "replication_results.csv"
            )

            expected_accuracy = previous_results.loc[
                previous_results["Strategy"] == strategy,
                "Validation Accuracy (%)"
            ].iloc[0]

        else:
            previous_results = pd.read_csv(
                seed_dir / f"{strategy}_results.csv"
            )

            expected_accuracy = previous_results[
                "Validation Accuracy (%)"
            ].iloc[0]

        # Check checkpoint consistency
        assert np.isclose(
            accuracy,
            expected_accuracy,
            atol=0.02
        ), (
            f"Checkpoint accuracy mismatch: "
            f"seed {seed}, strategy {strategy}"
        )

        results.append({
            "Seed": seed,
            "Strategy": strategy,
            "Accuracy (%)": accuracy,
            "Macro F1 (%)": macro_f1,
            "Errors": errors
        })

        print(f"Accuracy: {accuracy:.4f}%")
        print(f"Macro F1: {macro_f1:.4f}%")
        print(f"Incorrect predictions: {errors}")
        print("Checkpoint verification: PASSED")

        del model, probabilities, predictions
        gc.collect()

# ============================================================
# 4. DISPLAY RESULTS FOR ALL NINE MODELS
# ============================================================

f1_results_df = pd.DataFrame(results)

print("\n=== MULTI-SEED MACRO F1 RESULTS ===")

print(
    f1_results_df.round(4).to_string(index=False)
)

# ============================================================
# 5. COMPUTE MEAN AND STANDARD DEVIATION
# ============================================================

f1_summary_df = (
    f1_results_df
    .groupby("Strategy", sort=False)
    .agg(
        Runs=("Seed", "count"),
        Mean_Accuracy=("Accuracy (%)", "mean"),
        Std_Accuracy=("Accuracy (%)", "std"),
        Mean_Macro_F1=("Macro F1 (%)", "mean"),
        Std_Macro_F1=("Macro F1 (%)", "std"),
        Mean_Errors=("Errors", "mean")
    )
    .reset_index()
)

f1_summary_df.columns = [
    "Strategy",
    "Runs",
    "Mean Accuracy (%)",
    "Std Accuracy (pp)",
    "Mean Macro F1 (%)",
    "Std Macro F1 (pp)",
    "Mean Errors"
]

print("\n=== MACRO F1 STATISTICAL SUMMARY ===")

print(
    f1_summary_df.round(4).to_string(index=False)
)

# ============================================================
# 6. SAVE RESULTS
# ============================================================

results_path = (
    TABLES_DIR / "multi_seed_macro_f1_results.csv"
)

summary_path = (
    TABLES_DIR / "multi_seed_macro_f1_summary.csv"
)

f1_results_df.to_csv(
    results_path,
    index=False
)

f1_summary_df.to_csv(
    summary_path,
    index=False
)

# ============================================================
# 7. FINAL VERIFICATION
# ============================================================

assert len(f1_results_df) == 9

assert f1_summary_df["Runs"].eq(3).all()

print("\n=== MACRO F1 ANALYSIS VERIFICATION ===")

print("Evaluated checkpoints:", len(f1_results_df))
print("All checkpoint accuracies verified: YES")
print("New training performed: NO")
print("Test set used: NO")

print("\nResults saved to:", results_path)
print("Summary saved to:", summary_path)

print("\nMulti-seed Macro F1 analysis completed successfully!")


## Multi-Seed Per-Class Error Analysis

In [ ]:
# MULTI-SEED PER-CLASS ERROR ANALYSIS


import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

from pathlib import Path
from sklearn.metrics import f1_score

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

REPLICATIONS_DIR = PROJECT_DIR / "replications"
TABLES_DIR = PROJECT_DIR / "results/tables"
FIGURES_DIR = PROJECT_DIR / "results/figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [123, 456, 789]
STRATEGIES = ["baseline", "standard", "light"]
NUM_CLASSES = 28

arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

# ============================================================
# 2. VERIFY DATA AND SAVED CHECKPOINTS
# ============================================================

y_true = np.asarray(y_val_cnn).ravel()

assert X_val_cnn.shape == (2016, 32, 32, 1)
assert len(y_true) == 2016

class_counts = np.bincount(
    y_true,
    minlength=NUM_CLASSES
)

assert np.all(class_counts == 72)

for seed in SEEDS:
    for strategy in STRATEGIES:
        checkpoint = (
            REPLICATIONS_DIR
            / f"seed_{seed}"
            / f"{strategy}_best.keras"
        )
        assert checkpoint.exists(), (
            f"Missing checkpoint: {checkpoint}"
        )

print("=== PER-CLASS DATA VERIFICATION ===")
print("Validation samples:", len(y_true))
print("Samples per class: 72")
print("Saved checkpoints verified: 9")
print("Test set used: NO")

# ============================================================
# 3. CALCULATE PER-CLASS METRICS
# ============================================================

records = []

for seed in SEEDS:
    for strategy in STRATEGIES:

        tf.keras.backend.clear_session()

        model_path = (
            REPLICATIONS_DIR
            / f"seed_{seed}"
            / f"{strategy}_best.keras"
        )

        model = tf.keras.models.load_model(
            model_path,
            compile=False
        )

        probabilities = model.predict(
            X_val_cnn,
            batch_size=64,
            verbose=0
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        # Calculate accuracy separately for each class
        correct_per_class = np.bincount(
            y_true,
            weights=(predictions == y_true).astype(int),
            minlength=NUM_CLASSES
        )

        class_accuracy = (
            correct_per_class / class_counts * 100
        )

        # Calculate class-specific F1-scores
        class_f1 = f1_score(
            y_true,
            predictions,
            labels=np.arange(NUM_CLASSES),
            average=None,
            zero_division=0
        ) * 100

        for class_id in range(NUM_CLASSES):
            records.append({
                "Seed": seed,
                "Strategy": strategy,
                "Class": class_id + 1,
                "Character": arabic_characters[class_id],
                "Class Accuracy (%)": class_accuracy[class_id],
                "Class F1 (%)": class_f1[class_id],
                "Errors": int(
                    class_counts[class_id]
                    - correct_per_class[class_id]
                )
            })

        print(
            f"Evaluated: seed {seed} - {strategy}"
        )

        del model, probabilities, predictions
        gc.collect()

per_class_df = pd.DataFrame(records)

assert len(per_class_df) == 252

# ============================================================
# 4. COMPUTE PAIRED CLASS ACCURACY DIFFERENCES
# ============================================================

paired_class_df = (
    per_class_df.pivot(
        index=["Seed", "Class", "Character"],
        columns="Strategy",
        values="Class Accuracy (%)"
    )
    .reset_index()
)

paired_class_df["Standard Delta (pp)"] = (
    paired_class_df["standard"]
    - paired_class_df["baseline"]
)

paired_class_df["Light Delta (pp)"] = (
    paired_class_df["light"]
    - paired_class_df["baseline"]
)

# ============================================================
# 5. AGGREGATE PERFORMANCE ACROSS SEEDS
# ============================================================

class_summary = (
    paired_class_df
    .groupby(["Class", "Character"], as_index=False)
    [
        [
            "baseline",
            "standard",
            "light",
            "Standard Delta (pp)",
            "Light Delta (pp)"
        ]
    ]
    .mean()
)

class_summary = class_summary.rename(columns={
    "baseline": "Baseline Mean Accuracy (%)",
    "standard": "Standard Mean Accuracy (%)",
    "light": "Light Mean Accuracy (%)",
    "Standard Delta (pp)": "Standard Mean Delta (pp)",
    "Light Delta (pp)": "Light Mean Delta (pp)"
})

# Count how often each class was degraded
degradation_counts = (
    paired_class_df
    .groupby(["Class", "Character"], as_index=False)
    .agg(
        Standard_Degraded_Seeds=(
            "Standard Delta (pp)",
            lambda x: int((x < 0).sum())
        ),
        Light_Degraded_Seeds=(
            "Light Delta (pp)",
            lambda x: int((x < 0).sum())
        )
    )
)

class_summary = class_summary.merge(
    degradation_counts,
    on=["Class", "Character"]
)

# ============================================================
# 6. DISPLAY MAIN RESULTS
# ============================================================

print("\n=== FOCUS CLASSES: 3, 22, 25 ===")

focus_df = class_summary[
    class_summary["Class"].isin([3, 22, 25])
]

print(
    focus_df.round(2).to_string(index=False)
)

print("\n=== TOP 5 DEGRADED CLASSES: STANDARD ===")

print(
    class_summary.nsmallest(
        5,
        "Standard Mean Delta (pp)"
    )[
        [
            "Class",
            "Character",
            "Standard Mean Delta (pp)",
            "Standard_Degraded_Seeds"
        ]
    ].round(2).to_string(index=False)
)

print("\n=== TOP 5 DEGRADED CLASSES: LIGHT ===")

print(
    class_summary.nsmallest(
        5,
        "Light Mean Delta (pp)"
    )[
        [
            "Class",
            "Character",
            "Light Mean Delta (pp)",
            "Light_Degraded_Seeds"
        ]
    ].round(2).to_string(index=False)
)

# ============================================================
# 7. VISUALIZE CLASS-LEVEL DIFFERENCES
# ============================================================

fig, ax = plt.subplots(figsize=(12, 11))

positions = np.arange(NUM_CLASSES)

ax.barh(
    positions - 0.2,
    class_summary["Standard Mean Delta (pp)"],
    height=0.38,
    label="Standard vs Baseline"
)

ax.barh(
    positions + 0.2,
    class_summary["Light Mean Delta (pp)"],
    height=0.38,
    label="Light vs Baseline"
)

ax.axvline(
    0,
    linestyle="--",
    color="gray"
)

ax.set_yticks(positions)
ax.set_yticklabels(
    class_summary["Class"].astype(str)
)

ax.invert_yaxis()

ax.set_xlabel(
    "Mean Accuracy Difference (percentage points)"
)

ax.set_ylabel("AHCD Class ID")

ax.set_title(
    "Per-Class Augmentation Impact Across Three Seeds"
)

ax.legend()
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()

# ============================================================
# 8. SAVE EXPERIMENTAL RESULTS
# ============================================================

details_path = (
    TABLES_DIR / "multi_seed_per_class_results.csv"
)

paired_path = (
    TABLES_DIR / "multi_seed_per_class_paired.csv"
)

summary_path = (
    TABLES_DIR / "multi_seed_per_class_summary.csv"
)

figure_path = (
    FIGURES_DIR / "multi_seed_per_class_analysis.png"
)

per_class_df.to_csv(
    details_path,
    index=False,
    encoding="utf-8-sig"
)

paired_class_df.to_csv(
    paired_path,
    index=False,
    encoding="utf-8-sig"
)

class_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 9. FINAL VERIFICATION
# ============================================================

print("\n=== PER-CLASS ANALYSIS SUMMARY ===")

print("Evaluated models: 9")
print("Analyzed classes: 28")
print("Seeds per strategy: 3")
print("Training performed: NO")
print("Test set used: NO")

print("\nSaved summary:", summary_path)
print("Saved figure:", figure_path)

print("\nPer-class analysis completed successfully!")


## Cross-Seed Confusion Analysis

In [ ]:
# CROSS-SEED CONFUSION ANALYSIS


import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

from pathlib import Path

# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

SEEDS = [123, 456, 789]
STRATEGIES = ["baseline", "standard", "light"]
NUM_CLASSES = 28

arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

y_true = np.asarray(y_val_cnn).ravel()

assert len(y_true) == 2016

# ============================================================
# 2. COLLECT PREDICTIONS FROM ALL NINE CHECKPOINTS
# ============================================================

predictions_by_model = {}

confusions = {
    strategy: np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=int
    )
    for strategy in STRATEGIES
}

for seed in SEEDS:
    for strategy in STRATEGIES:

        tf.keras.backend.clear_session()

        model_path = (
            PROJECT_DIR
            / "replications"
            / f"seed_{seed}"
            / f"{strategy}_best.keras"
        )

        assert model_path.exists(), (
            f"Missing checkpoint: {model_path}"
        )

        model = tf.keras.models.load_model(
            model_path,
            compile=False
        )

        probabilities = model.predict(
            X_val_cnn,
            batch_size=64,
            verbose=0
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        predictions_by_model[
            (seed, strategy)
        ] = predictions

        np.add.at(
            confusions[strategy],
            (y_true, predictions),
            1
        )

        print(
            f"Evaluated: {strategy} - seed {seed}"
        )

        del model, probabilities
        gc.collect()

# ============================================================
# 3. BUILD CLASS-CONFUSION COMPARISON TABLE
# ============================================================

records = []

for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):

        if true_class == predicted_class:
            continue

        counts = {
            strategy: int(
                confusions[strategy][
                    true_class, predicted_class
                ]
            )
            for strategy in STRATEGIES
        }

        if sum(counts.values()) == 0:
            continue

        records.append({
            "True Class": true_class + 1,
            "True Character": arabic_characters[true_class],
            "Predicted Class": predicted_class + 1,
            "Predicted Character": arabic_characters[predicted_class],
            "Baseline Errors": counts["baseline"],
            "Standard Errors": counts["standard"],
            "Light Errors": counts["light"],
            "Standard Error Increase": (
                counts["standard"] - counts["baseline"]
            ),
            "Light Error Increase": (
                counts["light"] - counts["baseline"]
            )
        })

confusion_df = pd.DataFrame(records)

# ============================================================
# 4. DISPLAY MOST INCREASED CONFUSIONS
# ============================================================

display_columns = [
    "True Class",
    "True Character",
    "Predicted Class",
    "Predicted Character",
    "Baseline Errors",
    "Standard Errors",
    "Light Errors"
]

print("\n=== TOP INCREASED CONFUSIONS: STANDARD ===")

print(
    confusion_df.sort_values(
        "Standard Error Increase",
        ascending=False
    )[display_columns + [
        "Standard Error Increase"
    ]].head(8).to_string(index=False)
)

print("\n=== TOP INCREASED CONFUSIONS: LIGHT ===")

print(
    confusion_df.sort_values(
        "Light Error Increase",
        ascending=False
    )[display_columns + [
        "Light Error Increase"
    ]].head(8).to_string(index=False)
)

# ============================================================
# 5. EXAMINE CLASS 27 CONFUSIONS
# ============================================================

print("\n=== CLASS 27 CONFUSION DETAILS ===")

class_27 = confusion_df[
    confusion_df["True Class"] == 27
].sort_values(
    "Light Errors",
    ascending=False
)

print(
    class_27[display_columns].to_string(index=False)
)

# ============================================================
# 6. SELECT AUGMENTATION-RELATED ERROR EXAMPLES
# ============================================================

# Target classes:
# Class 4 = Tha
# Class 27 = Waw

target_classes = [4, 27]

fig, axes = plt.subplots(
    2, 6,
    figsize=(15, 6)
)

for row, original_class in enumerate(target_classes):

    class_id = original_class - 1

    flip_matrix = np.array([
        (
            (predictions_by_model[(seed, "baseline")] == y_true)
            &
            (predictions_by_model[(seed, "light")] != y_true)
        )
        for seed in SEEDS
    ])

    flip_counts = flip_matrix.sum(axis=0)

    candidate_indices = np.where(
        (y_true == class_id)
        & (flip_counts > 0)
    )[0]

    candidate_indices = sorted(
        candidate_indices,
        key=lambda idx: -flip_counts[idx]
    )[:6]

    for col, idx in enumerate(candidate_indices):

        ax = axes[row, col]

        ax.imshow(
            X_val_cnn[idx, :, :, 0],
            cmap="gray",
            vmin=0,
            vmax=1
        )

        ax.set_title(
            f"True: {original_class}\n"
            f"Light-only errors: {flip_counts[idx]}/3",
            fontsize=9
        )

        ax.axis("off")

    for col in range(len(candidate_indices), 6):
        axes[row, col].axis("off")

fig.suptitle(
    "Baseline-Correct and Light-Augmentation-Incorrect Examples",
    fontsize=13
)

plt.tight_layout()

# ============================================================
# 7. SAVE RESULTS
# ============================================================

TABLES_DIR = PROJECT_DIR / "results/tables"
FIGURES_DIR = PROJECT_DIR / "results/figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

table_path = (
    TABLES_DIR / "cross_seed_confusion_analysis.csv"
)

figure_path = (
    FIGURES_DIR / "cross_seed_confusion_examples.png"
)

confusion_df.to_csv(
    table_path,
    index=False,
    encoding="utf-8-sig"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n=== CONFUSION ANALYSIS SUMMARY ===")

print("Evaluated models: 9")
print("Random seeds: 3")
print("Number of classes: 28")
print("Training performed: NO")
print("Test set used: NO")

print("\nTable saved to:", table_path)
print("Figure saved to:", figure_path)

print("\nCross-seed confusion analysis completed successfully!")


## Build an Improved Residual CNN (Advanced CNN Architecture)

In [ ]:
# BUILD AND VERIFY THE RESIDUAL CNN


import tensorflow as tf
import numpy as np
from pathlib import Path

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

RANDOM_SEED = 42
INPUT_SHAPE = (32, 32, 1)
NUM_CLASSES = 28

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_SEED)

# ============================================================
# 2. DEFINE A RESIDUAL BLOCK
# ============================================================

def residual_block(inputs, filters, block_name):

    shortcut = inputs

    # First convolution
    x = tf.keras.layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        name=f"{block_name}_conv1"
    )(inputs)

    x = tf.keras.layers.BatchNormalization(
        name=f"{block_name}_bn1"
    )(x)

    x = tf.keras.layers.Activation(
        "relu",
        name=f"{block_name}_relu1"
    )(x)

    # Second convolution
    x = tf.keras.layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        name=f"{block_name}_conv2"
    )(x)

    x = tf.keras.layers.BatchNormalization(
        name=f"{block_name}_bn2"
    )(x)

    # Match shortcut channels when necessary
    if int(inputs.shape[-1]) != filters:

        shortcut = tf.keras.layers.Conv2D(
            filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            name=f"{block_name}_projection"
        )(shortcut)

        shortcut = tf.keras.layers.BatchNormalization(
            name=f"{block_name}_projection_bn"
        )(shortcut)

    # Add the shortcut to the learned features
    x = tf.keras.layers.Add(
        name=f"{block_name}_add"
    )([x, shortcut])

    x = tf.keras.layers.Activation(
        "relu",
        name=f"{block_name}_output"
    )(x)

    return x

# ============================================================
# 3. DEFINE THE RESIDUAL CNN ARCHITECTURE
# ============================================================

def build_residual_cnn(
    input_shape=(32, 32, 1),
    num_classes=28
):

    inputs = tf.keras.Input(
        shape=input_shape,
        name="input_image"
    )

    # Initial feature extraction at full resolution
    x = tf.keras.layers.Conv2D(
        32,
        kernel_size=3,
        padding="same",
        use_bias=False,
        name="stem_conv"
    )(inputs)

    x = tf.keras.layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = tf.keras.layers.Activation(
        "relu",
        name="stem_relu"
    )(x)

    # Residual stage 1: 32 x 32
    x = residual_block(
        x,
        filters=32,
        block_name="residual_block_1"
    )

    x = tf.keras.layers.MaxPooling2D(
        pool_size=2,
        name="pool_1"
    )(x)

    # Residual stage 2: 16 x 16
    x = residual_block(
        x,
        filters=64,
        block_name="residual_block_2"
    )

    x = tf.keras.layers.MaxPooling2D(
        pool_size=2,
        name="pool_2"
    )(x)

    # Residual stage 3: 8 x 8
    x = residual_block(
        x,
        filters=96,
        block_name="residual_block_3"
    )

    # Classification head
    x = tf.keras.layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = tf.keras.layers.Dense(
        64,
        activation="relu",
        name="dense_features"
    )(x)

    x = tf.keras.layers.Dropout(
        0.30,
        name="dropout"
    )(x)

    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="softmax",
        name="predictions"
    )(x)

    return tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="AHCD_Residual_CNN"
    )

# ============================================================
# 4. BUILD THE MODEL
# ============================================================

residual_cnn = build_residual_cnn(
    INPUT_SHAPE,
    NUM_CLASSES
)

print("=== RESIDUAL CNN ARCHITECTURE ===")
residual_cnn.summary()

# ============================================================
# 5. VERIFY THE MODEL
# ============================================================

assert residual_cnn.input_shape == (
    None, 32, 32, 1
)

assert residual_cnn.output_shape == (
    None, 28
)

assert residual_cnn.count_params() > 0

# Verify the three residual additions
residual_additions = [
    layer
    for layer in residual_cnn.layers
    if isinstance(layer, tf.keras.layers.Add)
]

assert len(residual_additions) == 3

# Test a forward pass without training
sample_images = X_val_cnn[:4]

probabilities = residual_cnn.predict(
    sample_images,
    verbose=0
)

assert probabilities.shape == (4, 28)

assert np.allclose(
    probabilities.sum(axis=1),
    1.0,
    atol=1e-5
)

# ============================================================
# 6. VERIFY PROJECT SAFETY
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

original_baseline_path = (
    PROJECT_DIR
    / "models"
    / "baseline_cnn_original_recovered.keras"
)

assert original_baseline_path.exists(), (
    "Original baseline backup is missing!"
)

# ============================================================
# 7. DISPLAY VERIFICATION RESULTS
# ============================================================

print("\n=== RESIDUAL CNN VERIFICATION ===")

print("Model:", residual_cnn.name)
print("Total parameters:", residual_cnn.count_params())
print("Residual blocks:", len(residual_additions))
print("Input shape:", residual_cnn.input_shape)
print("Output shape:", residual_cnn.output_shape)
print("Probability normalization: PASSED")
print("Original baseline backup: PRESERVED")
print("Data augmentation: NONE")
print("Model training: NOT STARTED")
print("Test set used: NO")

print("\nResidual CNN created successfully!")


## Residual CNN Training (Seed 123)

In [ ]:
# RESIDUAL CNN TRAINING
# EXPERIMENT: RESIDUAL CNN - SEED 123


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 123
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

RUN_DIR = (
    PROJECT_DIR / "advanced_models" / "residual_seed_123"
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

model_path = RUN_DIR / "residual_best.keras"
history_path = RUN_DIR / "residual_history.csv"
results_path = RUN_DIR / "residual_results.csv"

# ============================================================
# 2. VERIFY REQUIRED OBJECTS
# ============================================================

required_objects = [
    "build_residual_cnn",
    "X_train_cnn",
    "y_train_cnn",
    "X_val_cnn",
    "y_val_cnn"
]

for name in required_objects:
    assert name in globals(), f"Missing object: {name}"

# Prevent accidental overwriting
for path in [model_path, history_path, results_path]:
    if path.exists():
        raise RuntimeError(
            f"Existing file: {path.name}. "
            "Stop to prevent overwriting."
        )

# Verify the original baseline backup
baseline_backup = (
    PROJECT_DIR
    / "models"
    / "baseline_cnn_original_recovered.keras"
)

assert baseline_backup.exists(), (
    "Original baseline backup is missing!"
)

# ============================================================
# 3. BUILD A FRESH RESIDUAL CNN
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

residual_model_123 = build_residual_cnn(
    input_shape=(32, 32, 1),
    num_classes=NUM_CLASSES
)

assert residual_model_123.count_params() == 230780

# ============================================================
# 4. COMPILE THE MODEL
# ============================================================

residual_model_123.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

# ============================================================
# 5. PREPARE CONTROLLED DATA PIPELINES
# ============================================================

train_ds_residual = (
    tf.data.Dataset.from_tensor_slices(
        (X_train_cnn, y_train_cnn)
    )
    .shuffle(
        buffer_size=len(X_train_cnn),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_residual = (
    tf.data.Dataset.from_tensor_slices(
        (X_val_cnn, y_val_cnn)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 6. DEFINE TRAINING CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 7. TRAIN THE RESIDUAL CNN
# ============================================================

print("=== RESIDUAL CNN TRAINING ===")

print("Random seed:", SEED)
print("Model parameters:", residual_model_123.count_params())
print("Training samples:", len(X_train_cnn))
print("Validation samples:", len(X_val_cnn))
print("Maximum epochs:", EPOCHS)

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Test set used: NO")

print("\nStarting training...\n")

history_residual_123 = residual_model_123.fit(
    train_ds_residual,
    validation_data=val_ds_residual,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 8. LOAD AND EVALUATE THE BEST CHECKPOINT
# ============================================================

best_residual = tf.keras.models.load_model(
    model_path,
    compile=False
)

probabilities = best_residual.predict(
    X_val_cnn,
    batch_size=BATCH_SIZE,
    verbose=0
)

predictions = np.argmax(
    probabilities,
    axis=1
)

y_true = np.asarray(y_val_cnn).ravel()

validation_accuracy = (
    np.mean(predictions == y_true) * 100
)

macro_f1 = f1_score(
    y_true,
    predictions,
    average="macro",
    labels=np.arange(NUM_CLASSES),
    zero_division=0
) * 100

# Calculate categorical cross-entropy
probabilities_clipped = np.clip(
    probabilities,
    1e-7,
    1.0
)

validation_loss = -np.mean(
    np.log(
        probabilities_clipped[
            np.arange(len(y_true)),
            y_true
        ]
    )
)

best_epoch = int(
    np.argmin(history_residual_123.history["val_loss"])
) + 1

# ============================================================
# 9. SAVE EXPERIMENT RESULTS
# ============================================================

results_df = pd.DataFrame([{
    "Seed": SEED,
    "Model": "Residual CNN",
    "Parameters": residual_model_123.count_params(),
    "Completed Epochs": len(
        history_residual_123.history["loss"]
    ),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": validation_accuracy,
    "Macro F1 (%)": macro_f1,
    "Validation Loss": validation_loss,
    "Errors": int(np.sum(predictions != y_true))
}])

results_df.to_csv(
    results_path,
    index=False
)

# ============================================================
# 10. DISPLAY TRAINING SUMMARY
# ============================================================

print("\n=== RESIDUAL CNN TRAINING SUMMARY ===")

print(
    results_df.round(4).to_string(index=False)
)

print("\n=== BASELINE REFERENCE - SEED 123 ===")
print("Validation Accuracy: 94.4444 %")
print("Validation Loss: 0.1951")

print("\n=== FILE VERIFICATION ===")

print("Model saved:", model_path.exists())
print("History saved:", history_path.exists())
print("Results saved:", results_path.exists())
print("Original baseline backup preserved:", baseline_backup.exists())
print("Test set used: NO")

print("\nResidual CNN experiment completed successfully!")


In [ ]:

# ============================================================
# CHECK INTERRUPTED RESIDUAL CNN TRAINING
# ============================================================

from pathlib import Path
from google.colab import drive
import pandas as pd

# Mount Google Drive if necessary
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# Define experiment directory
RUN_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN/"
    "advanced_models/residual_seed_123"
)

model_path = RUN_DIR / "residual_best.keras"
history_path = RUN_DIR / "residual_history.csv"
results_path = RUN_DIR / "residual_results.csv"

print("=== RESIDUAL TRAINING RECOVERY CHECK ===")

print("Best checkpoint exists:", model_path.exists())
print("Training history exists:", history_path.exists())
print("Final results exist:", results_path.exists())

# Inspect completed epochs
if history_path.exists():

    history_df = pd.read_csv(history_path)

    print("\n=== RECORDED TRAINING PROGRESS ===")
    print("Completed epochs:", len(history_df))

    if len(history_df) > 0:
        print(
            "Best recorded validation loss:",
            round(history_df["val_loss"].min(), 4)
        )

        print(
            "Best recorded epoch:",
            int(history_df["val_loss"].idxmin() + 1)
        )

print("\nRecovery verification completed!")


In [ ]:

# ============================================================
# VERIFY GPU AND RECOVER RESIDUAL CNN RESULTS
# ============================================================

import tensorflow as tf
import pandas as pd
from pathlib import Path
import subprocess

# ============================================================
# 1. CHECK GPU AVAILABILITY
# ============================================================

print("=== HARDWARE VERIFICATION ===")

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU available: YES")

    for gpu in gpus:
        details = tf.config.experimental.get_device_details(gpu)
        print("GPU name:", details.get("device_name", gpu.name))

    gpu_info = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True
    )

    if gpu_info.returncode == 0:
        print("GPU information:", gpu_info.stdout.strip())

else:
    print("GPU available: NO")
    print("Current device: CPU")

# ============================================================
# 2. LOCATE SAVED EXPERIMENT RESULTS
# ============================================================

RUN_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN/"
    "advanced_models/residual_seed_123"
)

results_path = RUN_DIR / "residual_results.csv"
history_path = RUN_DIR / "residual_history.csv"
model_path = RUN_DIR / "residual_best.keras"

assert results_path.exists(), "Final results not found!"
assert history_path.exists(), "Training history not found!"
assert model_path.exists(), "Model checkpoint not found!"

# ============================================================
# 3. DISPLAY FINAL RESULTS
# ============================================================

results_df = pd.read_csv(results_path)
history_df = pd.read_csv(history_path)

print("\n=== RECOVERED RESIDUAL CNN RESULTS ===")
print(results_df.round(4).to_string(index=False))

print("\n=== TRAINING COMPLETION CHECK ===")
print("Completed epochs:", len(history_df))
print("Best epoch:", history_df["val_loss"].idxmin() + 1)
print("Best validation loss:", round(history_df["val_loss"].min(), 4))

# ============================================================
# 4. VERIFY SAVED CHECKPOINT
# ============================================================

print("\n=== FILE VERIFICATION ===")
print("Model checkpoint: FOUND")
print("Training history: FOUND")
print("Final results: FOUND")
print("New training performed: NO")

print("\nVerification completed successfully!")


In [ ]:

# ============================================================
# RESTORE PROJECT CONNECTION
# ============================================================

from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount("/content/drive")

# Define project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

# Verify required files
required_files = {
    "Baseline CNN - Seed 123":
        PROJECT_DIR / "replications/seed_123/baseline_best.keras",

    "Residual CNN - Seed 123":
        PROJECT_DIR / "advanced_models/residual_seed_123/residual_best.keras",

    "Preprocessed Dataset":
        PROJECT_DIR / "checkpoints/preprocessed_ahcd_v1.npz"
}

print("=== PROJECT RESTORATION CHECK ===")

for name, path in required_files.items():
    print(
        f"{name}:",
        "FOUND" if path.exists() else "MISSING"
    )

print("\nReady for Step 7.3!")


## Paired Validation Comparison

In [ ]:
# PAIRED VALIDATION COMPARISON
# BASELINE CNN VS RESIDUAL CNN - SEED 123


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

baseline_path = (
    PROJECT_DIR
    / "replications/seed_123/baseline_best.keras"
)

residual_path = (
    PROJECT_DIR
    / "advanced_models/residual_seed_123/residual_best.keras"
)

dataset_path = (
    PROJECT_DIR
    / "checkpoints/preprocessed_ahcd_v1.npz"
)

for path in [baseline_path, residual_path, dataset_path]:
    assert path.exists(), f"Missing file: {path}"

# ============================================================
# 2. RESTORE VALIDATION DATA ONLY
# ============================================================

with np.load(dataset_path, allow_pickle=False) as data:
    X_validation = data["X_val"]
    y_validation = data["y_val"].ravel()

assert X_validation.shape == (2016, 32, 32, 1)
assert len(y_validation) == 2016

NUM_CLASSES = 28

arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

print("=== VALIDATION DATA VERIFICATION ===")
print("Validation samples:", len(y_validation))
print("Number of classes:", NUM_CLASSES)
print("Test set used: NO")

# ============================================================
# 3. LOAD SAVED CNN MODELS
# ============================================================

tf.keras.backend.clear_session()

baseline_model = tf.keras.models.load_model(
    baseline_path,
    compile=False
)

residual_model = tf.keras.models.load_model(
    residual_path,
    compile=False
)

print("\n=== MODEL VERIFICATION ===")
print("Baseline parameters:", baseline_model.count_params())
print("Residual parameters:", residual_model.count_params())

assert baseline_model.count_params() == 103420
assert residual_model.count_params() == 230780

# ============================================================
# 4. GENERATE VALIDATION PREDICTIONS
# ============================================================

baseline_probs = baseline_model.predict(
    X_validation,
    batch_size=64,
    verbose=0
)

residual_probs = residual_model.predict(
    X_validation,
    batch_size=64,
    verbose=0
)

baseline_preds = np.argmax(baseline_probs, axis=1)
residual_preds = np.argmax(residual_probs, axis=1)

# ============================================================
# 5. COMPARE GLOBAL PERFORMANCE
# ============================================================

comparison_results = []

for name, predictions in [
    ("Baseline CNN", baseline_preds),
    ("Residual CNN", residual_preds)
]:

    comparison_results.append({
        "Model": name,
        "Accuracy (%)": accuracy_score(
            y_validation, predictions
        ) * 100,
        "Macro F1 (%)": f1_score(
            y_validation,
            predictions,
            labels=np.arange(NUM_CLASSES),
            average="macro",
            zero_division=0
        ) * 100,
        "Errors": int(
            np.sum(predictions != y_validation)
        )
    })

comparison_df = pd.DataFrame(comparison_results)

print("\n=== BASELINE VS RESIDUAL COMPARISON ===")
print(comparison_df.round(4).to_string(index=False))

# Verify expected checkpoint performance
assert np.isclose(
    comparison_df.iloc[0]["Accuracy (%)"],
    94.4444,
    atol=0.02
), "Baseline checkpoint does not match expected results!"

assert np.isclose(
    comparison_df.iloc[1]["Accuracy (%)"],
    97.8175,
    atol=0.02
), "Residual checkpoint does not match expected results!"

# ============================================================
# 6. ANALYZE PAIRED PREDICTION OUTCOMES
# ============================================================

baseline_correct = baseline_preds == y_validation
residual_correct = residual_preds == y_validation

both_correct = int(np.sum(
    baseline_correct & residual_correct
))

baseline_only = int(np.sum(
    baseline_correct & ~residual_correct
))

residual_only = int(np.sum(
    ~baseline_correct & residual_correct
))

both_wrong = int(np.sum(
    ~baseline_correct & ~residual_correct
))

assert (
    both_correct + baseline_only
    + residual_only + both_wrong
) == len(y_validation)

print("\n=== PAIRED PREDICTION ANALYSIS ===")

print("Correct by both:", both_correct)
print("Correct only by baseline:", baseline_only)
print("Correct only by residual:", residual_only)
print("Incorrect by both:", both_wrong)

print(
    "Net additional correct predictions:",
    residual_only - baseline_only
)

# ============================================================
# 7. ANALYZE PERFORMANCE BY CLASS
# ============================================================

class_results = []

for class_id in range(NUM_CLASSES):

    mask = y_validation == class_id

    baseline_acc = np.mean(
        baseline_correct[mask]
    ) * 100

    residual_acc = np.mean(
        residual_correct[mask]
    ) * 100

    class_results.append({
        "Class": class_id + 1,
        "Character": arabic_characters[class_id],
        "Samples": int(np.sum(mask)),
        "Baseline Accuracy (%)": baseline_acc,
        "Residual Accuracy (%)": residual_acc,
        "Improvement (pp)": residual_acc - baseline_acc
    })

class_df = pd.DataFrame(class_results)

print("\n=== TOP 5 IMPROVED CLASSES ===")

print(
    class_df.nlargest(
        5, "Improvement (pp)"
    ).round(2).to_string(index=False)
)

print("\n=== TOP 5 DEGRADED CLASSES ===")

print(
    class_df.nsmallest(
        5, "Improvement (pp)"
    ).round(2).to_string(index=False)
)

# ============================================================
# 8. SAVE COMPARISON RESULTS
# ============================================================

TABLES_DIR = PROJECT_DIR / "results/tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

comparison_path = (
    TABLES_DIR / "residual_vs_baseline_seed123.csv"
)

class_path = (
    TABLES_DIR / "residual_per_class_seed123.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

class_df.to_csv(
    class_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 9. FINAL VERIFICATION
# ============================================================

print("\n=== COMPARISON VERIFICATION ===")

print("Models evaluated: 2")
print("Random seed: 123")
print("Same validation images: YES")
print("New training performed: NO")
print("Test set used: NO")

print("\nComparison saved to:", comparison_path)
print("Per-class results saved to:", class_path)

print("\nPaired comparison completed successfully!")


## Residual CNN Training with Seed 456

In [ ]:
# RESIDUAL CNN TRAINING - SEED 456


import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score

# ============================================================
# 1. DEFINE EXPERIMENT PATHS
# ============================================================

SEED = 456
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

reference_path = (
    PROJECT_DIR
    / "advanced_models/residual_seed_123/residual_best.keras"
)

dataset_path = (
    PROJECT_DIR
    / "checkpoints/preprocessed_ahcd_v1.npz"
)

RUN_DIR = (
    PROJECT_DIR / "advanced_models/residual_seed_456"
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

model_path = RUN_DIR / "residual_best.keras"
history_path = RUN_DIR / "residual_history.csv"
results_path = RUN_DIR / "residual_results.csv"

# ============================================================
# 2. VERIFY SAVED FILES
# ============================================================

assert reference_path.exists(), (
    "Seed 123 reference checkpoint not found!"
)

assert dataset_path.exists(), (
    "Preprocessed dataset not found!"
)

# Prevent overwriting an existing experiment
for path in [model_path, history_path, results_path]:
    if path.exists():
        raise RuntimeError(
            f"Existing experiment file: {path.name}. "
            "Stop to prevent overwriting."
        )

# ============================================================
# 3. RESTORE TRAINING AND VALIDATION DATA
# ============================================================

with np.load(dataset_path, allow_pickle=False) as data:

    X_train = data["X_train"]
    y_train = data["y_train"].ravel()

    X_val = data["X_val"]
    y_val = data["y_val"].ravel()

assert X_train.shape == (11423, 32, 32, 1)
assert X_val.shape == (2016, 32, 32, 1)

# ============================================================
# 4. BUILD A FRESH RESIDUAL CNN
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

# Load architecture from the saved reference
reference_model = tf.keras.models.load_model(
    reference_path,
    compile=False
)

# Clone architecture without copying trained weights
tf.keras.utils.set_random_seed(SEED)

residual_model_456 = tf.keras.models.clone_model(
    reference_model
)

assert residual_model_456.count_params() == 230780

# Confirm that learned weights were not copied
same_weights = all(
    np.array_equal(a, b)
    for a, b in zip(
        reference_model.get_weights(),
        residual_model_456.get_weights()
    )
)

assert not same_weights, (
    "New model unexpectedly contains reference weights!"
)

del reference_model

print("=== MODEL INITIALIZATION VERIFICATION ===")
print("Random seed:", SEED)
print("Parameters:", residual_model_456.count_params())
print("Fresh model weights: VERIFIED")

# ============================================================
# 5. COMPILE THE MODEL
# ============================================================

residual_model_456.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

# ============================================================
# 6. PREPARE DATASETS
# ============================================================

train_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train, y_train)
    )
    .shuffle(
        buffer_size=len(X_train),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_456 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val, y_val)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 7. DEFINE TRAINING CALLBACKS
# ============================================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 8. TRAIN RESIDUAL CNN
# ============================================================

print("\n=== RESIDUAL CNN - SEED 456 TRAINING ===")

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Maximum epochs:", EPOCHS)
print("Test set used: NO")

history_456 = residual_model_456.fit(
    train_ds_456,
    validation_data=val_ds_456,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ============================================================
# 9. EVALUATE THE BEST CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    model_path,
    compile=False
)

probabilities = best_model.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=0
)

predictions = np.argmax(
    probabilities,
    axis=1
)

accuracy = accuracy_score(
    y_val, predictions
) * 100

macro_f1 = f1_score(
    y_val,
    predictions,
    labels=np.arange(NUM_CLASSES),
    average="macro",
    zero_division=0
) * 100

validation_loss = -np.mean(
    np.log(
        np.clip(
            probabilities[np.arange(len(y_val)), y_val],
            1e-7,
            1.0
        )
    )
)

best_epoch = int(
    np.argmin(history_456.history["val_loss"])
) + 1

# ============================================================
# 10. SAVE EXPERIMENT RESULTS
# ============================================================

results_df = pd.DataFrame([{
    "Seed": SEED,
    "Model": "Residual CNN",
    "Parameters": best_model.count_params(),
    "Completed Epochs": len(history_456.history["loss"]),
    "Best Epoch": best_epoch,
    "Validation Accuracy (%)": accuracy,
    "Macro F1 (%)": macro_f1,
    "Validation Loss": validation_loss,
    "Errors": int(np.sum(predictions != y_val))
}])

results_df.to_csv(
    results_path,
    index=False
)

# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

print("\n=== RESIDUAL CNN SEED 456 SUMMARY ===")
print(results_df.round(4).to_string(index=False))

print("\n=== FILE VERIFICATION ===")
print("Best checkpoint:", model_path.exists())
print("Training history:", history_path.exists())
print("Experiment results:", results_path.exists())
print("Seed 123 checkpoint preserved:", reference_path.exists())
print("Test set used: NO")

print("\nSeed 456 training completed successfully!")


## Residual CNN Training (Seed 789)

In [ ]:
# RESIDUAL CNN TRAINING - SEED 789


import tensorflow as tf
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

SEED = 789
BATCH_SIZE = 64
EPOCHS = 30
NUM_CLASSES = 28

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

reference_path = (
    PROJECT_DIR
    / "advanced_models/residual_seed_123/residual_best.keras"
)

dataset_path = (
    PROJECT_DIR
    / "checkpoints/preprocessed_ahcd_v1.npz"
)

RUN_DIR = (
    PROJECT_DIR
    / "advanced_models/residual_seed_789"
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

model_path = RUN_DIR / "residual_best.keras"
history_path = RUN_DIR / "residual_history.csv"
results_path = RUN_DIR / "residual_results.csv"

# ============================================================
# 2. PROTECT PREVIOUS EXPERIMENTS
# ============================================================

assert reference_path.exists(), "Reference model is missing!"
assert dataset_path.exists(), "Preprocessed dataset is missing!"

for path in [model_path, history_path, results_path]:
    if path.exists():
        raise RuntimeError(
            f"Existing experiment file: {path}. "
            "Stop to prevent overwriting."
        )

print("=== EXPERIMENT VERIFICATION ===")
print("Seed:", SEED)
print("Output directory:", RUN_DIR)
print("Previous experiments will not be modified.")

# ============================================================
# 3. RESTORE TRAINING AND VALIDATION DATA
# ============================================================

with np.load(dataset_path, allow_pickle=False) as data:
    X_train = data["X_train"]
    y_train = data["y_train"].ravel()

    X_val = data["X_val"]
    y_val = data["y_val"].ravel()

assert X_train.shape == (11423, 32, 32, 1)
assert X_val.shape == (2016, 32, 32, 1)

assert len(y_train) == 11423
assert len(y_val) == 2016

print("\n=== DATA VERIFICATION ===")
print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Test set used: NO")

# ============================================================
# 4. BUILD FRESH RESIDUAL CNN
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

reference_model = tf.keras.models.load_model(
    reference_path,
    compile=False
)

# Clone architecture without copying pretrained weights
residual_model_789 = tf.keras.models.clone_model(
    reference_model
)

assert residual_model_789.count_params() == 230780

same_weights = all(
    np.array_equal(a, b)
    for a, b in zip(
        reference_model.get_weights(),
        residual_model_789.get_weights()
    )
)

assert not same_weights, (
    "Fresh model unexpectedly contains trained weights!"
)

del reference_model

print("\n=== MODEL INITIALIZATION VERIFICATION ===")
print("Random seed:", SEED)
print("Parameters:", residual_model_789.count_params())
print("Fresh model weights: VERIFIED")

# ============================================================
# 5. COMPILE MODEL
# ============================================================

residual_model_789.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        )
    ]
)

# ============================================================
# 6. CREATE DATA PIPELINES
# ============================================================

train_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_train, y_train)
    )
    .shuffle(
        buffer_size=len(X_train),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds_789 = (
    tf.data.Dataset.from_tensor_slices(
        (X_val, y_val)
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# ============================================================
# 7. TRAINING CALLBACKS
# ============================================================

callbacks_789 = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        str(history_path)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ============================================================
# 8. TRAIN RESIDUAL CNN - SEED 789
# ============================================================

print("\n=== RESIDUAL CNN - SEED 789 TRAINING ===")

print(
    "Device:",
    "GPU" if tf.config.list_physical_devices("GPU")
    else "CPU"
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Maximum epochs:", EPOCHS)
print("Test set used: NO")

history_789 = residual_model_789.fit(
    train_ds_789,
    validation_data=val_ds_789,
    epochs=EPOCHS,
    callbacks=callbacks_789,
    verbose=2
)

# ============================================================
# 9. RESTORE BEST CHECKPOINT
# ============================================================

best_model_789 = tf.keras.models.load_model(
    model_path,
    compile=False
)

probabilities_789 = best_model_789.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=0
)

predictions_789 = np.argmax(
    probabilities_789,
    axis=1
)

# ============================================================
# 10. COMPUTE VALIDATION METRICS
# ============================================================

accuracy_789 = accuracy_score(
    y_val,
    predictions_789
) * 100

macro_f1_789 = f1_score(
    y_val,
    predictions_789,
    labels=np.arange(NUM_CLASSES),
    average="macro",
    zero_division=0
) * 100

validation_loss_789 = -np.mean(
    np.log(
        np.clip(
            probabilities_789[
                np.arange(len(y_val)),
                y_val
            ],
            1e-7,
            1.0
        )
    )
)

best_epoch_789 = int(
    np.argmin(history_789.history["val_loss"])
) + 1

# ============================================================
# 11. SAVE RESULTS
# ============================================================

results_df_789 = pd.DataFrame([{
    "Seed": SEED,
    "Model": "Residual CNN",
    "Parameters": best_model_789.count_params(),
    "Completed Epochs": len(history_789.history["loss"]),
    "Best Epoch": best_epoch_789,
    "Validation Accuracy (%)": accuracy_789,
    "Macro F1 (%)": macro_f1_789,
    "Validation Loss": validation_loss_789,
    "Errors": int(
        np.sum(predictions_789 != y_val)
    )
}])

results_df_789.to_csv(
    results_path,
    index=False
)

# ============================================================
# 12. DISPLAY EXPERIMENT SUMMARY
# ============================================================

print("\n=== RESIDUAL CNN SEED 789 SUMMARY ===")

print(
    results_df_789.round(4).to_string(index=False)
)

print("\n=== FILE VERIFICATION ===")
print("Best checkpoint:", model_path.exists())
print("Training history:", history_path.exists())
print("Experiment results:", results_path.exists())
print("Test set used: NO")

print("\nSeed 789 training completed successfully!")


## Final Multi-Seed Statistical Comparison

In [ ]:
# FINAL MULTI-SEED STATISTICAL COMPARISON


import tensorflow as tf
import numpy as np
import pandas as pd
import gc

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

SEEDS = [123, 456, 789]

TABLES_DIR = PROJECT_DIR / "results/tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

dataset_path = (
    PROJECT_DIR / "checkpoints/preprocessed_ahcd_v1.npz"
)

# ============================================================
# 2. LOAD VALIDATION DATA ONLY
# ============================================================

with np.load(dataset_path, allow_pickle=False) as data:
    X_val = data["X_val"]
    y_val = data["y_val"].ravel()

assert X_val.shape == (2016, 32, 32, 1)
assert len(y_val) == 2016

# ============================================================
# 3. EVALUATE ALL SAVED MODELS
# ============================================================

records = []

for seed in SEEDS:

    model_paths = {
        "Baseline CNN": (
            PROJECT_DIR
            / f"replications/seed_{seed}/baseline_best.keras"
        ),
        "Residual CNN": (
            PROJECT_DIR
            / f"advanced_models/residual_seed_{seed}/residual_best.keras"
        )
    }

    for model_name, model_path in model_paths.items():

        assert model_path.exists(), (
            f"Missing model: {model_path}"
        )

        tf.keras.backend.clear_session()

        model = tf.keras.models.load_model(
            model_path,
            compile=False
        )

        probabilities = model.predict(
            X_val,
            batch_size=64,
            verbose=0
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        accuracy = accuracy_score(
            y_val, predictions
        ) * 100

        macro_f1 = f1_score(
            y_val,
            predictions,
            labels=np.arange(28),
            average="macro",
            zero_division=0
        ) * 100

        errors = int(
            np.sum(predictions != y_val)
        )

        records.append({
            "Seed": seed,
            "Model": model_name,
            "Accuracy (%)": accuracy,
            "Macro F1 (%)": macro_f1,
            "Errors": errors,
            "Parameters": model.count_params()
        })

        print(
            f"Evaluated: {model_name} | "
            f"Seed: {seed} | "
            f"Accuracy: {accuracy:.4f}%"
        )

        del model
        gc.collect()

# ============================================================
# 4. COMPUTE MULTI-SEED STATISTICS
# ============================================================

raw_df = pd.DataFrame(records)

summary_df = (
    raw_df.groupby("Model")
    .agg(
        Mean_Accuracy=("Accuracy (%)", "mean"),
        SD_Accuracy=("Accuracy (%)", "std"),
        Mean_Macro_F1=("Macro F1 (%)", "mean"),
        SD_Macro_F1=("Macro F1 (%)", "std"),
        Mean_Errors=("Errors", "mean")
    )
    .reset_index()
)

# ============================================================
# 5. COMPUTE PAIRED SEED IMPROVEMENTS
# ============================================================

paired_df = raw_df.pivot(
    index="Seed",
    columns="Model",
    values="Accuracy (%)"
)

paired_df["Improvement (pp)"] = (
    paired_df["Residual CNN"]
    - paired_df["Baseline CNN"]
)

# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("\n=== ALL EXPERIMENTS ===")
print(raw_df.round(4).to_string(index=False))

print("\n=== FINAL STATISTICAL SUMMARY ===")
print(summary_df.round(4).to_string(index=False))

print("\n=== PAIRED SEED IMPROVEMENTS ===")
print(paired_df.round(4).to_string())

print("\n=== OVERALL IMPROVEMENT ===")
print(
    "Mean improvement (pp):",
    round(paired_df["Improvement (pp)"].mean(), 4)
)

print(
    "SD of improvement (pp):",
    round(paired_df["Improvement (pp)"].std(), 4)
)

# ============================================================
# 7. SAVE FINAL TABLES
# ============================================================

raw_df.to_csv(
    TABLES_DIR / "cnn_architecture_multiseed_raw.csv",
    index=False
)

summary_df.to_csv(
    TABLES_DIR / "cnn_architecture_multiseed_summary.csv",
    index=False
)

paired_df.to_csv(
    TABLES_DIR / "cnn_architecture_paired_improvements.csv"
)

print("\n=== EXPERIMENT VERIFICATION ===")
print("Models evaluated:", len(raw_df))
print("Seeds:", SEEDS)
print("Validation samples:", len(y_val))
print("New training performed: NO")
print("Test set used: NO")
print("Tables saved: 3")

print("\nMulti-seed comparison completed successfully!")


## Final Architecture Comparison Figures

In [ ]:
# FINAL ARCHITECTURE COMPARISON FIGURES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

TABLES_DIR = PROJECT_DIR / "results/tables"
FIGURES_DIR = PROJECT_DIR / "results/figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD SAVED EXPERIMENT RESULTS
# ============================================================

raw_df = pd.read_csv(
    TABLES_DIR / "cnn_architecture_multiseed_raw.csv"
)

paired_df = pd.read_csv(
    TABLES_DIR / "cnn_architecture_paired_improvements.csv"
)

summary_df = pd.read_csv(
    TABLES_DIR / "cnn_architecture_multiseed_summary.csv"
)

assert len(raw_df) == 6
assert len(paired_df) == 3
assert len(summary_df) == 2

print("=== DATA VERIFICATION ===")
print("Experiments:", len(raw_df))
print("Random seeds:", sorted(raw_df["Seed"].unique()))
print("Test set used: NO")

# ============================================================
# 3. DEFINE FIGURE COLORS
# ============================================================

baseline_color = "#D78CA8"
residual_color = "#8570C7"

# ============================================================
# 4. FIGURE 1: ACCURACY BY RANDOM SEED
# ============================================================

accuracy_table = raw_df.pivot(
    index="Seed",
    columns="Model",
    values="Accuracy (%)"
).sort_index()

seeds = accuracy_table.index.to_numpy()
x = np.arange(len(seeds))
width = 0.34

fig, ax = plt.subplots(figsize=(9, 5.5))

bars_baseline = ax.bar(
    x - width / 2,
    accuracy_table["Baseline CNN"],
    width,
    label="Baseline CNN",
    color=baseline_color
)

bars_residual = ax.bar(
    x + width / 2,
    accuracy_table["Residual CNN"],
    width,
    label="Residual CNN",
    color=residual_color
)

ax.bar_label(
    bars_baseline,
    fmt="%.2f%%",
    padding=3,
    fontsize=9
)

ax.bar_label(
    bars_residual,
    fmt="%.2f%%",
    padding=3,
    fontsize=9
)

ax.set_title(
    "Baseline CNN vs Residual CNN Across Random Seeds",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel("Random Seed")
ax.set_ylabel("Validation Accuracy (%)")

ax.set_xticks(x)
ax.set_xticklabels(seeds)

ax.set_ylim(0, 105)
ax.legend(loc="lower right")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()

figure_1_path = (
    FIGURES_DIR / "architecture_accuracy_by_seed.png"
)

plt.savefig(
    figure_1_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 5. FIGURE 2: PAIRED ACCURACY IMPROVEMENT
# ============================================================

improvements = paired_df["Improvement (pp)"].to_numpy()
mean_improvement = improvements.mean()

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    paired_df["Seed"].astype(str),
    improvements,
    color=residual_color,
    width=0.5
)

ax.bar_label(
    bars,
    fmt="+%.2f pp",
    padding=5,
    fontsize=11
)

ax.axhline(
    mean_improvement,
    linestyle="--",
    linewidth=1.8,
    color=baseline_color,
    label=f"Mean Improvement: +{mean_improvement:.2f} pp"
)

ax.set_title(
    "Residual CNN Improvement Over Baseline",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel("Random Seed")
ax.set_ylabel("Accuracy Improvement (percentage points)")

ax.set_ylim(0, 4.5)
ax.legend(loc="upper right")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax.set_axisbelow(True)

plt.tight_layout()

figure_2_path = (
    FIGURES_DIR / "architecture_paired_improvement.png"
)

plt.savefig(
    figure_2_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 6. VERIFY SAVED FIGURES
# ============================================================

print("\n=== FIGURE VERIFICATION ===")

print("Figure 1 saved:", figure_1_path.exists())
print("Figure 2 saved:", figure_2_path.exists())

print("\nMean accuracy improvement:")
print(f"+{mean_improvement:.4f} percentage points")

print("\nNew training performed: NO")
print("Test set used: NO")

print("\nArchitecture comparison figures completed!")


# Explainable AI (Grad-CAM)

### First Grad-CAM Visualization

In [ ]:
# GRAD-CAM VISUALIZATION
# RESIDUAL CNN - SEED 789


import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

MODEL_PATH = (
    PROJECT_DIR
    / "advanced_models/residual_seed_789/residual_best.keras"
)

DATA_PATH = (
    PROJECT_DIR
    / "checkpoints/preprocessed_ahcd_v1.npz"
)

FIGURES_DIR = PROJECT_DIR / "results/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_PATH.exists()
assert DATA_PATH.exists()

# ============================================================
# 2. LOAD VALIDATION DATA AND TRAINED MODEL
# ============================================================

with np.load(DATA_PATH, allow_pickle=False) as data:
    X_val = data["X_val"]
    y_val = data["y_val"].ravel()

assert X_val.shape == (2016, 32, 32, 1)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

# ============================================================
# 3. IDENTIFY THE LAST RESIDUAL FEATURE MAP
# ============================================================

target_layer = model.get_layer(
    "residual_block_3_output"
)

prediction_layer = model.get_layer(
    "predictions"
)

# Extract feature maps and the input to the final classifier
grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[
        target_layer.output,
        prediction_layer.input
    ]
)

print("=== GRAD-CAM MODEL VERIFICATION ===")
print("Model:", model.name)
print("Parameters:", model.count_params())
print("Target layer:", target_layer.name)
print("Feature map shape:", target_layer.output.shape)
print("Test set used: NO")

# ============================================================
# 4. GENERATE VALIDATION PREDICTIONS
# ============================================================

probabilities = model.predict(
    X_val,
    batch_size=64,
    verbose=0
)

predictions = np.argmax(
    probabilities,
    axis=1
)

correct_indices = np.flatnonzero(
    predictions == y_val
)

wrong_indices = np.flatnonzero(
    predictions != y_val
)

assert len(correct_indices) > 0
assert len(wrong_indices) > 0

# Select the first correct and first incorrect example
selected_indices = [
    int(correct_indices[0]),
    int(wrong_indices[0])
]

print("\n=== SELECTED VALIDATION EXAMPLES ===")

for index in selected_indices:
    print(
        f"Index: {index} | "
        f"True: {arabic_characters[y_val[index]]} | "
        f"Predicted: {arabic_characters[predictions[index]]}"
    )

# ============================================================
# 5. DEFINE GRAD-CAM FUNCTION
# ============================================================

def compute_gradcam(image, class_id):

    image_tensor = tf.convert_to_tensor(
        image[np.newaxis, ...],
        dtype=tf.float32
    )

    with tf.GradientTape() as tape:

        feature_maps, classifier_input = grad_model(
            image_tensor,
            training=False
        )

        # Compute pre-softmax class scores
        logits = tf.linalg.matmul(
            classifier_input,
            prediction_layer.kernel
        )

        if prediction_layer.bias is not None:
            logits = logits + prediction_layer.bias

        class_score = logits[:, class_id]

    gradients = tape.gradient(
        class_score,
        feature_maps
    )

    if gradients is None:
        raise RuntimeError(
            "Grad-CAM gradients could not be computed."
        )

    # Calculate channel importance weights
    weights = tf.reduce_mean(
        gradients,
        axis=(1, 2),
        keepdims=True
    )

    # Weighted combination of feature maps
    heatmap = tf.reduce_sum(
        feature_maps * weights,
        axis=-1
    )[0]

    heatmap = tf.nn.relu(heatmap)

    max_value = tf.reduce_max(heatmap)

    heatmap = heatmap / (
        max_value + 1e-8
    )

    # Resize to original input resolution
    heatmap = tf.image.resize(
        heatmap[None, ..., None],
        size=(32, 32),
        method="bilinear"
    )

    return heatmap.numpy()[0, :, :, 0]

# ============================================================
# 6. VISUALIZE CORRECT AND INCORRECT PREDICTIONS
# ============================================================

fig, axes = plt.subplots(
    2, 3,
    figsize=(11, 7)
)

for row, index in enumerate(selected_indices):

    image = X_val[index, :, :, 0]
    true_class = int(y_val[index])
    pred_class = int(predictions[index])

    heatmap = compute_gradcam(
        X_val[index],
        pred_class
    )

    is_correct = true_class == pred_class

    status = (
        "Correct Prediction"
        if is_correct
        else "Incorrect Prediction"
    )

    # Original image
    axes[row, 0].imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[row, 0].set_title(
        f"{status}\nOriginal Image"
    )

    # Grad-CAM heatmap
    axes[row, 1].imshow(
        heatmap,
        cmap="magma",
        vmin=0,
        vmax=1
    )

    axes[row, 1].set_title(
        "Grad-CAM Heatmap"
    )

    # Heatmap overlay
    axes[row, 2].imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[row, 2].imshow(
        heatmap,
        cmap="magma",
        alpha=0.45,
        vmin=0,
        vmax=1
    )

    axes[row, 2].set_title(
        "Grad-CAM Overlay"
    )

    for ax in axes[row]:
        ax.axis("off")

plt.suptitle(
    "Residual CNN - Grad-CAM Validation Examples",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()

# ============================================================
# 7. SAVE FIGURE
# ============================================================

FIGURE_PATH = (
    FIGURES_DIR
    / "gradcam_validation_seed789_examples.png"
)

plt.savefig(
    FIGURE_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 8. VERIFY OUTPUT
# ============================================================

print("\n=== GRAD-CAM VERIFICATION ===")
print("Model seed: 789")
print("Examples visualized:", len(selected_indices))
print("Figure saved:", FIGURE_PATH.exists())
print("New training performed: NO")
print("Test set used: NO")

print("\nGrad-CAM visualization completed!")


## Understanding the Confusion Between ر and ز

In [ ]:
# GRAD-CAM ANALYSIS OF CONFUSABLE CHARACTERS
# CASE STUDY: RA (ر) VS ZAY (ز)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# ============================================================
# 1. VERIFY PREVIOUS STEP
# ============================================================

required_variables = [
    "X_val", "y_val", "predictions",
    "probabilities", "compute_gradcam"
]

for name in required_variables:
    assert name in globals(), (
        f"Missing variable: {name}. "
        "Execute Step 8.1 first."
    )

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

FIGURES_DIR = PROJECT_DIR / "results/figures"
TABLES_DIR = PROJECT_DIR / "results/tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. DEFINE TARGET CHARACTERS
# ============================================================

RA_ID = 9
ZAY_ID = 10

character_names = {
    RA_ID: "Ra",
    ZAY_ID: "Zay"
}

# ============================================================
# 3. SELECT VALIDATION EXAMPLES
# ============================================================

correct_ra = np.flatnonzero(
    (y_val == RA_ID) &
    (predictions == RA_ID)
)

correct_zay = np.flatnonzero(
    (y_val == ZAY_ID) &
    (predictions == ZAY_ID)
)

ra_to_zay = np.flatnonzero(
    (y_val == RA_ID) &
    (predictions == ZAY_ID)
)

assert len(correct_ra) > 0
assert len(correct_zay) > 0
assert len(ra_to_zay) > 0

# Select representative correctly classified examples
ra_index = int(correct_ra[
    np.argmax(probabilities[correct_ra, RA_ID])
])

zay_index = int(correct_zay[
    np.argmax(probabilities[correct_zay, ZAY_ID])
])

# Reuse the misclassified example from Step 8.1
error_index = (
    46 if 46 in ra_to_zay
    else int(ra_to_zay[0])
)

selected_examples = [
    ("Correct Ra", ra_index),
    ("Correct Zay", zay_index),
    ("Ra Misclassified as Zay", error_index)
]

print("=== SELECTED EXAMPLES ===")

records = []

for description, index in selected_examples:

    ra_probability = float(
        probabilities[index, RA_ID]
    )

    zay_probability = float(
        probabilities[index, ZAY_ID]
    )

    records.append({
        "Example": description,
        "Index": index,
        "True Class": character_names.get(
            int(y_val[index]), "Other"
        ),
        "Predicted Class": character_names.get(
            int(predictions[index]), "Other"
        ),
        "Ra Probability": ra_probability,
        "Zay Probability": zay_probability
    })

results_df = pd.DataFrame(records)

print(results_df.round(4).to_string(index=False))

# ============================================================
# 4. COMPARE CLASS-SPECIFIC GRAD-CAM MAPS
# ============================================================

fig, axes = plt.subplots(
    3, 3,
    figsize=(11, 11)
)

for row, (description, index) in enumerate(
    selected_examples
):

    image = X_val[index, :, :, 0]

    ra_heatmap = compute_gradcam(
        X_val[index], RA_ID
    )

    zay_heatmap = compute_gradcam(
        X_val[index], ZAY_ID
    )

    # Original handwritten character
    axes[row, 0].imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[row, 0].set_title(
        f"{description}\nOriginal Image"
    )

    # Grad-CAM for the Ra class
    axes[row, 1].imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[row, 1].imshow(
        ra_heatmap,
        cmap="magma",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    axes[row, 1].set_title(
        f"Grad-CAM: Ra\n"
        f"P = {probabilities[index, RA_ID]:.3f}"
    )

    # Grad-CAM for the Zay class
    axes[row, 2].imshow(
        image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[row, 2].imshow(
        zay_heatmap,
        cmap="magma",
        alpha=0.5,
        vmin=0,
        vmax=1
    )

    axes[row, 2].set_title(
        f"Grad-CAM: Zay\n"
        f"P = {probabilities[index, ZAY_ID]:.3f}"
    )

    for ax in axes[row]:
        ax.axis("off")

plt.suptitle(
    "Class-Specific Grad-CAM: Ra vs Zay",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()

# ============================================================
# 5. SAVE FIGURE AND RESULTS
# ============================================================

figure_path = (
    FIGURES_DIR / "gradcam_ra_vs_zay_seed789.png"
)

table_path = (
    TABLES_DIR / "gradcam_ra_vs_zay_examples.csv"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

results_df.to_csv(
    table_path,
    index=False
)

# ============================================================
# 6. FINAL VERIFICATION
# ============================================================

print("\n=== GRAD-CAM CASE STUDY VERIFICATION ===")
print("Examples analyzed:", len(selected_examples))
print("Figure saved:", figure_path.exists())
print("Results table saved:", table_path.exists())
print("New training performed: NO")
print("Test set used: NO")

print("\nGrad-CAM case study completed!")


# Final Evaluation on the Unseen Test Set

In [ ]:
# FINAL TEST SET EVALUATION
# RESIDUAL CNN - SEED 789


import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

MODEL_PATH = (
    PROJECT_DIR
    / "advanced_models/residual_seed_789/residual_best.keras"
)

DATA_PATH = (
    PROJECT_DIR
    / "checkpoints/preprocessed_ahcd_v1.npz"
)

TABLES_DIR = PROJECT_DIR / "results/tables"
FIGURES_DIR = PROJECT_DIR / "results/figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = (
    TABLES_DIR / "final_test_metrics_seed789.csv"
)

# Prevent accidental repeated final evaluation
if METRICS_PATH.exists():
    raise RuntimeError(
        "Final test results already exist. "
        "Do not rerun without reviewing the saved evaluation."
    )

assert MODEL_PATH.exists()
assert DATA_PATH.exists()

# ============================================================
# 2. LOAD THE UNSEEN TEST SET
# ============================================================

with np.load(DATA_PATH, allow_pickle=False) as data:
    X_test = data["X_test"]
    y_test = data["y_test"].ravel()

assert X_test.shape == (3360, 32, 32, 1)
assert len(y_test) == 3360

assert np.array_equal(
    np.unique(y_test),
    np.arange(28)
)

arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

print("=== FINAL TEST DATA VERIFICATION ===")
print("Test samples:", len(y_test))
print("Number of classes:", len(arabic_characters))
print("Selected model: Residual CNN")
print("Selected seed: 789")

# ============================================================
# 3. LOAD FINAL SELECTED MODEL
# ============================================================

tf.keras.backend.clear_session()

final_model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

assert final_model.count_params() == 230780

print("\n=== FINAL MODEL VERIFICATION ===")
print("Model:", final_model.name)
print("Parameters:", final_model.count_params())

# ============================================================
# 4. GENERATE TEST PREDICTIONS
# ============================================================

test_probabilities = final_model.predict(
    X_test,
    batch_size=64,
    verbose=0
)

assert test_probabilities.shape == (3360, 28)

test_predictions = np.argmax(
    test_probabilities,
    axis=1
)

# ============================================================
# 5. COMPUTE FINAL TEST METRICS
# ============================================================

test_accuracy = accuracy_score(
    y_test,
    test_predictions
) * 100

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    labels=np.arange(28),
    average="macro",
    zero_division=0
) * 100

top3_predictions = np.argsort(
    test_probabilities,
    axis=1
)[:, -3:]

test_top3_accuracy = np.mean(
    np.any(
        top3_predictions == y_test[:, None],
        axis=1
    )
) * 100

test_loss = -np.mean(
    np.log(
        np.clip(
            test_probabilities[
                np.arange(len(y_test)),
                y_test
            ],
            1e-7,
            1.0
        )
    )
)

test_errors = int(
    np.sum(test_predictions != y_test)
)

metrics_df = pd.DataFrame([{
    "Model": "Residual CNN",
    "Seed": 789,
    "Test Samples": len(y_test),
    "Parameters": final_model.count_params(),
    "Test Accuracy (%)": test_accuracy,
    "Test Macro F1 (%)": test_macro_f1,
    "Test Top-3 Accuracy (%)": test_top3_accuracy,
    "Test Loss": test_loss,
    "Test Errors": test_errors
}])

print("\n=== FINAL TEST RESULTS ===")
print(metrics_df.round(4).to_string(index=False))

# ============================================================
# 6. COMPUTE PER-CLASS PERFORMANCE
# ============================================================

report = classification_report(
    y_test,
    test_predictions,
    labels=np.arange(28),
    output_dict=True,
    zero_division=0
)

class_records = []

for class_id in range(28):

    mask = y_test == class_id
    class_report = report[str(class_id)]

    class_records.append({
        "Class ID": class_id + 1,
        "Character": arabic_characters[class_id],
        "Samples": int(np.sum(mask)),
        "Accuracy (%)": np.mean(
            test_predictions[mask] == class_id
        ) * 100,
        "Precision": class_report["precision"],
        "Recall": class_report["recall"],
        "F1 Score": class_report["f1-score"]
    })

class_df = pd.DataFrame(class_records)

print("\n=== TOP 5 BEST RECOGNIZED CLASSES ===")

print(
    class_df.nlargest(
        5, "Accuracy (%)"
    ).round(4).to_string(index=False)
)

print("\n=== TOP 5 MOST DIFFICULT CLASSES ===")

print(
    class_df.nsmallest(
        5, "Accuracy (%)"
    ).round(4).to_string(index=False)
)

# ============================================================
# 7. COMPUTE CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=np.arange(28)
)

assert cm.sum() == 3360

fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(
    cm,
    interpolation="nearest",
    cmap="Purples"
)

fig.colorbar(im, ax=ax)

ax.set_title(
    "Final Test Confusion Matrix - Residual CNN",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Predicted Class ID")
ax.set_ylabel("True Class ID")

ax.set_xticks(np.arange(28))
ax.set_yticks(np.arange(28))

ax.set_xticklabels(np.arange(1, 29))
ax.set_yticklabels(np.arange(1, 29))

plt.setp(
    ax.get_xticklabels(),
    rotation=90
)

plt.tight_layout()

figure_path = (
    FIGURES_DIR
    / "final_test_confusion_matrix_seed789.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 8. IDENTIFY MOST FREQUENT CONFUSIONS
# ============================================================

confusion_records = []

for true_id in range(28):
    for pred_id in range(28):

        if true_id != pred_id and cm[true_id, pred_id] > 0:

            confusion_records.append({
                "True Class": true_id + 1,
                "True Character": arabic_characters[true_id],
                "Predicted Class": pred_id + 1,
                "Predicted Character": arabic_characters[pred_id],
                "Count": int(cm[true_id, pred_id])
            })

confusions_df = pd.DataFrame(confusion_records)

if not confusions_df.empty:

    confusions_df = confusions_df.sort_values(
        "Count",
        ascending=False
    )

    print("\n=== TOP 10 TEST CONFUSIONS ===")

    print(
        confusions_df.head(10).to_string(index=False)
    )

else:
    print("\nNo test classification errors found.")

# ============================================================
# 9. SAVE FINAL TEST RESULTS
# ============================================================

class_df.to_csv(
    TABLES_DIR / "final_test_per_class_seed789.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    cm,
    index=np.arange(1, 29),
    columns=np.arange(1, 29)
).to_csv(
    TABLES_DIR / "final_test_confusion_matrix_seed789.csv"
)

confusions_df.to_csv(
    TABLES_DIR / "final_test_confusions_seed789.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics_df.to_csv(
    METRICS_PATH,
    index=False
)

# ============================================================
# 10. FINAL VERIFICATION
# ============================================================

print("\n=== FINAL TEST VERIFICATION ===")
print("Model seed: 789")
print("Test images evaluated:", len(y_test))
print("Total correct:", len(y_test) - test_errors)
print("Total incorrect:", test_errors)
print("Metrics saved:", METRICS_PATH.exists())
print("Confusion matrix saved:", figure_path.exists())
print("New training performed: NO")

print("\nFinal test evaluation completed successfully!")


# Building Our Interactive Application

## Prepare the Final Model for Deployment

In [ ]:
# PREPARE FINAL MODEL FOR DEPLOYMENT


import tensorflow as tf
import numpy as np
import json
import shutil
import filecmp

from pathlib import Path

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

SOURCE_MODEL = (
    PROJECT_DIR
    / "advanced_models/residual_seed_789/residual_best.keras"
)

DATA_PATH = (
    PROJECT_DIR
    / "checkpoints/preprocessed_ahcd_v1.npz"
)

APP_DIR = PROJECT_DIR / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

DEPLOY_MODEL = APP_DIR / "arabic_residual_cnn.keras"
LABELS_PATH = APP_DIR / "class_labels.json"
CONFIG_PATH = APP_DIR / "model_config.json"

assert SOURCE_MODEL.exists()
assert DATA_PATH.exists()

# ============================================================
# 2. COPY FINAL MODEL SAFELY
# ============================================================

if DEPLOY_MODEL.exists():
    assert filecmp.cmp(
        SOURCE_MODEL,
        DEPLOY_MODEL,
        shallow=False
    ), "An existing deployment model is different!"

    print("Deployment model already verified.")

else:
    shutil.copy2(
        SOURCE_MODEL,
        DEPLOY_MODEL
    )

    print("Deployment model copied successfully.")

# ============================================================
# 3. DEFINE ARABIC CHARACTER LABELS
# ============================================================

arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

assert len(arabic_characters) == 28

labels = {
    str(index): character
    for index, character in enumerate(arabic_characters)
}

# ============================================================
# 4. SAVE LABELS AND MODEL CONFIGURATION
# ============================================================

config = {
    "model_name": "AHCD_Residual_CNN",
    "model_seed": 789,
    "dataset": "AHCD",
    "input_shape": [32, 32, 1],
    "num_classes": 28,
    "normalization": "float32 [0, 1]",
    "foreground": "light strokes on dark background",
    "label_encoding": "zero-based indices 0-27",
    "selection_criterion": "validation performance",
    "camera_required": False
}

def save_json_safely(path, content):
    if path.exists():
        with open(path, "r", encoding="utf-8") as file:
            existing = json.load(file)

        assert existing == content, (
            f"Existing configuration differs: {path}"
        )
    else:
        with open(path, "w", encoding="utf-8") as file:
            json.dump(
                content,
                file,
                ensure_ascii=False,
                indent=4
            )

save_json_safely(LABELS_PATH, labels)
save_json_safely(CONFIG_PATH, config)

# ============================================================
# 5. VERIFY THE EXPORTED MODEL
# ============================================================

with np.load(DATA_PATH, allow_pickle=False) as data:
    X_val_sample = data["X_val"][:16]

original_model = tf.keras.models.load_model(
    SOURCE_MODEL,
    compile=False
)

deployed_model = tf.keras.models.load_model(
    DEPLOY_MODEL,
    compile=False
)

original_predictions = original_model.predict(
    X_val_sample,
    verbose=0
)

deployed_predictions = deployed_model.predict(
    X_val_sample,
    verbose=0
)

np.testing.assert_allclose(
    original_predictions,
    deployed_predictions,
    rtol=1e-5,
    atol=1e-6
)

assert deployed_model.count_params() == 230780

# ============================================================
# 6. FINAL DEPLOYMENT VERIFICATION
# ============================================================

print("\n=== DEPLOYMENT PACKAGE VERIFICATION ===")
print("Model file:", DEPLOY_MODEL.exists())
print("Class labels:", LABELS_PATH.exists())
print("Model configuration:", CONFIG_PATH.exists())
print("Number of classes:", len(labels))
print("Model parameters:", deployed_model.count_params())
print("Prediction consistency: VERIFIED")
print("New training performed: NO")
print("Test set used: NO")

print("\nDeployment package prepared successfully!")


# Image Preprocessing for the Application

In [ ]:
# IMAGE PREPROCESSING FOR DEPLOYMENT


import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import sys

from pathlib import Path
from PIL import Image, ImageOps

# ============================================================
# 1. DEFINE PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN"
)

APP_DIR = PROJECT_DIR / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESS_PATH = APP_DIR / "preprocessing.py"

# ============================================================
# 2. CREATE REUSABLE PREPROCESSING MODULE
# ============================================================

module_code = '''
import numpy as np
from PIL import Image


def preprocess_handwritten_image(image):
    """
    Convert a handwritten character image to CNN input.

    Returns:
        float32 array with shape (32, 32, 1).
    """

    if image is None:
        raise ValueError("No image was provided.")

    # Composite transparent images over white.
    rgba = image.convert("RGBA")
    white_background = Image.new(
        "RGBA", rgba.size, (255, 255, 255, 255)
    )
    grayscale = Image.alpha_composite(
        white_background, rgba
    ).convert("L")

    pixels = np.asarray(
        grayscale, dtype=np.float32
    )

    # Estimate background intensity from the border.
    border = np.concatenate([
        pixels[0, :],
        pixels[-1, :],
        pixels[:, 0],
        pixels[:, -1]
    ])

    background = float(np.median(border))

    # Convert both polarities to light strokes on dark.
    if background >= 128:
        foreground = np.clip(
            background - pixels, 0, 255
        )
    else:
        foreground = np.clip(
            pixels - background, 0, 255
        )

    # Detect the handwritten character region.
    mask = foreground > 20

    if not np.any(mask):
        raise ValueError(
            "No visible handwritten character detected."
        )

    ys, xs = np.where(mask)

    x_min, x_max = xs.min(), xs.max() + 1
    y_min, y_max = ys.min(), ys.max() + 1

    cropped = foreground[
        y_min:y_max,
        x_min:x_max
    ]

    # Preserve the character aspect ratio.
    crop_height, crop_width = cropped.shape

    scale = 24 / max(crop_width, crop_height)

    new_width = max(
        1, int(round(crop_width * scale))
    )
    new_height = max(
        1, int(round(crop_height * scale))
    )

    cropped_image = Image.fromarray(
        cropped.astype(np.uint8)
    )

    resized = cropped_image.resize(
        (new_width, new_height),
        resample=Image.Resampling.LANCZOS
    )

    # Center the character on a 32x32 black canvas.
    canvas = Image.new("L", (32, 32), 0)

    left = (32 - new_width) // 2
    top = (32 - new_height) // 2

    canvas.paste(resized, (left, top))

    # Normalize exactly to the expected numerical range.
    processed = np.asarray(
        canvas, dtype=np.float32
    ) / 255.0

    return processed[..., np.newaxis]
'''

if PREPROCESS_PATH.exists():
    existing_code = PREPROCESS_PATH.read_text(
        encoding="utf-8"
    )
    assert existing_code == module_code, (
        "Existing preprocessing module differs. "
        "Review it before overwriting."
    )
else:
    PREPROCESS_PATH.write_text(
        module_code, encoding="utf-8"
    )

sys.path.insert(0, str(APP_DIR))

from preprocessing import preprocess_handwritten_image

print("=== PREPROCESSING MODULE VERIFICATION ===")
print("Module saved:", PREPROCESS_PATH.exists())

# ============================================================
# 3. RESTORE ONE VALIDATION EXAMPLE
# ============================================================

DATA_PATH = (
    PROJECT_DIR
    / "checkpoints/preprocessed_ahcd_v1.npz"
)

with np.load(DATA_PATH, allow_pickle=False) as data:
    original_image = data["X_val"][0]
    original_label = int(data["y_val"].ravel()[0])

# ============================================================
# 4. SIMULATE AN UPLOADED HANDWRITTEN IMAGE
# ============================================================

# Convert the dataset image into an enlarged PNG-like image.
source = Image.fromarray(
    (original_image[:, :, 0] * 255).astype(np.uint8)
)

# Simulate dark handwriting on a white background.
large_image = Image.new("L", (256, 256), 255)

large_strokes = ImageOps.invert(source).resize(
    (160, 160),
    resample=Image.Resampling.NEAREST
)

large_image.paste(large_strokes, (48, 48))

# Simulate the opposite polarity as well.
dark_background_image = ImageOps.invert(
    large_image
)

processed_white = preprocess_handwritten_image(
    large_image
)

processed_dark = preprocess_handwritten_image(
    dark_background_image
)

# ============================================================
# 5. VERIFY IMAGE PREPROCESSING
# ============================================================

assert processed_white.shape == (32, 32, 1)
assert processed_white.dtype == np.float32

assert processed_white.min() >= 0
assert processed_white.max() <= 1

np.testing.assert_allclose(
    processed_white,
    processed_dark,
    atol=1 / 255
)

print("\n=== IMAGE PREPROCESSING VERIFICATION ===")
print("Output shape:", processed_white.shape)
print("Output dtype:", processed_white.dtype)
print("Minimum pixel value:", processed_white.min())
print("Maximum pixel value:", processed_white.max())
print("Both input polarities: VERIFIED")

# ============================================================
# 6. COMPARE ORIGINAL AND PROCESSED IMAGES
# ============================================================

fig, axes = plt.subplots(
    1, 3, figsize=(11, 4)
)

axes[0].imshow(
    large_image,
    cmap="gray",
    vmin=0,
    vmax=255
)
axes[0].set_title("Simulated Upload")

axes[1].imshow(
    original_image[:, :, 0],
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[1].set_title("Original Validation Image")

axes[2].imshow(
    processed_white[:, :, 0],
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[2].set_title("Application Preprocessing")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    "Image Preprocessing for CNN Deployment",
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# ============================================================
# 7. CHECK MODEL PREDICTIONS
# ============================================================

MODEL_PATH = (
    APP_DIR / "arabic_residual_cnn.keras"
)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

original_probs = model.predict(
    original_image[np.newaxis, ...],
    verbose=0
)[0]

processed_probs = model.predict(
    processed_white[np.newaxis, ...],
    verbose=0
)[0]

original_prediction = int(
    np.argmax(original_probs)
)

processed_prediction = int(
    np.argmax(processed_probs)
)

print("\n=== PREDICTION CONSISTENCY CHECK ===")
print("True class ID:", original_label + 1)
print("Original predicted class ID:", original_prediction + 1)
print("Processed predicted class ID:", processed_prediction + 1)
print("Same prediction:",
      original_prediction == processed_prediction)

print("\nPreprocessing test completed!")


In [ ]:
# ============================================================
# CHECK GOOGLE DRIVE CONNECTION
# ============================================================

from pathlib import Path

drive_path = Path("/content/drive/MyDrive")

print("=== GOOGLE DRIVE STATUS ===")
print("Drive accessible:", drive_path.exists())

if drive_path.exists():
    print("Drive folders:")
    for folder in list(drive_path.iterdir())[:10]:
        print("-", folder.name)
else:
    print("Google Drive is not currently accessible.")

In [ ]:
# ============================================================
# RESTORE GOOGLE DRIVE CONNECTION
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True,
    timeout_ms=300000
)

# Create the Interactive Gradio Application

## Install Gradio

In [ ]:
%pip -q install "gradio>=5,<7"

In [ ]:
# ============================================================
#  RESTORE APP FILES
# ============================================================

import json
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Arabic_Handwritten_CNN")
APP_DIR = PROJECT_DIR / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_MODEL = PROJECT_DIR / "advanced_models/residual_seed_789/residual_best.keras"
DEPLOY_MODEL = APP_DIR / "arabic_residual_cnn.keras"
LABELS_PATH = APP_DIR / "class_labels.json"
CONFIG_PATH = APP_DIR / "model_config.json"
PREPROCESS_PATH = APP_DIR / "preprocessing.py"

assert SOURCE_MODEL.exists(), f"Missing source model: {SOURCE_MODEL}"

# ------------------------------------------------------------
# 1. Restore deployment model
# ------------------------------------------------------------
if not DEPLOY_MODEL.exists():
    shutil.copy2(SOURCE_MODEL, DEPLOY_MODEL)
    model_status = "RESTORED"
else:
    model_status = "ALREADY PRESENT"

# ------------------------------------------------------------
# 2. Restore class labels
# ------------------------------------------------------------
arabic_characters = [
    "ا", "ب", "ت", "ث", "ج", "ح", "خ",
    "د", "ذ", "ر", "ز", "س", "ش", "ص",
    "ض", "ط", "ظ", "ع", "غ", "ف", "ق",
    "ك", "ل", "م", "ن", "ه", "و", "ي"
]

labels = {str(i): ch for i, ch in enumerate(arabic_characters)}

if not LABELS_PATH.exists():
    with open(LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(labels, f, ensure_ascii=False, indent=4)
    labels_status = "RESTORED"
else:
    labels_status = "ALREADY PRESENT"

# ------------------------------------------------------------
# 3. Restore config
# ------------------------------------------------------------
config = {
    "model_name": "AHCD_Residual_CNN",
    "model_seed": 789,
    "dataset": "AHCD",
    "input_shape": [32, 32, 1],
    "num_classes": 28,
    "normalization": "float32 [0, 1]",
    "foreground": "light strokes on dark background",
    "label_encoding": "zero-based indices 0-27",
    "selection_criterion": "validation performance",
    "camera_required": False
}

if not CONFIG_PATH.exists():
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=4)
    config_status = "RESTORED"
else:
    config_status = "ALREADY PRESENT"

# ------------------------------------------------------------
# 4. Restore preprocessing module
# ------------------------------------------------------------
module_code = '''
import numpy as np
from PIL import Image

def preprocess_handwritten_image(image):
    """
    Convert a handwritten character image to CNN input.
    Returns a float32 array with shape (32, 32, 1).
    """

    if image is None:
        raise ValueError("No image was provided.")

    rgba = image.convert("RGBA")
    white_background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
    grayscale = Image.alpha_composite(white_background, rgba).convert("L")

    pixels = np.asarray(grayscale, dtype=np.float32)

    border = np.concatenate([
        pixels[0, :],
        pixels[-1, :],
        pixels[:, 0],
        pixels[:, -1]
    ])

    background = float(np.median(border))

    if background >= 128:
        foreground = np.clip(background - pixels, 0, 255)
    else:
        foreground = np.clip(pixels - background, 0, 255)

    mask = foreground > 20

    if not np.any(mask):
        raise ValueError("No visible handwritten character detected.")

    ys, xs = np.where(mask)

    x_min, x_max = xs.min(), xs.max() + 1
    y_min, y_max = ys.min(), ys.max() + 1

    cropped = foreground[y_min:y_max, x_min:x_max]

    crop_height, crop_width = cropped.shape
    scale = 24 / max(crop_width, crop_height)

    new_width = max(1, int(round(crop_width * scale)))
    new_height = max(1, int(round(crop_height * scale)))

    cropped_image = Image.fromarray(cropped.astype(np.uint8))
    resized = cropped_image.resize((new_width, new_height), resample=Image.Resampling.LANCZOS)

    canvas = Image.new("L", (32, 32), 0)

    left = (32 - new_width) // 2
    top = (32 - new_height) // 2

    canvas.paste(resized, (left, top))

    processed = np.asarray(canvas, dtype=np.float32) / 255.0
    return processed[..., np.newaxis]
'''

if not PREPROCESS_PATH.exists():
    PREPROCESS_PATH.write_text(module_code, encoding="utf-8")
    preprocess_status = "RESTORED"
else:
    preprocess_status = "ALREADY PRESENT"

print("=== APP FILE RECOVERY ===")
print("Deployment model:", model_status)
print("Class labels:", labels_status)
print("Model config:", config_status)
print("Preprocessing module:", preprocess_status)

print("\n=== FINAL CHECK ===")
print("arabic_residual_cnn.keras:", DEPLOY_MODEL.exists())
print("class_labels.json:", LABELS_PATH.exists())
print("model_config.json:", CONFIG_PATH.exists())
print("preprocessing.py:", PREPROCESS_PATH.exists())

## Build and Launch the Application

In [ ]:
# PALESTINE-INSPIRED ARABIC RECOGNITION APP


import gradio as gr
import tensorflow as tf
import numpy as np
import json
import sys
import html

from pathlib import Path
from PIL import Image

# ============================================================
# 1. LOAD THE DEPLOYMENT MODEL
# ============================================================

APP_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN/app"
)

MODEL_PATH = APP_DIR / "arabic_residual_cnn.keras"
LABELS_PATH = APP_DIR / "class_labels.json"
PREPROCESS_PATH = APP_DIR / "preprocessing.py"

for path in [MODEL_PATH, LABELS_PATH, PREPROCESS_PATH]:
    assert path.exists(), f"Missing file: {path}"

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

from preprocessing import preprocess_handwritten_image

with open(LABELS_PATH, "r", encoding="utf-8") as file:
    class_labels = json.load(file)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

assert model.count_params() == 230780
assert len(class_labels) == 28

print("=== MODEL VERIFICATION ===")
print("Model loaded: YES")
print("Gradio version:", gr.__version__)
print("Model parameters:", model.count_params())
print("Arabic characters:", len(class_labels))

# ============================================================
# 2. PALESTINE-INSPIRED VISUAL THEME
# ============================================================

custom_css = """
.gradio-container {
    max-width: 1050px !important;
    margin: auto !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
}

/* Minimal Palestinian flag ribbon */
.palestine-ribbon {
    position: relative;
    width: 100%;
    height: 17px;
    border-radius: 12px;
    overflow: hidden;
    background: linear-gradient(
        to bottom,
        #171717 0%,
        #171717 33.33%,
        #ffffff 33.33%,
        #ffffff 66.66%,
        #007a3d 66.66%,
        #007a3d 100%
    );
    box-shadow: 0 3px 12px rgba(0,0,0,0.10);
    margin-bottom: 25px;
}

.palestine-ribbon::before {
    content: "";
    position: absolute;
    left: 0;
    top: 0;
    width: 64px;
    height: 100%;
    background: #ce1126;
    clip-path: polygon(0 0, 100% 50%, 0 100%);
}

/* Hero section */
.hero {
    text-align: center;
    padding: 16px 18px 26px;
}

.hero-icon {
    font-size: 42px;
    margin-bottom: 8px;
}

.hero-title {
    font-size: 33px;
    font-weight: 800;
    line-height: 1.3;
    color: #153b31;
    margin: 0 0 10px;
}

.hero-subtitle {
    font-size: 16px;
    color: #64746f;
    line-height: 1.7;
    margin: 0;
}

/* Instruction area */
.guide-box {
    text-align: center;
    background: #f2f8f4;
    color: #264c3e;
    padding: 14px 18px;
    border: 1px solid #d7e8dc;
    border-radius: 14px;
    margin-bottom: 12px;
    line-height: 1.8;
}

/* Section heading */
.section-heading {
    font-size: 21px;
    font-weight: 800;
    color: #174e3d;
    margin: 20px 0 12px;
}

/* Prediction card */
.prediction-card {
    background: linear-gradient(
        135deg,
        #ffffff 0%,
        #f3f9f5 100%
    );
    border: 1px solid #d8e8de;
    border-radius: 20px;
    padding: 24px;
    text-align: center;
    box-shadow: 0 5px 20px rgba(15,60,40,0.06);
}

.prediction-heading {
    font-size: 15px;
    color: #687b72;
    font-weight: 600;
    letter-spacing: 1px;
    text-transform: uppercase;
}

.predicted-letter {
    font-family: 'Amiri', 'Noto Naskh Arabic',
                 'Traditional Arabic', serif;
    font-size: 94px;
    font-weight: 700;
    color: #087849;
    line-height: 1.5;
    direction: rtl;
}

.confidence-text {
    color: #33483e;
    font-size: 16px;
}

.confidence-note {
    color: #829187;
    font-size: 12px;
    margin-top: 9px;
}

/* Arabic cultural message */
.heritage-card {
    position: relative;
    margin: 22px 0 12px;
    padding: 30px 24px;
    border-radius: 22px;
    background: linear-gradient(
        135deg,
        #fdfaf7 0%,
        #f2f8f4 100%
    );
    border: 1px solid #e7e2d9;
    text-align: center;
    box-shadow: 0 8px 26px rgba(30,45,35,0.07);
    animation: gentle-appear 0.55s ease-out;
}

.heritage-card::before {
    content: "";
    position: absolute;
    top: 0;
    left: 12%;
    width: 76%;
    height: 4px;
    border-radius: 0 0 8px 8px;
    background: linear-gradient(
        to right,
        #ce1126,
        #f0d8d8,
        #007a3d
    );
}

.heritage-symbol {
    font-size: 24px;
    margin-bottom: 10px;
}

.heritage-quote {
    font-family: 'Amiri', 'Noto Naskh Arabic',
                 'Traditional Arabic', serif;
    font-size: 29px;
    font-weight: 700;
    color: #214e3e;
    line-height: 2.1;
    direction: rtl;
}

.heritage-caption {
    color: #7b867e;
    font-size: 12px;
    margin-top: 14px;
    letter-spacing: 0.5px;
}

@keyframes gentle-appear {
    from {
        opacity: 0;
        transform: translateY(10px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}

/* Footer */
.app-footer {
    text-align: center;
    color: #7d8b84;
    font-size: 12px;
    line-height: 1.8;
    padding: 18px 8px 8px;
}

/* Buttons */
button.pal-action {
    background: #087849 !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 700 !important;
    min-height: 46px !important;
}

button.pal-action:hover {
    background: #065d39 !important;
}

/* Responsive layout */
@media (max-width: 600px) {
    .hero-title {
        font-size: 25px;
    }

    .heritage-quote {
        font-size: 23px;
    }

    .predicted-letter {
        font-size: 76px;
    }

    .heritage-card {
        padding: 24px 15px;
    }
}
"""

# ============================================================
# 3. RESULT DISPLAY FUNCTIONS
# ============================================================

def make_prediction_card(letter, confidence):

    safe_letter = html.escape(letter)

    return f"""
    <div class="prediction-card">

        <div class="prediction-heading">
            Recognized Arabic Character
        </div>

        <div class="predicted-letter">
            {safe_letter}
        </div>

        <div class="confidence-text">
            Model confidence score:
            <strong>{confidence:.2%}</strong>
        </div>

        <div class="confidence-note">
            Confidence is not a guarantee of correctness.
        </div>

    </div>
    """


def make_heritage_message():

    return """
    <div class="heritage-card">

        <div class="heritage-symbol">
            ✨ 🇵🇸 ✨
        </div>

        <div class="heritage-quote" lang="ar" dir="rtl">
            اللغةُ العربيةُ هُوِيَّةٌ وبيان،
            وفلسطينُ حُرَّةٌ في القلبِ والوجدان
        </div>

        <div class="heritage-caption">
            A LETTER • A LANGUAGE • A HERITAGE
        </div>

    </div>
    """


# ============================================================
# 4. RECOGNITION FUNCTION
# ============================================================

def recognize_character(image):

    if image is None:
        return (
            "<div class='prediction-card'>"
            "Please draw or upload a character first."
            "</div>",
            None,
            {},
            ""
        )

    try:

        # Apply the same deployment preprocessing
        processed = preprocess_handwritten_image(image)

        input_batch = np.expand_dims(
            processed,
            axis=0
        )

        # Run Residual CNN inference
        probabilities = model(
            input_batch,
            training=False
        ).numpy()[0]

        top_indices = np.argsort(
            probabilities
        )[-3:][::-1]

        predicted_id = int(top_indices[0])

        predicted_letter = class_labels[
            str(predicted_id)
        ]

        confidence = float(
            probabilities[predicted_id]
        )

        # Create the main result
        result_html = make_prediction_card(
            predicted_letter,
            confidence
        )

        # Create top-3 predictions
        top3 = {
            class_labels[str(int(index))]:
                float(probabilities[index])
            for index in top_indices
        }

        # Show the actual 32x32 CNN input
        preview = np.clip(
            processed[:, :, 0] * 255,
            0,
            255
        ).astype(np.uint8)

        # Display Arabic heritage message
        heritage_html = make_heritage_message()

        return (
            result_html,
            preview,
            top3,
            heritage_html
        )

    except Exception as exc:

        safe_error = html.escape(str(exc))

        return (
            f"<div class='prediction-card'>"
            f"Unable to process image: {safe_error}"
            f"</div>",
            None,
            {},
            ""
        )


def recognize_drawing(editor_value):

    if editor_value is None:
        return recognize_character(None)

    composite = editor_value.get("composite")

    return recognize_character(composite)


# ============================================================
# 5. CLOSE PREVIOUS GRADIO INSTANCE IF PRESENT
# ============================================================

previous_demo = globals().get("demo")

if previous_demo is not None:
    try:
        previous_demo.close()
    except Exception:
        pass

# ============================================================
# 6. BUILD THE NEW INTERFACE
# ============================================================

with gr.Blocks(
    css=custom_css,
    title="Arabic Handwritten Character Recognition",
    theme=gr.themes.Soft(
        primary_hue="green",
        neutral_hue="slate"
    )
) as demo:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.HTML("""
    <div class="palestine-ribbon"></div>

    <div class="hero">

        <div class="hero-icon">✍️</div>

        <h1 class="hero-title">
            Arabic Handwritten Character Recognition
        </h1>

        <p class="hero-subtitle">
            Discover the beauty of Arabic handwriting
            through Artificial Intelligence.
            <br>
            Powered by a Residual Convolutional Neural Network.
        </p>

    </div>
    """)

    gr.HTML("""
    <div class="guide-box">
        Draw one Arabic letter with a thin brush
        or upload a handwritten image.
        <br>
        Explore how AI recognizes the 28 Arabic characters.
    </div>
    """)

    # --------------------------------------------------------
    # DRAWING AND UPLOAD TABS
    # --------------------------------------------------------

    with gr.Tabs():

        with gr.Tab("✍️ Draw a Letter"):

            drawing_input = gr.ImageEditor(
                value=Image.new(
                    "RGB",
                    (256, 256),
                    "white"
                ),
                type="pil",
                image_mode="RGB",
                sources=[],
                canvas_size=(256, 256),
                fixed_canvas=True,
                brush=gr.Brush(
                    colors=["#000000"],
                    default_color="#000000",
                    color_mode="fixed",
                    default_size=6
                ),
                layers=False,
                label="Draw One Arabic Character"
            )

            draw_button = gr.Button(
                "Recognize Drawing",
                variant="primary",
                elem_classes=["pal-action"]
            )

        with gr.Tab("📁 Upload an Image"):

            upload_input = gr.Image(
                type="pil",
                sources=["upload", "clipboard"],
                label="Upload an Arabic Handwritten Character",
                height=280
            )

            gr.Markdown(
                "For better results, use a clear, "
                "centered handwritten character "
                "on a simple background."
            )

            upload_button = gr.Button(
                "Recognize Uploaded Image",
                variant="primary",
                elem_classes=["pal-action"]
            )

    # --------------------------------------------------------
    # PREDICTION RESULTS
    # --------------------------------------------------------

    gr.HTML("""
    <div class="section-heading">
        ✨ Recognition Results
    </div>
    """)

    prediction_html = gr.HTML(
        "<div class='prediction-card'>"
        "Your recognized character will appear here."
        "</div>"
    )

    with gr.Row():

        processed_preview = gr.Image(
            label="CNN Input — 32 × 32",
            interactive=False,
            image_mode="L",
            height=190
        )

        top3_output = gr.Label(
            label="Top 3 Predictions",
            num_top_classes=3
        )

    # --------------------------------------------------------
    # ARABIC HERITAGE MESSAGE
    # --------------------------------------------------------

    heritage_message = gr.HTML(
        value=""
    )

    # --------------------------------------------------------
    # EVENT HANDLERS
    # --------------------------------------------------------

    draw_button.click(
        fn=recognize_drawing,
        inputs=drawing_input,
        outputs=[
            prediction_html,
            processed_preview,
            top3_output,
            heritage_message
        ]
    )

    upload_button.click(
        fn=recognize_character,
        inputs=upload_input,
        outputs=[
            prediction_html,
            processed_preview,
            top3_output,
            heritage_message
        ]
    )

    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.HTML("""
    <div class="app-footer">

        Residual CNN · AHCD Dataset · 28 Arabic Characters

        <br>

        Educational AI Prototype

        <br>

        Recognition accuracy on user-uploaded images
        may differ from the AHCD test results.

    </div>
    """)

# ============================================================
# 7. LAUNCH THE APPLICATION
# ============================================================

print("\n=== APPLICATION VERIFICATION ===")
print("Residual CNN loaded: YES")
print("Drawing input: ENABLED")
print("Image upload: ENABLED")
print("Thin brush (6 px): ENABLED")
print("Palestinian flag ribbon: ENABLED")
print("Arabic heritage quote: ENABLED")
print("Webcam: DISABLED")
print("New training performed: NO")

print("\nLaunching refined application...")

demo.launch(
    share=True,
    inline=False,
    show_error=True
)


In [ ]:

# ============================================================
#  CREATE STANDALONE GRADIO APPLICATION
# ============================================================

from pathlib import Path

# ============================================================
# 1. DEFINE DEPLOYMENT DIRECTORY
# ============================================================

APP_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN/app"
)

APP_PATH = APP_DIR / "app.py"

required_files = [
    "arabic_residual_cnn.keras",
    "class_labels.json",
    "preprocessing.py"
]

for filename in required_files:
    assert (APP_DIR / filename).exists(), (
        f"Missing deployment file: {filename}"
    )

# ============================================================
# 2. DEFINE STANDALONE APPLICATION SOURCE
# ============================================================

app_source = r'''
# ============================================================
# ARABIC HANDWRITTEN CHARACTER RECOGNITION
# PALESTINE-INSPIRED GRADIO APPLICATION
# ============================================================

import gradio as gr
import tensorflow as tf
import numpy as np
import json
import html
import sys

from pathlib import Path
from PIL import Image

# ============================================================
# 1. LOAD MODEL AND PREPROCESSING MODULE
# ============================================================

APP_DIR = Path(__file__).resolve().parent

sys.path.insert(0, str(APP_DIR))

from preprocessing import preprocess_handwritten_image

MODEL_PATH = APP_DIR / "arabic_residual_cnn.keras"
LABELS_PATH = APP_DIR / "class_labels.json"

with open(LABELS_PATH, "r", encoding="utf-8") as f:
    class_labels = json.load(f)

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

assert len(class_labels) == 28
assert model.count_params() == 230780

# ============================================================
# 2. PALESTINE-INSPIRED DESIGN
# ============================================================

CUSTOM_CSS = """
.gradio-container {
    max-width: 1050px !important;
    margin: auto !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
}

.palestine-ribbon {
    position: relative;
    width: 100%;
    height: 17px;
    border-radius: 12px;
    overflow: hidden;
    background: linear-gradient(
        to bottom,
        #171717 0%,
        #171717 33.33%,
        #ffffff 33.33%,
        #ffffff 66.66%,
        #007a3d 66.66%,
        #007a3d 100%
    );
    margin-bottom: 25px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.10);
}

.palestine-ribbon::before {
    content: "";
    position: absolute;
    left: 0;
    top: 0;
    width: 64px;
    height: 100%;
    background: #ce1126;
    clip-path: polygon(0 0, 100% 50%, 0 100%);
}

.hero {
    text-align: center;
    padding: 16px 18px 26px;
}

.hero-title {
    color: #153b31;
    font-size: 33px;
    font-weight: 800;
    line-height: 1.3;
    margin: 0 0 10px;
}

.hero-subtitle {
    color: #64746f;
    font-size: 16px;
    line-height: 1.7;
}

.guide-box {
    text-align: center;
    background: #f2f8f4;
    color: #264c3e;
    padding: 14px;
    border: 1px solid #d7e8dc;
    border-radius: 14px;
    margin-bottom: 12px;
}

.section-heading {
    color: #174e3d;
    font-size: 21px;
    font-weight: 800;
    margin: 20px 0 12px;
}

.prediction-card {
    background: linear-gradient(
        135deg, #ffffff, #f3f9f5
    );
    border: 1px solid #d8e8de;
    border-radius: 20px;
    padding: 24px;
    text-align: center;
    box-shadow: 0 5px 20px rgba(15,60,40,0.06);
}

.predicted-letter {
    font-family: 'Amiri', 'Noto Naskh Arabic',
                 'Traditional Arabic', serif;
    font-size: 94px;
    font-weight: 700;
    color: #087849;
    line-height: 1.5;
    direction: rtl;
}

.heritage-card {
    margin: 22px 0 12px;
    padding: 30px 24px;
    border-radius: 22px;
    background: linear-gradient(
        135deg, #fdfaf7, #f2f8f4
    );
    border: 1px solid #e7e2d9;
    text-align: center;
    box-shadow: 0 8px 26px rgba(30,45,35,0.07);
    animation: gentle-appear 0.55s ease-out;
}

.heritage-quote {
    font-family: 'Amiri', 'Noto Naskh Arabic',
                 'Traditional Arabic', serif;
    font-size: 29px;
    font-weight: 700;
    color: #214e3e;
    line-height: 2.1;
    direction: rtl;
}

.heritage-caption {
    color: #7b867e;
    font-size: 12px;
    margin-top: 14px;
    letter-spacing: 0.5px;
}

@keyframes gentle-appear {
    from {
        opacity: 0;
        transform: translateY(10px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}

button.pal-action {
    background: #087849 !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 700 !important;
    min-height: 46px !important;
}

button.pal-action:hover {
    background: #065d39 !important;
}

.app-footer {
    text-align: center;
    color: #7d8b84;
    font-size: 12px;
    line-height: 1.8;
    padding: 18px 8px;
}

@media (max-width: 600px) {
    .hero-title {
        font-size: 25px;
    }

    .heritage-quote {
        font-size: 23px;
    }

    .predicted-letter {
        font-size: 76px;
    }
}
"""

# ============================================================
# 3. BUILD PREDICTION DISPLAY
# ============================================================

def make_prediction_card(letter, confidence):

    safe_letter = html.escape(letter)

    return f"""
    <div class="prediction-card">

        <div style="color:#687b72;font-size:15px;">
            Recognized Arabic Character
        </div>

        <div class="predicted-letter">
            {safe_letter}
        </div>

        <div style="color:#33483e;font-size:16px;">
            Model confidence score:
            <strong>{confidence:.2%}</strong>
        </div>

        <p style="color:#829187;font-size:12px;">
            Confidence is not a guarantee of correctness.
        </p>

    </div>
    """


def make_heritage_message():

    return """
    <div class="heritage-card">

        <div style="font-size:24px;margin-bottom:10px;">
            ✨ 🇵🇸 ✨
        </div>

        <div class="heritage-quote" lang="ar" dir="rtl">
            اللغةُ العربيةُ هُوِيَّةٌ وبيان،
            وفلسطينُ حُرَّةٌ في القلبِ والوجدان
        </div>

        <div class="heritage-caption">
            A LETTER • A LANGUAGE • A HERITAGE
        </div>

    </div>
    """

# ============================================================
# 4. CHARACTER RECOGNITION
# ============================================================

def recognize_character(image):

    if image is None:
        return (
            "<div class='prediction-card'>"
            "Please draw or upload a character first."
            "</div>",
            None,
            {},
            ""
        )

    try:
        # Apply the deployment preprocessing pipeline
        processed = preprocess_handwritten_image(image)

        input_batch = np.expand_dims(
            processed,
            axis=0
        )

        # Predict using the trained Residual CNN
        probabilities = model(
            input_batch,
            training=False
        ).numpy()[0]

        top_indices = np.argsort(
            probabilities
        )[-3:][::-1]

        predicted_id = int(top_indices[0])

        predicted_letter = class_labels[
            str(predicted_id)
        ]

        confidence = float(
            probabilities[predicted_id]
        )

        # Generate display elements
        prediction_html = make_prediction_card(
            predicted_letter,
            confidence
        )

        top3 = {
            class_labels[str(int(idx))]:
                float(probabilities[idx])
            for idx in top_indices
        }

        preview = np.clip(
            processed[:, :, 0] * 255,
            0,
            255
        ).astype(np.uint8)

        heritage_html = make_heritage_message()

        return (
            prediction_html,
            preview,
            top3,
            heritage_html
        )

    except Exception as exc:

        return (
            "<div class='prediction-card'>"
            "Unable to process image: "
            + html.escape(str(exc))
            + "</div>",
            None,
            {},
            ""
        )


def recognize_drawing(editor_value):

    if editor_value is None:
        return recognize_character(None)

    composite = editor_value.get("composite")

    return recognize_character(composite)

# ============================================================
# 5. BUILD GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    css=CUSTOM_CSS,
    title="Arabic Handwritten Character Recognition",
    theme=gr.themes.Soft(
        primary_hue="green",
        neutral_hue="slate"
    )
) as demo:

    # Header
    gr.HTML("""
    <div class="palestine-ribbon"></div>

    <div class="hero">

        <div style="font-size:42px;">✍️</div>

        <h1 class="hero-title">
            Arabic Handwritten Character Recognition
        </h1>

        <p class="hero-subtitle">
            Discover the beauty of Arabic handwriting
            through Artificial Intelligence.
            <br>
            Powered by a Residual Convolutional
            Neural Network.
        </p>

    </div>
    """)

    # Instructions
    gr.HTML("""
    <div class="guide-box">
        Draw one Arabic letter with a thin brush
        or upload a handwritten image.
        <br>
        Explore how AI recognizes
        the 28 Arabic characters.
    </div>
    """)

    # Inputs
    with gr.Tabs():

        with gr.Tab("✍️ Draw a Letter"):

            drawing_input = gr.ImageEditor(
                value=Image.new(
                    "RGB", (256, 256), "white"
                ),
                type="pil",
                image_mode="RGB",
                sources=[],
                canvas_size=(256, 256),
                fixed_canvas=True,
                brush=gr.Brush(
                    colors=["#000000"],
                    default_color="#000000",
                    color_mode="fixed",
                    default_size=6
                ),
                layers=False,
                label="Draw One Arabic Character"
            )

            draw_button = gr.Button(
                "Recognize Drawing",
                variant="primary",
                elem_classes=["pal-action"]
            )

        with gr.Tab("📁 Upload an Image"):

            upload_input = gr.Image(
                type="pil",
                sources=["upload", "clipboard"],
                label="Upload an Arabic Handwritten Character",
                height=280
            )

            gr.Markdown(
                "Use a clear, centered handwritten "
                "character on a simple background."
            )

            upload_button = gr.Button(
                "Recognize Uploaded Image",
                variant="primary",
                elem_classes=["pal-action"]
            )

    # Results
    gr.HTML("""
    <div class="section-heading">
        ✨ Recognition Results
    </div>
    """)

    prediction_html = gr.HTML(
        "<div class='prediction-card'>"
        "Your recognized character will appear here."
        "</div>"
    )

    with gr.Row():

        processed_preview = gr.Image(
            label="CNN Input — 32 × 32",
            interactive=False,
            image_mode="L",
            height=190
        )

        top3_output = gr.Label(
            label="Top 3 Predictions",
            num_top_classes=3
        )

    heritage_message = gr.HTML(value="")

    # Connect prediction events
    draw_button.click(
        fn=recognize_drawing,
        inputs=drawing_input,
        outputs=[
            prediction_html,
            processed_preview,
            top3_output,
            heritage_message
        ]
    )

    upload_button.click(
        fn=recognize_character,
        inputs=upload_input,
        outputs=[
            prediction_html,
            processed_preview,
            top3_output,
            heritage_message
        ]
    )

    # Footer
    gr.HTML("""
    <div class="app-footer">
        Residual CNN · AHCD Dataset ·
        28 Arabic Characters

        <br>
        Educational AI Prototype

        <br>
        Recognition performance on user drawings
        may differ from the AHCD test results.
    </div>
    """)

# ============================================================
# 6. LAUNCH IN STANDALONE ENVIRONMENTS
# ============================================================

if __name__ == "__main__":
    demo.launch()
'''

# ============================================================
# 3. VALIDATE SOURCE CODE
# ============================================================

compile(
    app_source,
    str(APP_PATH),
    "exec"
)

# ============================================================
# 4. SAVE APPLICATION SAFELY
# ============================================================

if APP_PATH.exists():

    existing_source = APP_PATH.read_text(
        encoding="utf-8"
    )

    if existing_source != app_source:
        raise FileExistsError(
            "An app.py file already exists with different "
            "content. Review it before overwriting."
        )

    print("Existing app.py verified.")

else:

    APP_PATH.write_text(
        app_source,
        encoding="utf-8"
    )

    print("Standalone app.py created successfully.")

# ============================================================
# 5. FINAL VERIFICATION
# ============================================================

print("\n=== STANDALONE APP VERIFICATION ===")

print("Application file:", APP_PATH.exists())
print("Python syntax: VALID")
print("Original model:", (APP_DIR / "arabic_residual_cnn.keras").exists())
print("Class labels:", (APP_DIR / "class_labels.json").exists())
print("Preprocessing:", (APP_DIR / "preprocessing.py").exists())

print("\nApplication directory:")
for file in sorted(APP_DIR.iterdir()):
    if file.is_file():
        print("-", file.name)

print("\nStandalone application preparation completed!")


In [ ]:

# ============================================================
#  PREPARE APPLICATION DEPENDENCIES
# ============================================================

import tensorflow as tf
import gradio as gr

from pathlib import Path

# ============================================================
# 1. DEFINE DEPLOYMENT PATHS
# ============================================================

APP_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN/app"
)

REQUIREMENTS_PATH = APP_DIR / "requirements.txt"

assert (APP_DIR / "app.py").exists()
assert (APP_DIR / "arabic_residual_cnn.keras").exists()
assert (APP_DIR / "preprocessing.py").exists()
assert (APP_DIR / "class_labels.json").exists()

# ============================================================
# 2. IDENTIFY LIBRARY VERSIONS
# ============================================================

tensorflow_version = tf.__version__
gradio_version = gr.__version__

print("=== ENVIRONMENT VERIFICATION ===")
print("TensorFlow:", tensorflow_version)
print("Gradio:", gradio_version)

# ============================================================
# 3. PREPARE REQUIREMENTS
# ============================================================

requirements = (
    f"gradio=={gradio_version}\n"
    f"tensorflow-cpu=={tensorflow_version}\n"
    "numpy>=1.26,<2.3\n"
    "Pillow>=10,<13\n"
)

# ============================================================
# 4. SAVE REQUIREMENTS SAFELY
# ============================================================

if REQUIREMENTS_PATH.exists():

    existing = REQUIREMENTS_PATH.read_text(
        encoding="utf-8"
    )

    if existing != requirements:
        raise FileExistsError(
            "An existing requirements.txt differs. "
            "Review it before overwriting."
        )

    print("\nExisting requirements verified.")

else:

    REQUIREMENTS_PATH.write_text(
        requirements,
        encoding="utf-8"
    )

    print("\nrequirements.txt created successfully.")

# ============================================================
# 5. VERIFY DEPLOYMENT FILES
# ============================================================

print("\n=== DEPLOYMENT DEPENDENCIES ===")
print(REQUIREMENTS_PATH.read_text(encoding="utf-8"))

print("=== FINAL VERIFICATION ===")
print("Application file:", (APP_DIR / "app.py").exists())
print("Requirements file:", REQUIREMENTS_PATH.exists())
print("Trained model:", (APP_DIR / "arabic_residual_cnn.keras").exists())
print("Preprocessing module:", (APP_DIR / "preprocessing.py").exists())
print("Class labels:", (APP_DIR / "class_labels.json").exists())

print("\nDeployment dependencies prepared successfully!")


# Create the Hugging Face Space README

In [ ]:

# ============================================================
# PREPARE HUGGING FACE SPACES README


from pathlib import Path

# ============================================================
# 1. DEFINE DEPLOYMENT DIRECTORY
# ============================================================

APP_DIR = Path(
    "/content/drive/MyDrive/Arabic_Handwritten_CNN/app"
)

README_PATH = APP_DIR / "README.md"

required_files = [
    "app.py",
    "requirements.txt",
    "arabic_residual_cnn.keras",
    "class_labels.json",
    "preprocessing.py"
]

for filename in required_files:
    assert (APP_DIR / filename).exists(), (
        f"Missing deployment file: {filename}"
    )

# ============================================================
# 2. DEFINE README CONTENT
# ============================================================

readme_content = """---
title: Arabic Handwritten Character Recognition
emoji: 🇵🇸
colorFrom: green
colorTo: red
sdk: gradio
sdk_version: 6.30.0
python_version: "3.11"
app_file: app.py
suggested_hardware: cpu-basic
short_description: Recognize 28 handwritten Arabic letters with CNN
---

# Arabic Handwritten Character Recognition

An interactive AI application for recognizing
28 handwritten Arabic characters using a
Residual Convolutional Neural Network.

## Application Features

- Draw Arabic letters using a thin brush.
- Upload handwritten character images.
- Display the predicted Arabic character.
- Display the top three predictions.
- Preview the preprocessed 32 x 32 image.
- Use the application without a webcam.

## Model

- Architecture: Residual CNN
- Dataset: Arabic Handwritten Characters Dataset (AHCD)
- Number of classes: 28
- Model parameters: 230,780
- Selected random seed: 789
- Input image shape: 32 x 32 x 1

## Evaluation

Final held-out AHCD test results:

| Metric | Value |
|---|---|
| Test accuracy | 98.3929% |
| Macro F1-score | 98.3896% |
| Top-3 accuracy | 99.8810% |
| Test images | 3,360 |

The model was selected using validation results
before its final test evaluation.

These results apply to the AHCD test dataset.
Performance on user-drawn or uploaded images
may differ.

## Cultural Design

The interface features a Palestine-inspired
visual theme and the following original
Arabic-language message:

**اللغةُ العربيةُ هُوِيَّةٌ وبيان، وفلسطينُ حُرَّةٌ في القلبِ والوجدان**

## Technical Stack

- Python
- TensorFlow / Keras
- Gradio
- NumPy
- Pillow

## Dataset

Arabic Handwritten Characters Dataset (AHCD)

Dataset reference:
https://github.com/mloey/Arabic-Handwritten-Characters-Dataset

## Usage

Open the application, draw one Arabic character
with a thin brush or upload an image, and click
the recognition button.

Model confidence scores should not be interpreted
as guaranteed probabilities of correctness.
"""

# ============================================================
# 3. SAVE README SAFELY
# ============================================================

if README_PATH.exists():
    existing_content = README_PATH.read_text(
        encoding="utf-8"
    )

    assert existing_content == readme_content, (
        "Existing README differs. Review before overwriting."
    )

    print("Existing README verified.")

else:
    README_PATH.write_text(
        readme_content,
        encoding="utf-8"
    )

    print("Hugging Face README created successfully.")

# ============================================================
# 4. VERIFY FILES AND SIZES
# ============================================================

print("\n=== HUGGING FACE DEPLOYMENT CHECK ===")

for filename in required_files + ["README.md"]:
    path = APP_DIR / filename
    size_mb = path.stat().st_size / (1024 ** 2)

    print(
        f"{filename}: FOUND | {size_mb:.2f} MB"
    )

# ============================================================
# 5. FINAL VERIFICATION
# ============================================================

print("\n=== README CONFIGURATION ===")

for line in readme_content.splitlines()[:12]:
    print(line)

print("\nREADME created:", README_PATH.exists())
print("Deployment performed: NO")
print("New training performed: NO")

print("\nHugging Face package preparation completed!")
